## MERCADO LIVRE - CASE DE PREVENÇÃO A FRAUDE

### 1. Importando Bibliotecas

In [1]:
# ============================================================
# IMPORTAÇÕES
# ============================================================

import os
import gc
import re
import json
import pickle
import joblib
import warnings
import unicodedata

import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix
)

from scipy.stats import ks_2samp

import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score
)

from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from nltk.corpus import stopwords

from sklearn.base import clone

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import (
    StandardScaler,
    RobustScaler,
    MinMaxScaler,
    OneHotEncoder,
    LabelEncoder
)

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV,
    RandomizedSearchCV,
    ParameterGrid
)

from sklearn.feature_selection import mutual_info_classif

from sklearn.cluster import (
    KMeans,
    AgglomerativeClustering
)

from sklearn.linear_model import LogisticRegression

from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    balanced_accuracy_score,
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score,
    pairwise_distances
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


# ============================================================
# K-MEDOIDS (OPCIONAL)
# ============================================================

try:
    from sklearn_extra.cluster import KMedoids
    KMEDOIDS_DISPONIVEL = True
except:
    KMEDOIDS_DISPONIVEL = False


warnings.filterwarnings("ignore")

### 2. Funções

In [38]:
# ============================================================
# 3. FUNÇÕES AUXILIARES
# ============================================================

def remover_acentos(texto):
    """
    Remove acentos sem depender de biblioteca externa.
    """
    texto = unicodedata.normalize('NFKD', texto)

    return ''.join(
        caractere
        for caractere in texto
        if not unicodedata.combining(caractere)
    )


def limpar_produto(texto, stop_words=STOP_WORDS):
    """
    Padroniza o nome do produto:
    - preenche texto vazio;
    - converte para minúsculas;
    - corrige espaços;
    - remove acentos;
    - remove caracteres especiais;
    - mantém números;
    - remove stop words.
    """

    if pd.isna(texto):
        texto = 'DESCONHECIDO'

    texto = str(texto).lower().strip()
    texto = remover_acentos(texto)

    # Mantém letras, números e espaços
    texto = re.sub(r'[^a-z0-9\s]', ' ', texto)

    # Remove espaços duplicados
    texto = re.sub(r'\s+', ' ', texto).strip()

    palavras = [
        palavra
        for palavra in texto.split()
        if palavra not in stop_words
    ]

    texto_limpo = ' '.join(palavras)

    if texto_limpo == '':
        texto_limpo = 'desconhecido'

    return texto_limpo


def media_segura(serie, valor_padrao=0.0):
    """
    Retorna a média ignorando nulos e infinitos.
    """

    serie = pd.to_numeric(serie, errors='coerce')
    serie = serie.replace([np.inf, -np.inf], np.nan)

    media = serie.mean()

    if pd.isna(media):
        return valor_padrao

    return float(media)


def criar_mapa_risco_suavizado(
    df,
    coluna,
    target=TARGET,
    suavizacao=SUAVIZACAO_RISCO,
    frequencia_minima=1
):
    """
    Cria target encoding suavizado.

    Fórmula:
        (soma_fraudes + suavizacao * media_global)
        / (quantidade + suavizacao)
    """

    fraude_media = media_segura(df[target], valor_padrao=0.0)

    estatisticas = (
        df
        .groupby(coluna, dropna=False)[target]
        .agg(['count', 'sum'])
    )

    estatisticas = estatisticas[
        estatisticas['count'] >= frequencia_minima
    ].copy()

    estatisticas['risco'] = (
        estatisticas['sum']
        + suavizacao * fraude_media
    ) / (
        estatisticas['count']
        + suavizacao
    )

    return estatisticas['risco'].to_dict()


def preparar_chaves(df):
    """
    Cria as versões padronizadas usadas para aprender os mapas.
    Não altera o DataFrame original.
    """

    base = df.copy()

    # Produto bruto preenchido
    base['i_tratado'] = (
        base['i']
        .fillna('DESCONHECIDO')
        .astype(str)
    )

    # Produto limpo
    base['produto_limpo'] = (
        base['i_tratado']
        .apply(limpar_produto)
    )

    # Categoria numérica tratada
    mediana_l = pd.to_numeric(
        base['l'],
        errors='coerce'
    ).median()

    if pd.isna(mediana_l):
        mediana_l = -1

    base['l_tratado'] = (
        pd.to_numeric(base['l'], errors='coerce')
        .fillna(mediana_l)
    )

    # País tratado
    base['g_tratado'] = (
        base['g']
        .fillna('DESCONHECIDO')
        .astype(str)
        .str.strip()
        .str.upper()
    )

    return base



def aprender_parametros(df_treino):
    """
    Aprende no treino:
    - medianas;
    - percentis;
    - frequências;
    - mapas de risco;
    - média global de fraude.
    """

    treino = df_treino.copy()

    if TARGET not in treino.columns:
        raise ValueError(
            f'A coluna alvo "{TARGET}" não está no treino.'
        )

    treino[TARGET] = pd.to_numeric(
        treino[TARGET],
        errors='coerce'
    )

    if treino[TARGET].isna().any():
        raise ValueError(
            'A variável fraude contém valores nulos ou inválidos.'
        )

    treino_chaves = preparar_chaves(treino)

    params = {}

    # --------------------------------------------------------
    # Preenchimentos categóricos
    # --------------------------------------------------------

    params['o_fill'] = 'NA'
    params['g_fill'] = 'DESCONHECIDO'

    # --------------------------------------------------------
    # Medianas numéricas
    # --------------------------------------------------------

    for coluna in COLUNAS_NUMERICAS_IMPUTAR:

        mediana = pd.to_numeric(
            treino[coluna],
            errors='coerce'
        ).median()

        if pd.isna(mediana):
            mediana = 0.0

        params[f'{coluna}_mediana'] = float(mediana)

    # Compatibilidade com nomes antigos, caso necessário
    for coluna in COLUNAS_NUMERICAS_IMPUTAR:
        params[f'{coluna}_median'] = (
            params[f'{coluna}_mediana']
        )

    # --------------------------------------------------------
    # Monto e score
    # --------------------------------------------------------

    monto_treino = pd.to_numeric(
        treino['monto'],
        errors='coerce'
    )

    score_treino = pd.to_numeric(
        treino['score'],
        errors='coerce'
    )

    params['monto_mediana'] = (
        monto_treino.median()
    )
    
    params['score_mediana'] = (
        score_treino.median()
    )

    params['p10'] = float(
        monto_treino.quantile(0.10)
    )

    params['p90'] = float(
        monto_treino.quantile(0.90)
    )

    if pd.isna(params['p10']):
        params['p10'] = params['monto_mediana']

    if pd.isna(params['p90']):
        params['p90'] = params['monto_mediana']

    # --------------------------------------------------------
    # Média global da fraude
    # --------------------------------------------------------

    params['fraude_media'] = float(
        treino[TARGET].mean()
    )

    # --------------------------------------------------------
    # Frequência do produto limpo
    # --------------------------------------------------------

    params['produto_freq'] = (
        treino_chaves['produto_limpo']
        .value_counts(dropna=False)
        .to_dict()
    )

    # --------------------------------------------------------
    # Frequência da categoria
    # --------------------------------------------------------

    params['categoria_freq'] = (
        treino_chaves['l_tratado']
        .value_counts(dropna=False)
        .to_dict()
    )

    # --------------------------------------------------------
    # Target encoding suavizado do produto
    # --------------------------------------------------------

    base_produto = pd.DataFrame({
        'produto_limpo': treino_chaves['produto_limpo'],
        TARGET: treino[TARGET].values
    })

    params['produto_risco'] = criar_mapa_risco_suavizado(
        df=base_produto,
        coluna='produto_limpo',
        target=TARGET,
        suavizacao=SUAVIZACAO_RISCO,
        frequencia_minima=MIN_FREQ_PRODUTO_RISCO
    )

    # --------------------------------------------------------
    # Target encoding suavizado da categoria
    # --------------------------------------------------------

    base_categoria = pd.DataFrame({
        'l_tratado': treino_chaves['l_tratado'],
        TARGET: treino[TARGET].values
    })

    params['categoria_risco'] = criar_mapa_risco_suavizado(
        df=base_categoria,
        coluna='l_tratado',
        target=TARGET,
        suavizacao=SUAVIZACAO_RISCO,
        frequencia_minima=1
    )

    # --------------------------------------------------------
    # Target encoding suavizado do país
    # --------------------------------------------------------

    base_pais = pd.DataFrame({
        'g_tratado': treino_chaves['g_tratado'],
        TARGET: treino[TARGET].values
    })

    params['pais_risco'] = criar_mapa_risco_suavizado(
        df=base_pais,
        coluna='g_tratado',
        target=TARGET,
        suavizacao=SUAVIZACAO_RISCO,
        frequencia_minima=1
    )

    # Metadados importantes para reprodução
    params['suavizacao_risco'] = SUAVIZACAO_RISCO
    params['min_freq_produto_risco'] = (
        MIN_FREQ_PRODUTO_RISCO
    )

    return params


# ============================================================
# 5. APLICAR O TRATAMENTO
# ============================================================

def aplicar_tratamento(df, params):
    """
    Aplica exatamente o mesmo tratamento em qualquer base.

    A função não aprende nada usando validação ou teste.
    """

    base = df.copy()

    # ========================================================
    # PRODUTO
    # ========================================================

    base['i'] = (
        base['i']
        .fillna('DESCONHECIDO')
        .astype(str)
    )

    # Features antes da limpeza
    base['produto_tamanho'] = (
        base['i']
        .str.len()
        .astype('int32')
    )

    base['produto_qtd_palavras'] = (
        base['i']
        .str.split()
        .str.len()
        .fillna(0)
        .astype('int16')
    )

    base['produto_tem_numero'] = (
        base['i']
        .str.contains(r'\d', regex=True, na=False)
        .astype('int8')
    )

    # Produto limpo
    base['produto_limpo'] = (
        base['i']
        .apply(limpar_produto)
    )

    # Frequência aprendida no treino
    base['produto_freq'] = (
        base['produto_limpo']
        .map(params['produto_freq'])
        .fillna(0)
        .astype('int32')
    )


    # Risco aprendido no treino
    base['produto_risco'] = (
        base['produto_limpo']
        .map(params['produto_risco'])
        .fillna(params['fraude_media'])
        .astype(float)
    )

    # Produto raro
    base['produto_raro'] = (
        base['produto_freq'] <= 3
    ).astype('int8')

    base['produto_muito_raro'] = (
        base['produto_freq'] == 1
    ).astype('int8')

    # ========================================================
    # VARIÁVEIS NUMÉRICAS E MISSING FLAGS
    # ========================================================

    for coluna in COLUNAS_NUMERICAS_IMPUTAR:

        base[coluna] = pd.to_numeric(
            base[coluna],
            errors='coerce'
        )

        base[f'{coluna}_missing'] = (
            base[coluna]
            .isna()
            .astype('int8')
        )

        base[coluna] = (
            base[coluna]
            .fillna(params[f'{coluna}_mediana'])
        )

    # ========================================================
    # CATEGORIA
    # ========================================================

    base['freq_categoria'] = (
        base['l']
        .map(params['categoria_freq'])
        .fillna(0)
        .astype('int32')
    )

    base['categoria_risco'] = (
        base['l']
        .map(params['categoria_risco'])
        .fillna(params['fraude_media'])
        .astype(float)
    )

    base['categoria_rara'] = (
        base['freq_categoria'] <= 10
    ).astype('int8')

    # ========================================================
    # VARIÁVEL O
    # ========================================================

    base['o_missing'] = (
        base['o']
        .isna()
        .astype('int8')
    )

    base['o'] = (
        base['o']
        .fillna(params['o_fill'])
        .astype(str)
    )

    # ========================================================
    # PAÍS
    # ========================================================

    base['g_missing'] = (
        base['g']
        .isna()
        .astype('int8')
    )

    base['g'] = (
        base['g']
        .fillna(params['g_fill'])
        .astype(str)
        .str.strip()
        .str.upper()
    )

    base['pais_risco'] = (
        base['g']
        .map(params['pais_risco'])
        .fillna(params['fraude_media'])
        .astype(float)
    )


    # ========================================================
    # MONTO E SCORE
    # ========================================================

    base['monto'] = pd.to_numeric(
        base['monto'],
        errors='coerce'
    )

    base['score'] = pd.to_numeric(
        base['score'],
        errors='coerce'
    )

    base['monto_missing'] = (
        base['monto']
        .isna()
        .astype('int8')
    )

    base['score_missing'] = (
        base['score']
        .isna()
        .astype('int8')
    )

    base['monto'] = (
        base['monto']
        .fillna(params['monto_mediana'])
    )

    base['score'] = (
        base['score']
        .fillna(params['score_mediana'])
    )

    # Evita erro em monto negativo
    monto_nao_negativo = base['monto'].clip(lower=0)

    base['log_monto'] = np.log1p(
        monto_nao_negativo
    )

    # Limites aprendidos no treino, não valores arbitrários
    base['monto_baixo'] = (
        base['monto'] < params['p10']
    ).astype('int8')

    base['monto_alto'] = (
        base['monto'] > params['p90']
    ).astype('int8')


    base['monto_por_palavra'] = (
        base['monto']
        / (base['produto_qtd_palavras'] + 1)
    )

    # ========================================================
    # DATA
    # ========================================================

    base['fecha'] = pd.to_datetime(
        base['fecha'],
        errors='coerce'
    )

    base['fecha_missing'] = (
        base['fecha']
        .isna()
        .astype('int8')
    )

    base['dia'] = (
        base['fecha']
        .dt.day
        .fillna(-1)
        .astype('int16')
    )

    base['mes'] = (
        base['fecha']
        .dt.month
        .fillna(-1)
        .astype('int8')
    )

    base['hora'] = (
        base['fecha']
        .dt.hour
        .fillna(-1)
        .astype('int8')
    )

    base['dia_semana'] = (
        base['fecha']
        .dt.dayofweek
        .fillna(-1)
        .astype('int8')
    )

    base['fim_semana'] = (
        base['dia_semana']
        .isin([5, 6])
        .astype('int8')
    )

    base['madrugada'] = (
        base['hora']
        .between(0, 5)
        .astype('int8')
    )

    base['hora_comercial'] = (
        base['hora']
        .between(8, 18)
        .astype('int8')
    )

    # ========================================================
    # RISCO CONSOLIDADO
    # ========================================================

    base['risco_medio_historico'] = (
        base[
            [
                'produto_risco',
                'categoria_risco',
                'pais_risco'
            ]
        ]
        .mean(axis=1)
    )

    # ========================================================
    # PROTEÇÃO CONTRA INFINITOS
    # ========================================================

    colunas_numericas = (
        base
        .select_dtypes(include=np.number)
        .columns
    )

    base[colunas_numericas] = (
        base[colunas_numericas]
        .replace([np.inf, -np.inf], np.nan)
    )

    return base


# ============================================================
# 6. TARGET ENCODING OUT-OF-FOLD PARA O TREINO
# ============================================================

def criar_riscos_oof(
    df_treino,
    n_splits=N_SPLITS_OOF,
    random_state=RANDOM_STATE
):
    """
    Gera produto_risco, categoria_risco e pais_risco no treino
    sem utilizar o target da própria linha.

    Frequências e demais transformações não usam a variável-alvo.
    Somente os mapas de risco precisam de OOF.
    """

    treino = df_treino.copy()

    y = (
        pd.to_numeric(
            treino[TARGET],
            errors='coerce'
        )
        .astype(int)
    )

    riscos_oof = pd.DataFrame(
        index=treino.index,
        columns=[
            'produto_risco',
            'categoria_risco',
            'pais_risco'
        ],
        dtype=float
    )

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    for indice_fit, indice_fold in skf.split(treino, y):

        treino_fit = treino.iloc[indice_fit].copy()
        treino_fold = treino.iloc[indice_fold].copy()

        params_fold = aprender_parametros(treino_fit)

        fold_tratado = aplicar_tratamento(
            treino_fold,
            params_fold
        )

        riscos_oof.loc[
            treino_fold.index,
            'produto_risco'
        ] = fold_tratado['produto_risco'].values

        riscos_oof.loc[
            treino_fold.index,
            'categoria_risco'
        ] = fold_tratado['categoria_risco'].values

        riscos_oof.loc[
            treino_fold.index,
            'pais_risco'
        ] = fold_tratado['pais_risco'].values

    return riscos_oof


# ============================================================
# 12. VALIDAÇÕES DAS BASES
# ============================================================

def validar_base(nome, df):

    print("\n" + "="*80)
    print(nome)
    print("="*80)

    print(f"Shape: {df.shape}")

    # -------------------------------------------------------
    # NULOS
    # -------------------------------------------------------

    nulos = df.isnull().sum()
    nulos = nulos[nulos > 0]

    print(f"\nTotal nulos: {nulos.sum():,}")

    if len(nulos) > 0:

        print("\nColunas com nulos:")

        display(
            pd.DataFrame({
                'nulos': nulos,
                '%_nulos':
                (nulos / len(df) * 100).round(2)
            })
            .sort_values(
                'nulos',
                ascending=False
            )
        )

    # -------------------------------------------------------
    # INFINITOS
    # -------------------------------------------------------

    numericas = (
        df
        .select_dtypes(
            include=np.number
        )
        .columns
    )

    infinitos = np.isinf(
        df[numericas]
    ).sum().sum()

    print(f"\nTotal infinitos: {infinitos:,}")

    # -------------------------------------------------------
    # DUPLICADOS
    # -------------------------------------------------------

    duplicados = df.duplicated().sum()

    print(
        f"\nLinhas duplicadas: "
        f"{duplicados:,}"
    )

    # -------------------------------------------------------
    # COLUNAS CONSTANTES
    # -------------------------------------------------------

    constantes = [
        c
        for c in df.columns
        if df[c].nunique(dropna=False) <= 1
    ]

    print(
        f"\nColunas constantes: "
        f"{len(constantes)}"
    )

    if len(constantes) > 0:

        print(constantes)

    # -------------------------------------------------------
    # TIPO DAS COLUNAS
    # -------------------------------------------------------

    print("\nTipos:")

    display(
        pd.DataFrame({
            'tipo': df.dtypes
        })
        .value_counts()
        .reset_index(name='qtd')
    )

    # -------------------------------------------------------
    # FRAUDE
    # -------------------------------------------------------

    if 'fraude' in df.columns:

        print(
            f"\nTaxa fraude: "
            f"{df['fraude'].mean()*100:.2f}%"
        )

    print("="*80)
	


# ============================================================
# 2. FUNÇÕES AUXILIARES
# ============================================================

def tratar_coluna_p(serie):
    """
    Converte a variável p para 0/1.
    Valores desconhecidos são convertidos para NaN
    e posteriormente imputados pela moda do treino.
    """

    valores = (
        serie
        .astype("string")
        .str.strip()
        .str.upper()
    )

    return valores.map({
        "N": 0,
        "Y": 1,
        "0": 0,
        "1": 1
    })


def tratar_coluna_o(serie):
    """
    Padroniza a variável categórica o.
    """

    return (
        serie
        .astype("string")
        .str.strip()
        .str.upper()
        .fillna("NA")
        .replace({
            "": "NA",
            "<NA>": "NA",
            "NAN": "NA",
            "NONE": "NA"
        })
    )


def recuperar_variavel_original(
    nome_feature,
    categoricas_originais
):
    """
    Recupera o nome da variável original após o ColumnTransformer
    e o OneHotEncoder.
    """

    if nome_feature.startswith("numericas__"):
        return nome_feature.replace(
            "numericas__",
            "",
            1
        )

    if nome_feature.startswith("binarias__"):
        return nome_feature.replace(
            "binarias__",
            "",
            1
        )

    if nome_feature.startswith("categoricas__"):

        restante = nome_feature.replace(
            "categoricas__",
            "",
            1
        )

        # Ordenar pelo maior nome evita conflito quando
        # uma variável é prefixo de outra.
        for variavel in sorted(
            categoricas_originais,
            key=len,
            reverse=True
        ):
            if restante.startswith(f"{variavel}_"):
                return variavel

        return restante

    return nome_feature


def preparar_target(df, target, nome_base):
    """
    Converte e valida o target binário.
    """

    if target not in df.columns:
        raise ValueError(
            f"A coluna target '{target}' não existe "
            f"na base de {nome_base}."
        )

    y = pd.to_numeric(
        df[target],
        errors="coerce"
    )

    if y.isna().any():
        quantidade = int(y.isna().sum())

        raise ValueError(
            f"A base de {nome_base} possui {quantidade} "
            f"linhas com target ausente ou inválido."
        )

    y = y.astype(int)

    valores = sorted(y.unique().tolist())

    if not set(valores).issubset({0, 1}):
        raise ValueError(
            f"O target da base de {nome_base} deve conter "
            f"somente 0 e 1. Valores encontrados: {valores}"
        )

    return y


def calcular_metricas_threshold(
    y_real,
    probabilidades,
    limite=0.50
):
    """
    Calcula métricas usando o limite informado.
    A AUC é calculada com as probabilidades do modelo.
    """

    y_real = np.asarray(y_real).astype(int)
    probabilidades = np.asarray(probabilidades)

    predicoes = (
        probabilidades >= limite
    ).astype(int)

    metricas = {
        "precision": precision_score(
            y_real,
            predicoes,
            zero_division=0
        ),
        "recall": recall_score(
            y_real,
            predicoes,
            zero_division=0
        ),
        "f1": f1_score(
            y_real,
            predicoes,
            zero_division=0
        ),
        "auc": (
            roc_auc_score(
                y_real,
                probabilidades
            )
            if len(np.unique(y_real)) == 2
            else np.nan
        )
    }

    return metricas


# ============================================================
# 3. FUNÇÃO PRINCIPAL
# ============================================================

def gerar_ranking_completo(
    df_treino,
    df_teste,
    target="fraude",
    colunas_excluir=None,
    categoricas_informadas=None,
    binarias_informadas=None,
    top_n=50,
    salvar_objetos=True
):

    if colunas_excluir is None:
        colunas_excluir = []

    if categoricas_informadas is None:
        categoricas_informadas = []

    if binarias_informadas is None:
        binarias_informadas = []

    # --------------------------------------------------------
    # 3.1 Fazer cópias
    # --------------------------------------------------------

    treino = df_treino.copy()
    teste = df_teste.copy()

    for df_aux in [treino, teste]:

        df_aux.columns = (
            df_aux.columns
            .astype(str)
            .str.strip()
        )

        if "p" in df_aux.columns:
            df_aux["p"] = tratar_coluna_p(
                df_aux["p"]
            )

        if "o" in df_aux.columns:
            df_aux["o"] = tratar_coluna_o(
                df_aux["o"]
            )

    # --------------------------------------------------------
    # 3.2 Preparar os targets
    # --------------------------------------------------------

    y_treino = preparar_target(
        treino,
        target,
        "treino"
    )

    y_teste = preparar_target(
        teste,
        target,
        "teste"
    )

    # --------------------------------------------------------
    # 3.3 Detectar datas usando somente o treino
    # --------------------------------------------------------

    colunas_datetime = (
        treino
        .select_dtypes(
            include=["datetime", "datetimetz"]
        )
        .columns
        .tolist()
    )

    colunas_excluir_final = list(
        dict.fromkeys(
            colunas_excluir
            + colunas_datetime
            + [target]
        )
    )

    # --------------------------------------------------------
    # 3.4 Separar as variáveis explicativas
    # --------------------------------------------------------

    X_treino = treino.drop(
        columns=colunas_excluir_final,
        errors="ignore"
    ).copy()

    X_teste = teste.drop(
        columns=colunas_excluir_final,
        errors="ignore"
    ).copy()

    print("\nColunas excluídas:")
    print(colunas_excluir_final)

    # --------------------------------------------------------
    # 3.5 Garantir as mesmas colunas
    # --------------------------------------------------------

    colunas_treino = X_treino.columns.tolist()

    colunas_faltantes_teste = [
        coluna
        for coluna in colunas_treino
        if coluna not in X_teste.columns
    ]

    colunas_extras_teste = [
        coluna
        for coluna in X_teste.columns
        if coluna not in colunas_treino
    ]

    print("\nColunas faltantes na teste:")
    print(colunas_faltantes_teste)

    print("\nColunas extras na teste:")
    print(colunas_extras_teste)

    X_teste = X_teste.reindex(
        columns=colunas_treino
    )

    # --------------------------------------------------------
    # 3.6 Definir os grupos de variáveis
    # --------------------------------------------------------

    categoricas = [
        coluna
        for coluna in categoricas_informadas
        if coluna in X_treino.columns
    ]

    binarias = [
        coluna
        for coluna in binarias_informadas
        if coluna in X_treino.columns
        and coluna not in categoricas
    ]

    numericas = [
        coluna
        for coluna in X_treino.columns
        if coluna not in categoricas
        and coluna not in binarias
        and pd.api.types.is_numeric_dtype(
            X_treino[coluna]
        )
    ]

    classificadas = set(
        numericas
        + binarias
        + categoricas
    )

    nao_classificadas = [
        coluna
        for coluna in X_treino.columns
        if coluna not in classificadas
    ]

    print("\nNuméricas:")
    print(numericas)

    print("\nBinárias:")
    print(binarias)

    print("\nCategóricas:")
    print(categoricas)

    print("\nNão classificadas e descartadas:")
    print(nao_classificadas)

    # --------------------------------------------------------
    # 3.7 Padronizar os tipos
    # --------------------------------------------------------

    for df_aux in [X_treino, X_teste]:

        for coluna in numericas:
            df_aux[coluna] = pd.to_numeric(
                df_aux[coluna],
                errors="coerce"
            )

        for coluna in binarias:
            df_aux[coluna] = pd.to_numeric(
                df_aux[coluna],
                errors="coerce"
            )

        for coluna in categoricas:
            df_aux[coluna] = (
                df_aux[coluna]
                .astype("string")
                .fillna("DESCONHECIDO")
                .astype(object)
            )

        colunas_numericas_base = (
            numericas + binarias
        )

        if colunas_numericas_base:
            df_aux[colunas_numericas_base] = (
                df_aux[colunas_numericas_base]
                .replace(
                    [np.inf, -np.inf],
                    np.nan
                )
            )

    # --------------------------------------------------------
    # 3.8 Validar as variáveis binárias no treino
    # --------------------------------------------------------

    problemas_binarias = []

    for coluna in binarias:

        valores = set(
            X_treino[coluna]
            .dropna()
            .unique()
            .tolist()
        )

        if not valores.issubset({0, 1}):
            problemas_binarias.append({
                "variavel": coluna,
                "valores": sorted(valores)
            })

    if problemas_binarias:
        raise ValueError(
            "Existem variáveis classificadas como binárias "
            "com valores diferentes de 0/1: "
            f"{problemas_binarias}"
        )

    # ========================================================
    # 4. CRIAR O PRÉ-PROCESSADOR
    # ========================================================

    # Para LightGBM, não é necessário aplicar RobustScaler.
    pipeline_numericas = Pipeline(
        steps=[
            (
                "imputacao",
                SimpleImputer(
                    strategy="median"
                )
            )
        ]
    )

    pipeline_binarias = Pipeline(
        steps=[
            (
                "imputacao",
                SimpleImputer(
                    strategy="most_frequent"
                )
            )
        ]
    )

    pipeline_categoricas = Pipeline(
        steps=[
            (
                "imputacao",
                SimpleImputer(
                    strategy="constant",
                    fill_value="DESCONHECIDO"
                )
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    min_frequency=5,
                    sparse_output=True,
                    dtype=np.float32
                )
            )
        ]
    )

    transformadores = []

    if numericas:
        transformadores.append(
            (
                "numericas",
                pipeline_numericas,
                numericas
            )
        )

    if binarias:
        transformadores.append(
            (
                "binarias",
                pipeline_binarias,
                binarias
            )
        )

    if categoricas:
        transformadores.append(
            (
                "categoricas",
                pipeline_categoricas,
                categoricas
            )
        )

    if not transformadores:
        raise ValueError(
            "Nenhuma variável válida foi encontrada."
        )

    preprocessador = ColumnTransformer(
        transformers=transformadores,
        remainder="drop",
        sparse_threshold=0.3,
        verbose_feature_names_out=True
    )

    # --------------------------------------------------------
    # 4.1 Fit somente no treino
    # --------------------------------------------------------

    X_treino_processado = (
        preprocessador.fit_transform(
            X_treino,
            y_treino
        )
    )

    # Somente transform na teste
    X_teste_processado = (
        preprocessador.transform(
            X_teste
        )
    )

    print("\nPré-processamento concluído.")

    print(
        "Dimensão original do treino:",
        X_treino.shape
    )

    print(
        "Dimensão processada do treino:",
        X_treino_processado.shape
    )

    print(
        "Dimensão processada da teste:",
        X_teste_processado.shape
    )

    if (
        X_treino_processado.shape[1]
        != X_teste_processado.shape[1]
    ):
        raise ValueError(
            "Treino e teste ficaram com números "
            "diferentes de features."
        )

    # ========================================================
    # 5. MUTUAL INFORMATION NAS VARIÁVEIS ORIGINAIS
    # ========================================================

    X_mi = X_treino[
        numericas
        + binarias
        + categoricas
    ].copy()

    colunas_discretas = set(
        binarias + categoricas
    )

    # LabelEncoder é utilizado somente para calcular MI.
    for coluna in categoricas:

        valores = (
            X_mi[coluna]
            .fillna("DESCONHECIDO")
            .astype(str)
        )

        encoder = LabelEncoder()

        X_mi[coluna] = encoder.fit_transform(
            valores
        )

    for coluna in numericas + binarias:

        X_mi[coluna] = pd.to_numeric(
            X_mi[coluna],
            errors="coerce"
        )

    X_mi = X_mi.replace(
        [np.inf, -np.inf],
        np.nan
    )

    # Imputação aprendida somente com o treino.
    for coluna in X_mi.columns:

        if not X_mi[coluna].isna().any():
            continue

        if coluna in colunas_discretas:

            moda = X_mi[coluna].mode(
                dropna=True
            )

            valor_imputacao = (
                moda.iloc[0]
                if not moda.empty
                else 0
            )

        else:

            valor_imputacao = (
                X_mi[coluna].median()
            )

            if pd.isna(valor_imputacao):
                valor_imputacao = 0

        X_mi[coluna] = (
            X_mi[coluna]
            .fillna(valor_imputacao)
        )

    mascara_discretas = np.array([
        coluna in colunas_discretas
        for coluna in X_mi.columns
    ])

    mi = mutual_info_classif(
        X_mi,
        y_treino,
        discrete_features=mascara_discretas,
        random_state=42
    )

    mi_df = pd.DataFrame({
        "variavel": X_mi.columns,
        "mutual_information": mi
    })

    mi_df["rank_mi"] = (
        mi_df["mutual_information"]
        .rank(
            ascending=False,
            method="average"
        )
    )

    # ========================================================
    # 6. TREINAR O LIGHTGBM PARA SELEÇÃO
    # ========================================================

    y_lgbm_treino = np.asarray(
        y_treino
    ).astype(int)

    y_lgbm_teste = np.asarray(
        y_teste
    ).astype(int)

    modelo_lgbm = LGBMClassifier(
        objective="binary",
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.80,
        colsample_bytree=0.80,
        importance_type="gain",
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )

    modelo_lgbm.fit(
        X_treino_processado,
        y_lgbm_treino
    )

    # ========================================================
    # 7. MÉTRICAS NA teste
    # ========================================================

    proba_teste = (
        modelo_lgbm.predict_proba(
            X_teste_processado
        )[:, 1]
    )

    metricas_teste = calcular_metricas_threshold(
        y_real=y_lgbm_teste,
        probabilidades=proba_teste,
        limite=0.50
    )

    print("\n" + "=" * 80)
    print("DESEMPENHO LIGHTGBM NA teste")
    print("=" * 80)

    print(
        f"Precisão : "
        f"{metricas_teste['precision']:.4f}"
    )

    print(
        f"Cobertura: "
        f"{metricas_teste['recall']:.4f}"
    )

    print(
        f"F1       : "
        f"{metricas_teste['f1']:.4f}"
    )

    print(
        f"AUC      : "
        f"{metricas_teste['auc']:.4f}"
    )

    # ========================================================
    # 8. IMPORTÂNCIA DAS FEATURES TRANSFORMADAS
    # ========================================================

    features_transformadas = (
        preprocessador.get_feature_names_out()
    )

    importancias = (
        modelo_lgbm.feature_importances_
    )

    if len(features_transformadas) != len(importancias):
        raise ValueError(
            "A quantidade de nomes das features é diferente "
            "da quantidade de importâncias."
        )

    importancia_transformada = pd.DataFrame({
        "feature_transformada": features_transformadas,
        "importancia": importancias
    })

    importancia_transformada[
        "variavel_original"
    ] = (
        importancia_transformada[
            "feature_transformada"
        ]
        .apply(
            lambda nome: recuperar_variavel_original(
                nome,
                categoricas
            )
        )
    )

    # ========================================================
    # 9. AGRUPAR ONE-HOT NA VARIÁVEL ORIGINAL
    # ========================================================

    imp_df = (
        importancia_transformada
        .groupby(
            "variavel_original",
            as_index=False
        )
        .agg(
            importancia=(
                "importancia",
                "sum"
            ),
            qtd_features_geradas=(
                "feature_transformada",
                "count"
            )
        )
        .rename(
            columns={
                "variavel_original": "variavel"
            }
        )
    )

    imp_df["rank_importancia"] = (
        imp_df["importancia"]
        .rank(
            ascending=False,
            method="average"
        )
    )

    # ========================================================
    # 10. COMBINAR MUTUAL INFORMATION E IMPORTÂNCIA
    # ========================================================

    ranking = (
        mi_df
        .merge(
            imp_df,
            on="variavel",
            how="inner"
        )
    )

    ranking["ranking_medio"] = (
        ranking[
            [
                "rank_mi",
                "rank_importancia"
            ]
        ]
        .mean(axis=1)
    )

    maior_mi = (
        ranking["mutual_information"].max()
    )

    maior_importancia = (
        ranking["importancia"].max()
    )

    ranking["mi_norm"] = (
        ranking["mutual_information"]
        / (maior_mi + 1e-9)
    )

    ranking["imp_norm"] = (
        ranking["importancia"]
        / (maior_importancia + 1e-9)
    )

    # Score principal: média simples, sem pesos arbitrários.
    ranking["score_combinado"] = (
        ranking[
            [
                "mi_norm",
                "imp_norm"
            ]
        ]
        .mean(axis=1)
    )

    # Score complementar: exige impacto nos dois critérios.
    ranking[
        "score_combinado_duplo_impacto"
    ] = (
        ranking["mi_norm"]
        * ranking["imp_norm"]
    )

    ranking = (
        ranking
        .sort_values(
            by=[
                "score_combinado",
                "score_combinado_duplo_impacto",
                "ranking_medio"
            ],
            ascending=[
                False,
                False,
                True
            ]
        )
        .reset_index(drop=True)
    )

    # ========================================================
    # 11. SELECIONAR VARIÁVEIS ACIMA DA MEDIANA
    # ========================================================

    mediana_score = (
        ranking["score_combinado"].median()
    )

    variaveis_selecionadas = (
        ranking.loc[
            ranking["score_combinado"]
            > mediana_score,
            "variavel"
        ]
        .tolist()
    )

    # Garantir pelo menos uma variável.
    if not variaveis_selecionadas and not ranking.empty:
        variaveis_selecionadas = [
            ranking.iloc[0]["variavel"]
        ]

    # ========================================================
    # 12. EXIBIR RESULTADOS
    # ========================================================

    colunas_resultado = [
        "variavel",
        "mutual_information",
        "importancia",
        "qtd_features_geradas",
        "rank_mi",
        "rank_importancia",
        "ranking_medio",
        "mi_norm",
        "imp_norm",
        "score_combinado",
        "score_combinado_duplo_impacto"
    ]

    print("\n" + "=" * 100)
    print(f"TOP {top_n} VARIÁVEIS ORIGINAIS")
    print("=" * 100)

    print(
        ranking[
            colunas_resultado
        ]
        .head(top_n)
        .to_string(index=False)
    )

    print(
        "\nMediana do score combinado:",
        mediana_score
    )

    print(
        "\nVariáveis selecionadas acima da mediana:"
    )

    print(variaveis_selecionadas)

    # ========================================================
    # 13. SALVAR OBJETOS
    # ========================================================

    if salvar_objetos:

        joblib.dump(
            preprocessador,
            "models/preprocessador_ranking.pkl"
        )

        joblib.dump(
            modelo_lgbm,
            "models/modelo_lgbm_selecao.pkl"
        )

        joblib.dump(
            variaveis_selecionadas,
            "models/variaveis_selecionadas.pkl"
        )

        ranking.to_csv(
            "outputs/analytics/rranking_variaveis.csv",
            index=False,
            encoding="utf-8-sig"
        )

        print(
            "\nObjetos salvos:"
        )

        print(
            "- preprocessador_ranking.pkl"
        )

        print(
            "- modelo_lgbm_selecao.pkl"
        )

        print(
            "- variaveis_selecionadas.pkl"
        )

        print(
            "- ranking_variaveis.csv"
        )

    # ========================================================
    # 14. RETORNAR RESULTADOS
    # ========================================================

    return {
        "ranking": ranking,
        "variaveis_selecionadas": (
            variaveis_selecionadas
        ),
        "mediana_score": mediana_score,
        "metricas_teste": (
            metricas_teste
        ),
        "importancia_transformada": (
            importancia_transformada
        ),
        "modelo_lgbm": modelo_lgbm,
        "preprocessador": preprocessador,
        "X_treino": X_treino,
        "X_teste": X_teste,
        "X_treino_processado": (
            X_treino_processado
        ),
        "X_teste_processado": (
            X_teste_processado
        ),
        "y_treino": y_treino,
        "y_teste": y_teste,
        "numericas": numericas,
        "binarias": binarias,
        "categoricas": categoricas,
        "nao_classificadas": (
            nao_classificadas
        )
    }


def calcular_estabilidade(
    algoritmo,
    X,
    k
):

    lista_labels = []

    for seed in SEMENTES:

        if algoritmo == "KMeans":

            modelo = KMeans(
                n_clusters=k,
                random_state=seed,
                n_init=30,
                max_iter=500
            )

        else:

            modelo = KMedoids(
                n_clusters=k,
                random_state=seed,
                init="k-medoids++"
            )

        labels = modelo.fit_predict(X)

        lista_labels.append(
            labels
        )

    aris = []

    for l1, l2 in combinations(
        lista_labels,
        2
    ):

        aris.append(
            adjusted_rand_score(
                l1,
                l2
            )
        )

    return np.mean(aris)
	

def aplicar_clusterizacao(
    df
):

    X = (
        df[
            features_cluster
        ]
        .copy()
        .apply(
            pd.to_numeric,
            errors="coerce"
        )
    )

    X = pd.DataFrame(
        imputer.transform(X),
        columns=features_cluster,
        index=df.index
    )

    for coluna in X.columns:

        X[coluna] = (
            X[coluna]
            .clip(
                lower=limites_winsor[coluna]["p1"],
                upper=limites_winsor[coluna]["p99"]
            )
        )

    X = scaler.transform(X)

    base = df.copy()

    labels = (
        melhor_modelo
        .predict(X)
        .astype(int)
    )

    base["cluster"] = labels

    distancias = (
        melhor_modelo.transform(X)
    )

    for i in range(
        distancias.shape[1]
    ):

        base[
            f"distancia_cluster_{i}"
        ] = (
            distancias[:, i]
        )

    base[
        "menor_distancia_cluster"
    ] = (
        distancias.min(
            axis=1
        )
    )

    base[
        "distancia_cluster_atribuido"
    ] = (
        distancias[
            np.arange(len(base)),
            labels
        ]
    )

    return base

	
def avaliar_top_percent(
    df,
    coluna_score='score',
    target='fraude',
    pct=0.1
):

    base = df.copy()

    # ordena do maior score para o menor
    base = base.sort_values(
        coluna_score,
        ascending=False
    ).reset_index(drop=True)

    total_base = len(base)

    total_fraude = (
        base[target] == 1
    ).sum()

    n_top = int(total_base * pct)

    # cria predição
    base['pred'] = 0

    base.loc[
        :n_top-1,
        'pred'
    ] = 1

    # matriz de confusão
    tp = (
        (base['pred'] == 1)
        &
        (base[target] == 1)
    ).sum()

    fp = (
        (base['pred'] == 1)
        &
        (base[target] == 0)
    ).sum()

    fn = (
        (base['pred'] == 0)
        &
        (base[target] == 1)
    ).sum()

    tn = (
        (base['pred'] == 0)
        &
        (base[target] == 0)
    ).sum()

    # Precisão
    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0
    )

    # Cobertura
    recall = (
        tp / total_fraude
        if total_fraude > 0
        else 0
    )

    #AUC
    auc = roc_auc_score(
        base[target],
        base[coluna_score]
    )


    # F1
    f1 = (
        2 * precision * recall
        /
        (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    balanced_acc = (
        (
            tp / (tp + fn)
            +
            tn / (tn + fp)
        ) / 2
    )

    threshold = (
        base.iloc[n_top-1][coluna_score]
    )

    return {

    'Total Base': total_base,

    'Total Fraudes': int(total_fraude),

    'Precisão': round(precision,4),

    'Cobertura': round(recall,4),

    'F1': round(f1,4),

    'AUC': round(auc,4),


    'Erro_por_Acerto': round(
        fp / tp if tp > 0 else np.nan,
        2
    ),

    'Lift': round(
        precision /
        (total_fraude / total_base),
        2
    ),

    'Selecionados': int(n_top),

    'Fraudes Encontradas': int(tp),

    'Threshold': round(
        float(threshold),
        6
    )
}


# ============================================================
# 6. VALIDAR FEATURE SETS
# ============================================================

def validar_feature_sets(
    feature_sets,
    X,
    incluir_todas=False
):

    feature_sets_validos = {}

    for nome_set, features in feature_sets.items():

        features_unicas = list(
            dict.fromkeys(features)
        )

        features_existentes = [
            coluna
            for coluna in features_unicas
            if coluna in X.columns
        ]

        features_faltantes = [
            coluna
            for coluna in features_unicas
            if coluna not in X.columns
        ]

        if features_faltantes:

            print(
                f'\nAtenção no set "{nome_set}": '
                f"features não encontradas: {features_faltantes}"
            )

        if features_existentes:

            feature_sets_validos[nome_set] = (
                features_existentes
            )

    if incluir_todas:

        feature_sets_validos[
            "todas_features"
        ] = X.columns.tolist()

    if not feature_sets_validos:
        raise ValueError(
            "Nenhum feature set válido foi encontrado."
        )

    return feature_sets_validos



# ============================================================
# 7. CRIAR PRÉ-PROCESSADOR PARA CADA FEATURE SET
# ============================================================

def criar_preprocessador(
    X,
    features
):

    features = [
        coluna
        for coluna in dict.fromkeys(features)
        if coluna in X.columns
    ]

    # --------------------------------------------------------
    # Categóricas
    # --------------------------------------------------------

    categoricas_informadas_set = [
        coluna
        for coluna in features
        if coluna in CATEGORICAS_INFORMADAS
    ]

    categoricas_automaticas = [
        coluna
        for coluna in features
        if coluna not in categoricas_informadas_set
        and (
            pd.api.types.is_object_dtype(X[coluna])
            or pd.api.types.is_string_dtype(X[coluna])
            or isinstance(
                X[coluna].dtype,
                pd.CategoricalDtype
            )
        )
    ]

    colunas_categoricas = list(
        dict.fromkeys(
            categoricas_informadas_set
            + categoricas_automaticas
        )
    )

    # --------------------------------------------------------
    # Binárias
    # --------------------------------------------------------

    colunas_binarias = [
        coluna
        for coluna in features
        if coluna in BINARIAS_INFORMADAS
        and coluna not in colunas_categoricas
    ]

    # --------------------------------------------------------
    # Numéricas
    # --------------------------------------------------------

    colunas_numericas = [
        coluna
        for coluna in features
        if coluna not in colunas_categoricas
        and coluna not in colunas_binarias
        and pd.api.types.is_numeric_dtype(
            X[coluna]
        )
    ]

    # --------------------------------------------------------
    # Não classificadas
    # --------------------------------------------------------

    classificadas = set(
        colunas_numericas
        + colunas_binarias
        + colunas_categoricas
    )

    nao_classificadas = [
        coluna
        for coluna in features
        if coluna not in classificadas
    ]

    if nao_classificadas:

        print(
            "Atenção, variáveis descartadas por tipo "
            f"não reconhecido: {nao_classificadas}"
        )

    # --------------------------------------------------------
    # Validar binárias
    # --------------------------------------------------------

    erros_binarias = []

    for coluna in colunas_binarias:

        valores = set(
            pd.to_numeric(
                X[coluna],
                errors="coerce"
            )
            .dropna()
            .unique()
            .tolist()
        )

        if not valores.issubset({0, 1}):

            erros_binarias.append({
                "variavel": coluna,
                "valores": sorted(valores)
            })

    if erros_binarias:

        raise ValueError(
            "Variáveis classificadas como binárias possuem "
            f"valores diferentes de 0/1: {erros_binarias}"
        )

    # --------------------------------------------------------
    # Pipeline numérico
    # --------------------------------------------------------

    pipeline_numerico = Pipeline(
        steps=[
            (
                "imputacao",
                SimpleImputer(
                    strategy="median"
                )
            ),
            (
                "escala",
                RobustScaler()
            )
        ]
    )

    # --------------------------------------------------------
    # Pipeline binário
    # --------------------------------------------------------

    pipeline_binario = Pipeline(
        steps=[
            (
                "imputacao",
                SimpleImputer(
                    strategy="most_frequent"
                )
            )
        ]
    )

    # --------------------------------------------------------
    # Pipeline categórico
    # --------------------------------------------------------

    pipeline_categorico = Pipeline(
        steps=[
            (
                "imputacao",
                SimpleImputer(
                    strategy="constant",
                    fill_value="DESCONHECIDO"
                )
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    min_frequency=5,
                    sparse_output=True,
                    dtype=np.float32
                )
            )
        ]
    )

    # --------------------------------------------------------
    # ColumnTransformer
    # --------------------------------------------------------

    transformadores = []

    if colunas_numericas:

        transformadores.append(
            (
                "numericas",
                pipeline_numerico,
                colunas_numericas
            )
        )

    if colunas_binarias:

        transformadores.append(
            (
                "binarias",
                pipeline_binario,
                colunas_binarias
            )
        )

    if colunas_categoricas:

        transformadores.append(
            (
                "categoricas",
                pipeline_categorico,
                colunas_categoricas
            )
        )

    if not transformadores:

        raise ValueError(
            "Nenhuma variável válida encontrada "
            "no feature set."
        )

    preprocessador = ColumnTransformer(
        transformers=transformadores,
        remainder="drop",
        sparse_threshold=0.30,
        verbose_feature_names_out=True
    )

    return (
        preprocessador,
        colunas_numericas,
        colunas_binarias,
        colunas_categoricas,
        nao_classificadas
    )


# ============================================================
# 8. MODELOS E PARÂMETROS
# ============================================================

def obter_modelos():

    return {

        'LogisticRegression': (
            LogisticRegression(max_iter=1000, random_state=42),
            {'clf__C': [0.1, 1, 10], 'clf__solver': ['lbfgs', 'liblinear']}
        ),
        'DecisionTree': (
            DecisionTreeClassifier(random_state=42),
            {'clf__max_depth': [None, 5, 10, 20], 'clf__min_samples_split': [2, 5, 10]}
        ),
        'RandomForest': (
            RandomForestClassifier(random_state=42),
            {'clf__n_estimators': [100, 200], 'clf__max_depth': [None, 5, 10]}
        ),
        'XGBClassifier': (
            XGBClassifier(eval_metric='logloss', tree_method='hist', random_state=42),
            {'clf__n_estimators': [100, 200], 'clf__max_depth': [3, 6], 'clf__learning_rate': [0.05, 0.1]}
        ),

        "LightGBM": (
            LGBMClassifier(
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=1,
                verbosity=-1
            ),
            {
                "clf__n_estimators": [
                    200,
                    400
                ],
                "clf__learning_rate": [
                    0.05,
                    0.10
                ],
                "clf__num_leaves": [
                    15,
                    31
                ],
                "clf__subsample": [
                    0.8,
                    1.0
                ],
                "clf__colsample_bytree": [
                    0.8,
                    1.0
                ]
            }
        )
    }


# ============================================================
# 9. PREDIÇÃO PELO TOP PERCENTUAL
# ============================================================

def predicao_top_percentual(
    scores,
    proporcao_top=0.10
):

    scores = np.asarray(
        scores,
        dtype=float
    ).reshape(-1)

    quantidade_top = max(
        1,
        int(
            np.ceil(
                len(scores)
                * proporcao_top
            )
        )
    )

    indices_top = np.argsort(
        -scores
    )[:quantidade_top]

    predicao = np.zeros(
        len(scores),
        dtype=int
    )

    predicao[indices_top] = 1

    threshold = float(
        scores[indices_top[-1]]
    )

    return (
        predicao,
        quantidade_top,
        threshold
    )


# ============================================================
# 10. KS
# ============================================================

def calcular_ks(
    y_real,
    scores
):

    base = pd.DataFrame({
        "target": np.asarray(y_real),
        "score": np.asarray(scores)
    }).sort_values(
        "score",
        ascending=False
    )

    total_fraude = int(
        (base["target"] == 1).sum()
    )

    total_nao_fraude = int(
        (base["target"] == 0).sum()
    )

    if (
        total_fraude == 0
        or total_nao_fraude == 0
    ):
        return np.nan

    fraude_acumulada = (
        (base["target"] == 1).cumsum()
        / total_fraude
    )

    nao_fraude_acumulada = (
        (base["target"] == 0).cumsum()
        / total_nao_fraude
    )

    return float(
        np.max(
            np.abs(
                fraude_acumulada
                - nao_fraude_acumulada
            )
        )
    )


# ============================================================
# 11. MÉTRICAS
# ============================================================

def calcular_metricas(
    y_real,
    scores,
    proporcao_top=0.10
):

    y_real = np.asarray(
        y_real,
        dtype=int
    ).reshape(-1)

    scores = np.asarray(
        scores,
        dtype=float
    ).reshape(-1)

    (
        predicao,
        selecionados,
        threshold
    ) = predicao_top_percentual(
        scores=scores,
        proporcao_top=proporcao_top
    )

    tn, fp, fn, tp = confusion_matrix(
        y_real,
        predicao,
        labels=[0, 1]
    ).ravel()

    total_base = len(y_real)

    total_fraudes = int(
        np.sum(y_real == 1)
    )

    taxa_fraude = (
        total_fraudes / total_base
        if total_base > 0
        else 0
    )

    precisao = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0
    )

    cobertura = (
        tp / total_fraudes
        if total_fraudes > 0
        else 0
    )

    f1 = (
        2 * precisao * cobertura
        / (precisao + cobertura)
        if (precisao + cobertura) > 0
        else 0
    )

    erro_por_acerto = (
        fp / tp
        if tp > 0
        else np.nan
    )

    lift = (
        precisao / taxa_fraude
        if taxa_fraude > 0
        else np.nan
    )

    auc = (
        roc_auc_score(
            y_real,
            scores
        )
        if len(np.unique(y_real)) == 2
        else np.nan
    )

    average_precision = (
        average_precision_score(
            y_real,
            scores
        )
        if len(np.unique(y_real)) == 2
        else np.nan
    )

    balanced_accuracy = (
        balanced_accuracy_score(
            y_real,
            predicao
        )
    )

    ks = calcular_ks(
        y_real,
        scores
    )

    return {
        "Total Base": total_base,
        "Total Fraudes": total_fraudes,
        "Selecionados": int(selecionados),
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn),
        "Precisão": float(precisao),
        "Cobertura": float(cobertura),
        "F1": float(f1),
        "AUC": float(auc),
        "Average Precision": float(
            average_precision
        ),
        "KS": float(ks),
        "Balanced Accuracy": float(
            balanced_accuracy
        ),
        "Erro_por_Acerto": float(
            erro_por_acerto
        ),
        "Lift": float(lift),
        "Threshold": float(threshold)
    }


# ============================================================
# 12. TREINAR TODOS OS SETS E MODELOS
# ============================================================

def executar_classificacao(
    X_treino,
    y_treino,
    X_teste,
    y_teste,
    feature_sets
):

    resultados = []
    modelos_treinados = {}

    y_treino_array = np.asarray(
        y_treino,
        dtype=int
    )

    y_teste_array = np.asarray(
        y_teste,
        dtype=int
    )

    cv = StratifiedKFold(
        n_splits=N_SPLITS_CV,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    modelos = obter_modelos()

    for nome_set, features in feature_sets.items():

        features = [
            coluna
            for coluna in dict.fromkeys(features)
            if coluna in X_treino.columns
        ]

        if not features:
            continue

        (
            preprocessador,
            colunas_numericas,
            colunas_binarias,
            colunas_categoricas,
            colunas_descartadas
        ) = criar_preprocessador(
            X=X_treino,
            features=features
        )

        print("\n" + "=" * 100)
        print(f"FEATURE SET: {nome_set}")
        print("=" * 100)

        print(
            f"Total original: {len(features)} | "
            f"Numéricas: {len(colunas_numericas)} | "
            f"Binárias: {len(colunas_binarias)} | "
            f"Categóricas: {len(colunas_categoricas)}"
        )

        for nome_modelo, (
            estimador,
            parametros
        ) in modelos.items():

            pipeline = Pipeline(
                steps=[
                    (
                        "preprocessor",
                        clone(preprocessador)
                    ),
                    (
                        "clf",
                        clone(estimador)
                    )
                ]
            )

            grid = GridSearchCV(
                estimator=pipeline,
                param_grid=parametros,
                scoring=SCORING_GRID,
                cv=cv,
                n_jobs=N_JOBS,
                refit=True,
                verbose=0,
                return_train_score=False,
                error_score="raise"
            )

            grid.fit(
                X_treino[features],
                y_treino_array
            )

            probabilidades = (
                grid
                .best_estimator_
                .predict_proba(
                    X_teste[features]
                )[:, 1]
            )

            metricas = calcular_metricas(
                y_real=y_teste_array,
                scores=probabilidades,
                proporcao_top=PROPORCAO_TOP
            )

            chave_modelo = (
                f"{nome_set}__{nome_modelo}"
            )

            modelos_treinados[chave_modelo] = {
                "modelo": grid.best_estimator_,
                "feature_set": nome_set,
                "features": features.copy(),
                "numericas": colunas_numericas.copy(),
                "binarias": colunas_binarias.copy(),
                "categoricas": colunas_categoricas.copy(),
                "melhores_parametros": (
                    grid.best_params_
                ),
                "cv_average_precision": float(
                    grid.best_score_
                ),
                "metricas_teste": metricas
            }

            resultados.append({
                "Feature Set": nome_set,
                "Modelo": nome_modelo,
                "Qtd Features Originais": len(features),
                "Qtd Numéricas": len(colunas_numericas),
                "Qtd Binárias": len(colunas_binarias),
                "Qtd Categóricas": len(colunas_categoricas),
                "CV Average Precision": float(
                    grid.best_score_
                ),
                "Melhores Parâmetros": (
                    grid.best_params_
                ),
                **metricas
            })

            print(
                f"➡️ {nome_modelo}"
                f" | F1: {metricas['F1']:.4f}"
                f" | Precisão: {metricas['Precisão']:.4f}"
                f" | Cobertura: {metricas['Cobertura']:.4f}"
                f" | AUC: {metricas['AUC']:.4f}"
                f" | AP: {metricas['Average Precision']:.4f}"
                f" | Erros/Acerto: {metricas['Erro_por_Acerto']:.2f}"
            )

    comparativo = pd.DataFrame(
        resultados
    )

    if comparativo.empty:

        raise ValueError(
            "Nenhum modelo foi treinado."
        )

    comparativo = (
        comparativo
        .sort_values(
            by=[
                "F1",
                "Average Precision",
                "AUC",
                "KS"
            ],
            ascending=[
                False,
                False,
                False,
                False
            ]
        )
        .reset_index(drop=True)
    )

    melhor_linha = comparativo.iloc[0]

    melhor_chave = (
        f"{melhor_linha['Feature Set']}"
        f"__{melhor_linha['Modelo']}"
    )

    melhor_resultado = (
        modelos_treinados[melhor_chave]
    )

    return (
        comparativo,
        modelos_treinados,
        melhor_resultado
    )


# ============================================================
# 3. PREPARAR AS BASES
# ============================================================

def preparar_base_modelagem(df, nome_base):

    base = df.copy()

    base.columns = (
        base.columns
        .astype(str)
        .str.strip()
    )

    colunas_obrigatorias = [
        TARGET,
        COLUNA_VALOR
    ]

    faltantes_obrigatorias = [
        coluna
        for coluna in colunas_obrigatorias
        if coluna not in base.columns
    ]

    if faltantes_obrigatorias:
        raise ValueError(
            f"Colunas obrigatórias ausentes em {nome_base}: "
            f"{faltantes_obrigatorias}"
        )

    features_ausentes = [
        coluna
        for coluna in melhores_features
        if coluna not in base.columns
    ]

    if features_ausentes:
        raise ValueError(
            f"Features do modelo ausentes em {nome_base}: "
            f"{features_ausentes}"
        )

    # Target
    base[TARGET] = pd.to_numeric(
        base[TARGET],
        errors="coerce"
    )

    # Valor da transação
    base[COLUNA_VALOR] = pd.to_numeric(
        base[COLUNA_VALOR],
        errors="coerce"
    )

    # p: binária
    if "p" in base.columns:

        p_original = (
            base["p"]
            .astype("string")
            .str.strip()
            .str.upper()
        )

        valores_validos = {
            "N",
            "Y",
            "0",
            "1"
        }

        valores_invalidos = (
            set(p_original.dropna().unique())
            - valores_validos
        )

        if valores_invalidos:
            raise ValueError(
                f"Valores inválidos em 'p' na base {nome_base}: "
                f"{sorted(valores_invalidos)}"
            )

        base["p"] = p_original.map({
            "N": 0,
            "Y": 1,
            "0": 0,
            "1": 1
        })

    # o: categórica
    if "o" in base.columns:

        base["o"] = (
            base["o"]
            .astype("string")
            .str.strip()
            .str.upper()
            .fillna("NA")
        )

    # cluster: categórica
    if "cluster" in base.columns:

        base["cluster"] = (
            base["cluster"]
            .astype("string")
            .fillna("DESCONHECIDO")
        )

    # Remover apenas registros sem target ou valor
    base = (
        base
        .dropna(
            subset=[
                TARGET,
                COLUNA_VALOR
            ]
        )
        .reset_index(drop=True)
    )

    base[TARGET] = (
        base[TARGET]
        .astype(int)
    )

    return base


# ============================================================
# 8. FUNÇÃO PARA CALCULAR MÉTRICAS E LUCRO
# ============================================================

def calcular_resultado_corte(
    base,
    threshold,
    decil_acumulado,
    percentual_teorico
):

    y_real = (
        base[TARGET]
        .to_numpy(dtype=int)
    )

    scores = (
        base["score_modelo"]
        .to_numpy(dtype=float)
    )

    valores = (
        base[COLUNA_VALOR]
        .to_numpy(dtype=float)
    )

    # --------------------------------------------------------
    # Top risco = selecionado para bloqueio/reprovação
    # --------------------------------------------------------

    selecionado = (
        scores >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_real,
        selecionado,
        labels=[0, 1]
    ).ravel()

    total_base = len(base)

    total_fraudes = int(
        np.sum(y_real == 1)
    )

    quantidade_selecionada = int(
        selecionado.sum()
    )

    percentual_real_selecionado = (
        quantidade_selecionada / total_base
        if total_base > 0
        else 0
    )

    precisao = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0
    )

    cobertura = (
        tp / total_fraudes
        if total_fraudes > 0
        else 0
    )

    f1 = (
        2 * precisao * cobertura
        / (precisao + cobertura)
        if (precisao + cobertura) > 0
        else 0
    )

    erro_por_acerto = (
        fp / tp
        if tp > 0
        else np.nan
    )

    # --------------------------------------------------------
    # Resultado financeiro
    #
    # selecionado = bloqueado
    # não selecionado = aprovado
    # --------------------------------------------------------

    aprovado = selecionado == 0
    bloqueado = selecionado == 1

    legitima = y_real == 0
    fraude = y_real == 1

    legitima_aprovada = (
        aprovado
        & legitima
    )

    fraude_aprovada = (
        aprovado
        & fraude
    )

    legitima_bloqueada = (
        bloqueado
        & legitima
    )

    fraude_bloqueada = (
        bloqueado
        & fraude
    )

    valor_legitimo_aprovado = float(
        valores[legitima_aprovada].sum()
    )

    valor_fraude_aprovada = float(
        valores[fraude_aprovada].sum()
    )

    valor_legitimo_bloqueado = float(
        valores[legitima_bloqueada].sum()
    )

    valor_fraude_bloqueada = float(
        valores[fraude_bloqueada].sum()
    )

    receita_comissao = (
        TAXA_COMISSAO
        * valor_legitimo_aprovado
    )

    perda_com_fraude = (
        PERDA_FRAUDE
        * valor_fraude_aprovada
    )

    lucro_liquido = (
        receita_comissao
        - perda_com_fraude
    )

    # Comissão que deixou de ser recebida ao bloquear
    # uma transação legítima
    comissao_perdida_bloqueio = (
        TAXA_COMISSAO
        * valor_legitimo_bloqueado
    )

    return {
        "Decil Acumulado": decil_acumulado,
        "Top Acumulado": (
            f"Top {percentual_teorico:.0%}"
        ),
        "Threshold Treino": float(threshold),
        "Total Base": int(total_base),
        "Total Fraudes": int(total_fraudes),
        "Selecionados para Bloqueio": (
            quantidade_selecionada
        ),
        "% Real Selecionado": float(
            percentual_real_selecionado
        ),
        "TP - Fraudes Bloqueadas": int(tp),
        "FP - Legítimas Bloqueadas": int(fp),
        "FN - Fraudes Aprovadas": int(fn),
        "TN - Legítimas Aprovadas": int(tn),
        "Precisão": float(precisao),
        "Cobertura": float(cobertura),
        "F1": float(f1),
        "AUC Global": float(
            auc_global_validacao
        ),
        "Erro_por_Acerto": float(
            erro_por_acerto
        ),
        "Valor Legítimo Aprovado": (
            valor_legitimo_aprovado
        ),
        "Valor Fraude Aprovada": (
            valor_fraude_aprovada
        ),
        "Valor Legítimo Bloqueado": (
            valor_legitimo_bloqueado
        ),
        "Valor Fraude Bloqueada": (
            valor_fraude_bloqueada
        ),
        "Receita Comissão": float(
            receita_comissao
        ),
        "Perda com Fraude": float(
            perda_com_fraude
        ),
        "Comissão Perdida por Bloqueio": float(
            comissao_perdida_bloqueio
        ),
        "Lucro Líquido": float(
            lucro_liquido
        )
    }


def atribuir_decil_por_threshold(score):

    for decil, threshold in enumerate(
        thresholds_decis,
        start=1
    ):

        if score >= threshold:
            return decil

    return 10
	

# ============================================================
# 2. PREPARAR AS BASES
# ============================================================

def preparar_base_score(
    df,
    nome_base
):

    base = df.copy()

    base.columns = (
        base.columns
        .astype(str)
        .str.strip()
    )

    colunas_obrigatorias = [
        TARGET,
        COLUNA_SCORE,
        COLUNA_VALOR
    ]

    colunas_ausentes = [
        coluna
        for coluna in colunas_obrigatorias
        if coluna not in base.columns
    ]

    if colunas_ausentes:

        raise ValueError(
            f"Colunas obrigatórias ausentes em {nome_base}: "
            f"{colunas_ausentes}"
        )

    # --------------------------------------------------------
    # Converter target
    # --------------------------------------------------------

    base[TARGET] = pd.to_numeric(
        base[TARGET],
        errors="coerce"
    )

    # --------------------------------------------------------
    # Converter score
    # --------------------------------------------------------

    base[COLUNA_SCORE] = pd.to_numeric(
        base[COLUNA_SCORE],
        errors="coerce"
    )

    # --------------------------------------------------------
    # Converter valor da transação
    # --------------------------------------------------------

    base[COLUNA_VALOR] = pd.to_numeric(
        base[COLUNA_VALOR],
        errors="coerce"
    )

    # --------------------------------------------------------
    # Remover registros sem informações essenciais
    # --------------------------------------------------------

    quantidade_antes = len(base)

    base = (
        base
        .dropna(
            subset=[
                TARGET,
                COLUNA_SCORE,
                COLUNA_VALOR
            ]
        )
        .reset_index(drop=True)
    )

    quantidade_removida = (
        quantidade_antes
        - len(base)
    )

    if quantidade_removida > 0:

        print(
            f"{nome_base}: "
            f"{quantidade_removida} registros removidos "
            "por target, score ou monto ausente."
        )

    base[TARGET] = (
        base[TARGET]
        .astype(int)
    )

    valores_target = set(
        base[TARGET]
        .unique()
        .tolist()
    )

    if not valores_target.issubset({0, 1}):

        raise ValueError(
            f"O target da base {nome_base} contém "
            f"valores diferentes de 0 e 1: "
            f"{sorted(valores_target)}"
        )

    # --------------------------------------------------------
    # Criar score orientado ao risco
    # --------------------------------------------------------
    # O restante do código sempre considera:
    # quanto maior score_risco, maior o risco.

    if SCORE_MAIOR_INDICA_MAIS_RISCO:

        base["score_risco"] = (
            base[COLUNA_SCORE]
            .astype(float)
        )

    else:

        base["score_risco"] = (
            -base[COLUNA_SCORE]
            .astype(float)
        )

    return base
	

# ============================================================
# 6. FUNÇÃO PARA CALCULAR MÉTRICAS ACUMULADAS
# ============================================================

def calcular_resultado_acumulado_score(
    base,
    threshold_score_risco,
    decil_acumulado,
    percentual_teorico,
    auc_global,
    average_precision_global
):

    y_real = (
        base[TARGET]
        .to_numpy(dtype=int)
    )

    scores_risco = (
        base["score_risco"]
        .to_numpy(dtype=float)
    )

    valores = (
        base[COLUNA_VALOR]
        .to_numpy(dtype=float)
    )

    # --------------------------------------------------------
    # Bloquear todos os registros com score de risco
    # maior ou igual ao threshold definido no treino
    # --------------------------------------------------------

    selecionado_bloqueio = (
        scores_risco >= threshold_score_risco
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_real,
        selecionado_bloqueio,
        labels=[0, 1]
    ).ravel()

    total_base = len(base)

    total_fraudes = int(
        np.sum(y_real == 1)
    )

    total_legitimas = int(
        np.sum(y_real == 0)
    )

    quantidade_selecionada = int(
        selecionado_bloqueio.sum()
    )

    percentual_real_selecionado = (
        quantidade_selecionada / total_base
        if total_base > 0
        else 0
    )

    precisao = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0
    )

    cobertura = (
        tp / total_fraudes
        if total_fraudes > 0
        else 0
    )

    f1 = (
        2 * precisao * cobertura
        / (precisao + cobertura)
        if (precisao + cobertura) > 0
        else 0
    )

    erro_por_acerto = (
        fp / tp
        if tp > 0
        else np.nan
    )

    taxa_fraude_base = (
        total_fraudes / total_base
        if total_base > 0
        else 0
    )

    lift = (
        precisao / taxa_fraude_base
        if taxa_fraude_base > 0
        else np.nan
    )

    especificidade = (
        tn / total_legitimas
        if total_legitimas > 0
        else 0
    )

    balanced_accuracy = (
        (cobertura + especificidade) / 2
    )

    # --------------------------------------------------------
    # Resultado financeiro acumulado
    #
    # Bloqueado:
    # score dentro do Top acumulado
    #
    # Aprovado:
    # score abaixo do threshold
    # --------------------------------------------------------

    bloqueado = (
        selecionado_bloqueio == 1
    )

    aprovado = (
        selecionado_bloqueio == 0
    )

    fraude = (
        y_real == 1
    )

    legitima = (
        y_real == 0
    )

    fraude_bloqueada = (
        bloqueado
        & fraude
    )

    legitima_bloqueada = (
        bloqueado
        & legitima
    )

    fraude_aprovada = (
        aprovado
        & fraude
    )

    legitima_aprovada = (
        aprovado
        & legitima
    )

    valor_fraude_bloqueada = float(
        valores[fraude_bloqueada].sum()
    )

    valor_legitimo_bloqueado = float(
        valores[legitima_bloqueada].sum()
    )

    valor_fraude_aprovada = float(
        valores[fraude_aprovada].sum()
    )

    valor_legitimo_aprovado = float(
        valores[legitima_aprovada].sum()
    )

    # --------------------------------------------------------
    # Comissão recebida nas legítimas aprovadas
    # --------------------------------------------------------

    receita_comissao = (
        TAXA_COMISSAO
        * valor_legitimo_aprovado
    )

    # --------------------------------------------------------
    # Perda causada pelas fraudes aprovadas
    # --------------------------------------------------------

    perda_com_fraude = (
        PERDA_FRAUDE
        * valor_fraude_aprovada
    )

    # --------------------------------------------------------
    # Comissão não recebida ao bloquear legítimas
    # Métrica informativa.
    # Não deve ser descontada novamente do lucro.
    # --------------------------------------------------------

    comissao_perdida_bloqueio = (
        TAXA_COMISSAO
        * valor_legitimo_bloqueado
    )

    # --------------------------------------------------------
    # Lucro líquido da estratégia
    # --------------------------------------------------------

    lucro_liquido = (
        receita_comissao
        - perda_com_fraude
    )

    # --------------------------------------------------------
    # Threshold na escala original do score
    # --------------------------------------------------------

    if SCORE_MAIOR_INDICA_MAIS_RISCO:

        threshold_original = (
            threshold_score_risco
        )

    else:

        threshold_original = (
            -threshold_score_risco
        )

    return {
        "Origem Score": "Score Atual",
        "Base Avaliada": "Teste",
        "Decil Acumulado": int(
            decil_acumulado
        ),
        "Top Acumulado": (
            f"Top {percentual_teorico:.0%}"
        ),
        "Threshold Score Original": float(
            threshold_original
        ),
        "Total Base": int(
            total_base
        ),
        "Total Fraudes": int(
            total_fraudes
        ),
        "Selecionados para Bloqueio": int(
            quantidade_selecionada
        ),
        "% Real Selecionado": float(
            percentual_real_selecionado
        ),
        "TP - Fraudes Bloqueadas": int(tp),
        "FP - Legítimas Bloqueadas": int(fp),
        "FN - Fraudes Aprovadas": int(fn),
        "TN - Legítimas Aprovadas": int(tn),
        "Precisão": float(
            precisao
        ),
        "Cobertura": float(
            cobertura
        ),
        "F1": float(
            f1
        ),
        "AUC Global": float(
            auc_global
        ),
        "Average Precision Global": float(
            average_precision_global
        ),
        "Balanced Accuracy": float(
            balanced_accuracy
        ),
        "Erro_por_Acerto": float(
            erro_por_acerto
        ),
        "Lift": float(
            lift
        ),
        "Valor Fraude Bloqueada": float(
            valor_fraude_bloqueada
        ),
        "Valor Legítimo Bloqueado": float(
            valor_legitimo_bloqueado
        ),
        "Valor Fraude Aprovada": float(
            valor_fraude_aprovada
        ),
        "Valor Legítimo Aprovado": float(
            valor_legitimo_aprovado
        ),
        "Receita Comissão": float(
            receita_comissao
        ),
        "Perda com Fraude": float(
            perda_com_fraude
        ),
        "Comissão Perdida por Bloqueio": float(
            comissao_perdida_bloqueio
        ),
        "Lucro Líquido": float(
            lucro_liquido
        )
    }


def atribuir_decil_score(score_risco):

    for decil, threshold in enumerate(
        thresholds_score_risco,
        start=1
    ):

        if score_risco >= threshold:
            return decil

    return 10



# ============================================================
# 5. FUNÇÃO PARA CALCULAR MÉTRICAS GLOBAIS
# ============================================================

def calcular_metricas_globais(
    base,
    coluna_score="score_modelo"
):

    if (
        base[TARGET].nunique() == 2
        and base[coluna_score].nunique() > 1
    ):

        auc = roc_auc_score(
            base[TARGET],
            base[coluna_score]
        )

        average_precision = average_precision_score(
            base[TARGET],
            base[coluna_score]
        )

    else:

        auc = np.nan
        average_precision = np.nan

    return auc, average_precision


# ============================================================
# 6. FUNÇÃO PARA CALCULAR O CENÁRIO APROVAR TUDO
# ============================================================

def calcular_cenario_aprovar_tudo(base):

    valor_legitimo_total = float(
        base.loc[
            base[TARGET] == 0,
            COLUNA_VALOR
        ].sum()
    )

    valor_fraude_total = float(
        base.loc[
            base[TARGET] == 1,
            COLUNA_VALOR
        ].sum()
    )

    receita_base = (
        TAXA_COMISSAO
        * valor_legitimo_total
    )

    perda_base = (
        PERDA_FRAUDE
        * valor_fraude_total
    )

    lucro_base = (
        receita_base
        - perda_base
    )

    return {
        "Valor Legítimo Total": valor_legitimo_total,
        "Valor Fraude Total": valor_fraude_total,
        "Receita Aprovar Tudo": receita_base,
        "Perda Aprovar Tudo": perda_base,
        "Lucro Aprovar Tudo": lucro_base
    }


# ============================================================
# 7. FUNÇÃO PARA AVALIAR UM PONTO DE CORTE
# ============================================================

def calcular_resultado_corte_model(
    base,
    threshold,
    nome_base,
    auc_global,
    average_precision_global,
    lucro_base_aprovar_tudo,
    decil_acumulado=None,
    percentual_teorico=None,
    top_acumulado=None
):

    # 1 = bloquear
    # 0 = aprovar
    predicao_bloqueio = (
        base["score_modelo"]
        >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        base[TARGET],
        predicao_bloqueio,
        labels=[0, 1]
    ).ravel()

    precisao = precision_score(
        base[TARGET],
        predicao_bloqueio,
        zero_division=0
    )

    cobertura = recall_score(
        base[TARGET],
        predicao_bloqueio,
        zero_division=0
    )

    f1 = f1_score(
        base[TARGET],
        predicao_bloqueio,
        zero_division=0
    )

    balanced_accuracy = balanced_accuracy_score(
        base[TARGET],
        predicao_bloqueio
    )

    selecionados_bloqueio = int(
        predicao_bloqueio.sum()
    )

    percentual_real_selecionado = (
        selecionados_bloqueio / len(base)
        if len(base) > 0
        else np.nan
    )

    erro_por_acerto = (
        fp / tp
        if tp > 0
        else np.nan
    )

    taxa_fraude_base = base[TARGET].mean()

    lift = (
        precisao / taxa_fraude_base
        if taxa_fraude_base > 0
        else np.nan
    )

    mascara_bloqueada = (
        predicao_bloqueio == 1
    )

    mascara_aprovada = (
        predicao_bloqueio == 0
    )

    # Receita gerada apenas por transações legítimas aprovadas
    valor_legitimo_aprovado = float(
        base.loc[
            mascara_aprovada
            & (base[TARGET] == 0),
            COLUNA_VALOR
        ].sum()
    )

    # Fraudes que passaram pelo bloqueio
    valor_fraude_aprovada = float(
        base.loc[
            mascara_aprovada
            & (base[TARGET] == 1),
            COLUNA_VALOR
        ].sum()
    )

    # Legítimas bloqueadas
    valor_legitimo_bloqueado = float(
        base.loc[
            mascara_bloqueada
            & (base[TARGET] == 0),
            COLUNA_VALOR
        ].sum()
    )

    receita_comissao = (
        TAXA_COMISSAO
        * valor_legitimo_aprovado
    )

    perda_com_fraude = (
        PERDA_FRAUDE
        * valor_fraude_aprovada
    )

    comissao_perdida_bloqueio = (
        TAXA_COMISSAO
        * valor_legitimo_bloqueado
    )

    lucro_liquido = (
        receita_comissao
        - perda_com_fraude
    )

    ganho_vs_aprovar_tudo = (
        lucro_liquido
        - lucro_base_aprovar_tudo
    )

    ganho_percentual_vs_aprovar_tudo = (
        ganho_vs_aprovar_tudo
        / abs(lucro_base_aprovar_tudo)
        if lucro_base_aprovar_tudo != 0
        else np.nan
    )

    return {
        "Base Avaliada": nome_base,
        "Decil Acumulado": decil_acumulado,
        "Top Acumulado": top_acumulado,
        "Percentual Teórico": percentual_teorico,
        "Threshold Treino": threshold,
        "Total Base": len(base),
        "Selecionados para Bloqueio": (
            selecionados_bloqueio
        ),
        "% Real Selecionado": (
            percentual_real_selecionado
        ),
        "TP - Fraudes Bloqueadas": int(tp),
        "FP - Legítimas Bloqueadas": int(fp),
        "FN - Fraudes Aprovadas": int(fn),
        "TN - Legítimas Aprovadas": int(tn),
        "Precisão": precisao,
        "Cobertura": cobertura,
        "F1": f1,
        "AUC Global": auc_global,
        "Average Precision Global": (
            average_precision_global
        ),
        "Balanced Accuracy": balanced_accuracy,
        "Erro_por_Acerto": erro_por_acerto,
        "Lift": lift,
        "Receita Comissão": receita_comissao,
        "Perda com Fraude": perda_com_fraude,
        "Comissão Perdida por Bloqueio": (
            comissao_perdida_bloqueio
        ),
        "Lucro Líquido": lucro_liquido,
        "Lucro Aprovar Tudo": (
            lucro_base_aprovar_tudo
        ),
        "Ganho vs Aprovar Tudo": (
            ganho_vs_aprovar_tudo
        ),
        "Ganho Percentual vs Aprovar Tudo": (
            ganho_percentual_vs_aprovar_tudo
        )
    }

def atribuir_decil_por_threshold(score):

    for decil, threshold in enumerate(
        thresholds_decis,
        start=1
    ):

        if score >= threshold:
            return decil

    return 10


# ============================================================
# 2. FUNÇÃO PARA PREPARAR AS BASES
# ============================================================

def preparar_base_score_original(df, nome_base):

    colunas_obrigatorias = [
        TARGET,
        COLUNA_SCORE,
        COLUNA_VALOR
    ]

    colunas_ausentes = [
        coluna
        for coluna in colunas_obrigatorias
        if coluna not in df.columns
    ]

    if colunas_ausentes:
        raise ValueError(
            f"Colunas ausentes na base {nome_base}: "
            f"{colunas_ausentes}"
        )

    base = df.copy()

    base[TARGET] = pd.to_numeric(
        base[TARGET],
        errors="coerce"
    )

    base[COLUNA_SCORE] = pd.to_numeric(
        base[COLUNA_SCORE],
        errors="coerce"
    )

    base[COLUNA_VALOR] = pd.to_numeric(
        base[COLUNA_VALOR],
        errors="coerce"
    )

    quantidade_inicial = len(base)

    base = base.dropna(
        subset=[
            TARGET,
            COLUNA_SCORE,
            COLUNA_VALOR
        ]
    ).copy()

    base = base[
        base[TARGET].isin([0, 1])
    ].copy()

    base[COLUNA_VALOR] = base[
        COLUNA_VALOR
    ].clip(lower=0)

    base[TARGET] = base[TARGET].astype(int)

    if SCORE_MAIOR_INDICA_MAIS_RISCO:

        base["score_risco"] = base[
            COLUNA_SCORE
        ].astype(float)

    else:

        base["score_risco"] = (
            -base[COLUNA_SCORE].astype(float)
        )

    base["nome_base"] = nome_base

    quantidade_final = len(base)

    removidos = quantidade_inicial - quantidade_final

    if removidos > 0:
        print(
            f"ATENÇÃO: {removidos} registros foram removidos "
            f"da base {nome_base} por valores inválidos."
        )

    return base.reset_index(drop=True)


# ============================================================
# 3. FUNÇÃO PARA CALCULAR MÉTRICAS GLOBAIS
# ============================================================

def calcular_metricas_globais_original(base):

    if (
        base[TARGET].nunique() == 2
        and base["score_risco"].nunique() > 1
    ):

        auc = roc_auc_score(
            base[TARGET],
            base["score_risco"]
        )

        average_precision = average_precision_score(
            base[TARGET],
            base["score_risco"]
        )

        score_fraude = base.loc[
            base[TARGET] == 1,
            "score_risco"
        ]

        score_legitimo = base.loc[
            base[TARGET] == 0,
            "score_risco"
        ]

        ks = ks_2samp(
            score_fraude,
            score_legitimo
        ).statistic

    else:

        auc = np.nan
        average_precision = np.nan
        ks = np.nan

    return auc, average_precision, ks


# ============================================================
# 4. FUNÇÃO PARA CALCULAR O CENÁRIO APROVAR TUDO
# ============================================================

def calcular_cenario_aprovar_tudo_original(base):

    valor_legitimo = float(
        base.loc[
            base[TARGET] == 0,
            COLUNA_VALOR
        ].sum()
    )

    valor_fraude = float(
        base.loc[
            base[TARGET] == 1,
            COLUNA_VALOR
        ].sum()
    )

    receita = (
        TAXA_COMISSAO
        * valor_legitimo
    )

    perda_fraude = (
        PERDA_FRAUDE
        * valor_fraude
    )

    lucro = receita - perda_fraude

    return {
        "Valor Legítimo Total": valor_legitimo,
        "Valor de Fraude Total": valor_fraude,
        "Receita Aprovar Tudo": receita,
        "Perda Aprovar Tudo": perda_fraude,
        "Lucro Aprovar Tudo": lucro
    }


# ============================================================
# 5. FUNÇÃO PARA AVALIAR UM THRESHOLD
# ============================================================

def avaliar_threshold_original(
    base,
    threshold_score_risco,
    nome_base,
    auc_global,
    average_precision_global,
    ks_global,
    lucro_base_aprovar_tudo,
    decil_acumulado=None,
    percentual_teorico=None,
    top_acumulado=None
):

    predicao_bloqueio = (
        base["score_risco"]
        >= threshold_score_risco
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        base[TARGET],
        predicao_bloqueio,
        labels=[0, 1]
    ).ravel()

    precisao = precision_score(
        base[TARGET],
        predicao_bloqueio,
        zero_division=0
    )

    cobertura = recall_score(
        base[TARGET],
        predicao_bloqueio,
        zero_division=0
    )

    f1 = f1_score(
        base[TARGET],
        predicao_bloqueio,
        zero_division=0
    )

    balanced_accuracy = balanced_accuracy_score(
        base[TARGET],
        predicao_bloqueio
    )

    taxa_fraude_base = base[TARGET].mean()

    lift = (
        precisao / taxa_fraude_base
        if taxa_fraude_base > 0
        else np.nan
    )

    erro_por_acerto = (
        fp / tp
        if tp > 0
        else np.nan
    )

    selecionados_bloqueio = int(
        predicao_bloqueio.sum()
    )

    percentual_real_selecionado = (
        selecionados_bloqueio / len(base)
        if len(base) > 0
        else np.nan
    )

    mascara_bloqueada = (
        predicao_bloqueio == 1
    )

    mascara_aprovada = (
        predicao_bloqueio == 0
    )

    valor_legitimo_aprovado = float(
        base.loc[
            mascara_aprovada
            & (base[TARGET] == 0),
            COLUNA_VALOR
        ].sum()
    )

    valor_fraude_aprovada = float(
        base.loc[
            mascara_aprovada
            & (base[TARGET] == 1),
            COLUNA_VALOR
        ].sum()
    )

    valor_legitimo_bloqueado = float(
        base.loc[
            mascara_bloqueada
            & (base[TARGET] == 0),
            COLUNA_VALOR
        ].sum()
    )

    receita_comissao = (
        TAXA_COMISSAO
        * valor_legitimo_aprovado
    )

    perda_com_fraude = (
        PERDA_FRAUDE
        * valor_fraude_aprovada
    )

    comissao_perdida_bloqueio = (
        TAXA_COMISSAO
        * valor_legitimo_bloqueado
    )

    lucro_liquido = (
        receita_comissao
        - perda_com_fraude
    )

    ganho_vs_aprovar_tudo = (
        lucro_liquido
        - lucro_base_aprovar_tudo
    )

    ganho_percentual = (
        ganho_vs_aprovar_tudo
        / abs(lucro_base_aprovar_tudo)
        if lucro_base_aprovar_tudo != 0
        else np.nan
    )

    threshold_original = (
        threshold_score_risco
        if SCORE_MAIOR_INDICA_MAIS_RISCO
        else -threshold_score_risco
    )

    return {
        "Origem Score": "Score Atual",
        "Base Avaliada": nome_base,
        "Decil Acumulado": decil_acumulado,
        "Top Acumulado": top_acumulado,
        "Percentual Teórico": percentual_teorico,
        "Threshold Score Risco": threshold_score_risco,
        "Threshold Score Original": threshold_original,
        "Total Base": len(base),
        "Selecionados para Bloqueio": selecionados_bloqueio,
        "% Real Selecionado": percentual_real_selecionado,
        "TP - Fraudes Bloqueadas": int(tp),
        "FP - Legítimas Bloqueadas": int(fp),
        "FN - Fraudes Aprovadas": int(fn),
        "TN - Legítimas Aprovadas": int(tn),
        "Precisão": precisao,
        "Cobertura": cobertura,
        "F1": f1,
        "AUC Global": auc_global,
        "Average Precision Global": average_precision_global,
        "KS Global": ks_global,
        "Balanced Accuracy": balanced_accuracy,
        "Erro_por_Acerto": erro_por_acerto,
        "Lift": lift,
        "Receita Comissão": receita_comissao,
        "Perda com Fraude": perda_com_fraude,
        "Comissão Perdida por Bloqueio": comissao_perdida_bloqueio,
        "Lucro Líquido": lucro_liquido,
        "Lucro Aprovar Tudo": lucro_base_aprovar_tudo,
        "Ganho vs Aprovar Tudo": ganho_vs_aprovar_tudo,
        "Ganho Percentual vs Aprovar Tudo": ganho_percentual
    }


def atribuir_decil_score_original(score_risco):

    for numero_decil, threshold in enumerate(
        thresholds_score_risco,
        start=1
    ):

        if score_risco >= threshold:
            return numero_decil

    return 10


### 3. Carga dos Dados

###### Nesta etapa, realizo a carga da base de dados e valido se algumas funções estão globais.

In [14]:
print("preparar_chaves" in globals())
print("criar_mapa_risco_suavizado" in globals())
print("aprender_parametros" in globals())

True
True
True


In [4]:
# Caminho do arquivo
df =  pd.read_csv("https://hr-projects-assets-prod.s3.amazonaws.com/3st1g8gm56o/ed953178414189325c980dc1a9214b97/data.csv")

#df =  pd.read_csv(r'C:\Users\F8090385\PROJETO_MERCADO_LIVRE\data.csv', sep = ',')

### 4. Divisão do Dataframe Original

###### Nesta etapa, analiso a distribuição da variável alvo e realizo a separação da base em conjuntos de treino, validação e teste. Inicialmente, verifico a quantidade total de registros, o volume de fraudes e a taxa de fraude da base completa. Em seguida, realizo uma divisão estratificada em 70% para treino, 15% para validação e 15% para teste, preservando a mesma proporção de fraudes em todas as amostras. Por fim, reconstruo os DataFrames completos de cada conjunto, mantendo as variáveis explicativas e a variável alvo, garantindo bases independentes para treinamento, seleção de parâmetros e avaliação final dos modelos.

In [15]:
qtd_total = len(df)
qtd_fraude = df['fraude'].sum()
perc_fraude = (qtd_fraude / qtd_total) * 100

print(f'Total registros: {qtd_total:,}')
print(f'Total fraudes: {qtd_fraude:,}')
print(f'% Fraude: {perc_fraude:.2f}%')
print('-----------------------------------------')


# Features
X = df.drop(columns=['fraude'])

# Target
y = df['fraude']

################################################
# 70% treino
# 30% temp (validação + teste)
################################################

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

################################################
# 15% validação
# 15% teste
################################################

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print('Treino:', X_train.shape)
print('Validação:', X_valid.shape)
print('Teste:', X_test.shape)
print('-----------------------------------------')
print('Taxa Fraude')
print('Treino:', round(y_train.mean()*100,2),'%')
print('Validação:', round(y_valid.mean()*100,2),'%')
print('Teste:', round(y_test.mean()*100,2),'%')

# Reconstruir os datasets completos

df_treino = X_train.copy()
df_treino['fraude'] = y_train

df_validacao = X_valid.copy()
df_validacao['fraude'] = y_valid

df_teste = X_test.copy()
df_teste['fraude'] = y_test

os.makedirs(
    "outputs/preprocessing",
    exist_ok=True
)

df_treino.to_pickle(
    "outputs/preprocessing/df_treino.pkl"
)

df_validacao.to_pickle(
    "outputs/preprocessing/df_validacao.pkl"
)

df_teste.to_pickle(
    "outputs/preprocessing/df_teste.pkl"
)

Total registros: 150,000
Total fraudes: 7,500
% Fraude: 5.00%
-----------------------------------------
Treino: (105000, 19)
Validação: (22500, 19)
Teste: (22500, 19)
-----------------------------------------
Taxa Fraude
Treino: 5.0 %
Validação: 5.0 %
Teste: 5.0 %


### 5. Preparação dos Dados

#### 5.1 Conhecendo os dados

###### Nesta etapa, realizo uma inspeção inicial da base de dados para entender sua estrutura e qualidade. Verifico a quantidade de registros e variáveis disponíveis, visualizo uma amostra dos dados e gero estatísticas descritivas para todas as colunas. Também analiso o tipo de cada variável, a quantidade e o percentual de valores ausentes e o número de valores únicos existentes. Essa análise permite identificar possíveis problemas de qualidade dos dados e fornece uma visão geral importante para orientar as etapas de tratamento, engenharia de atributos e modelagem.

In [ ]:
print("\nTabela de vendas:")
print("Dimensão:", df.shape)
display(df.head(3))

#### 5.2 Describe dos Dados

###### Nesta etapa, realizo uma análise exploratória inicial da base de treino para compreender sua estrutura e qualidade dos dados. Avalio a quantidade de registros e variáveis disponíveis, gero estatísticas descritivas para variáveis numéricas e categóricas e identifico possíveis problemas relacionados à qualidade dos dados. Também analiso os tipos das variáveis, a quantidade e o percentual de valores ausentes e o número de valores únicos de cada coluna. Essa análise fornece uma visão geral da base e apoia as decisões de tratamento, engenharia de atributos e modelagem realizadas nas etapas seguintes.

In [ ]:
print(f'Linhas: {df_treino.shape,}')
print(f'Colunas: {df_treino.shape[1]}')

display(df_treino.describe(include='all').T)

display(
    pd.DataFrame({
        'Tipo': df_treino.dtypes,
        'Nulos': df_treino.isnull().sum(),
        '% Nulos': (df_treino.isnull().mean()*100).round(2),
        'Únicos': df_treino.nunique()
    }).sort_values('% Nulos', ascending=False)

)

#### 5.3 Impacto dos Valores Ausentes na Fraude
###### Avaliando se o valor ausente carrega informação de fraude, ou seja, quando essa variável está nula, a taxa de fraude muda?
###### A análise mostra que os valores ausentes possuem poder preditivo para fraude. Nas variáveis b, c, d, m e g, registros com valores nulos apresentam taxas de fraude superiores à média da base (5%), indicando que a ausência da informação está associada a maior risco. Já a variável o apresentou comportamento inverso, com taxa de fraude significativamente maior quando preenchida. Esses resultados sugerem que os padrões de preenchimento são informativos e que a criação de indicadores de ausência (missing flags) pode contribuir para o desempenho do modelo.

In [ ]:
for col in ['b','c','d','m','g','o']:
    
    taxa = (
        df_treino.groupby(
            df_treino[col].isna()
        )['fraude']
        .mean()
    )

    print(col)
    print(taxa)

#### 5.4 Tratamento e Imputação de Dados
###### Em relação aos valores ausentes, a variável o apresenta alta proporção de registros nulos, as variáveis b e c apresentam proporção moderada, enquanto d, m, g, f e l possuem baixa incidência de nulos, indicando impacto reduzido na qualidade dos dados. Desse modo, alguns tratamentos serão comuns dada a classificação semântica feita anteriormente.
###### Considerando o tema em questão ("fraude") optei por não excluir a variável, pois ausência de informação costuma ser sinal.
###### Considerando a distribuição assimétrica comumente existente no prolbema em questão, optei por usar a mediana.
###### Realizei a remoção de stop words, espaço e caracters (Itens da etapa de pré processamento de mineração de texto)

In [16]:
# ============================================================
# 1. CONFIGURAÇÕES
# ============================================================

TARGET = 'fraude'

COLUNAS_NUMERICAS_IMPUTAR = [
    'b', 'c', 'd', 'm', 'f', 'l'
]

RANDOM_STATE = 42
N_SPLITS_OOF = 5

# Quanto maior, maior a regularização do target encoding.
# Ajuda a evitar riscos extremos em grupos com poucos registros.
SUAVIZACAO_RISCO = 20

# Produto precisa ter pelo menos esta quantidade para receber
# target encoding próprio. Caso contrário, recebe a média global.
MIN_FREQ_PRODUTO_RISCO = 20

CAMINHO_ARQUIVO = 'models/tratamento.pkl'


# ============================================================
# 2. STOP WORDS
# ============================================================

STOP_WORDS = {
    'a', 'ao', 'aos', 'as',
    'com', 'da', 'das', 'de', 'do', 'dos',
    'e', 'em', 'na', 'nas', 'no', 'nos',
    'o', 'os', 'ou', 'para', 'por',
    'um', 'uma', 'uns', 'umas'
}


# ============================================================
# 7. APRENDER PARÂMETROS NO TREINO COMPLETO
# ============================================================

params = aprender_parametros(df_treino)

print('Parâmetros aprendidos no treino.')
print(params.keys())


# ============================================================
# 8. SALVAR PARÂMETROS
# ============================================================

Path('models').mkdir(
    parents=True,
    exist_ok=True
)

pacote_tratamento = {
    'params': params,
    'stop_words': STOP_WORDS,
    'target': TARGET,
    'versao': '1.0'
}

with open(CAMINHO_ARQUIVO, 'wb') as arquivo:
    pickle.dump(
        pacote_tratamento,
        arquivo
    )

print(
    f'Arquivo salvo com sucesso em: '
    f'{CAMINHO_ARQUIVO}'
)


# ============================================================
# 9. CARREGAR PARÂMETROS NOVAMENTE
# ============================================================

with open(CAMINHO_ARQUIVO, 'rb') as arquivo:
    pacote_tratamento = pickle.load(arquivo)

params = pacote_tratamento['params']

print('Parâmetros carregados com sucesso.')


# ============================================================
# 10. TRANSFORMAR TREINO
# ============================================================

# Transformação geral usando os parâmetros completos
df_treino_tratado = aplicar_tratamento(
    df_treino,
    params
)

# ============================================================
# 11. TRANSFORMAR VALIDAÇÃO E TESTE
# ============================================================

df_validacao_tratado = aplicar_tratamento(
    df_validacao,
    params
)

df_teste_tratado = aplicar_tratamento(
    df_teste,
    params
)


constantes = [
    'o_missing',
    'b_missing',
    'c_missing',
    'd_missing',
    'm_missing',
    'f_missing',
    'l_missing',
    'g_missing',
    'monto_missing',
    'score_missing',
    'fecha_missing'
]

df_treino_tratado.drop(
    columns=constantes,
    inplace=True
)

df_validacao_tratado.drop(
    columns=constantes,
    inplace=True
)

df_teste_tratado.drop(
    columns=constantes,
    inplace=True
)

validar_base(
    'TREINO',
    df_treino_tratado
)

validar_base(
    'VALIDAÇÃO',
    df_validacao_tratado
)

validar_base(
    'TESTE',
    df_teste_tratado
)




Parâmetros aprendidos no treino.
dict_keys(['o_fill', 'g_fill', 'b_mediana', 'c_mediana', 'd_mediana', 'm_mediana', 'f_mediana', 'l_mediana', 'b_median', 'c_median', 'd_median', 'm_median', 'f_median', 'l_median', 'monto_mediana', 'score_mediana', 'p10', 'p90', 'fraude_media', 'produto_freq', 'categoria_freq', 'produto_risco', 'categoria_risco', 'pais_risco', 'suavizacao_risco', 'min_freq_produto_risco'])
Arquivo salvo com sucesso em: models/tratamento.pkl
Parâmetros carregados com sucesso.

TREINO
Shape: (105000, 44)

Total nulos: 0

Total infinitos: 0

Linhas duplicadas: 0

Colunas constantes: 0

Tipos:


,tipo,qtd
0,float64,15
1,int8,12
2,object,6
3,int64,5
4,int32,3
5,int16,2
6,datetime64[ns],1



Taxa fraude: 5.00%

VALIDAÇÃO
Shape: (22500, 44)

Total nulos: 0

Total infinitos: 0

Linhas duplicadas: 0

Colunas constantes: 0

Tipos:


,tipo,qtd
0,float64,15
1,int8,12
2,object,6
3,int64,5
4,int32,3
5,int16,2
6,datetime64[ns],1



Taxa fraude: 5.00%

TESTE
Shape: (22500, 44)

Total nulos: 0

Total infinitos: 0

Linhas duplicadas: 0

Colunas constantes: 0

Tipos:


,tipo,qtd
0,float64,15
1,int8,12
2,object,6
3,int64,5
4,int32,3
5,int16,2
6,datetime64[ns],1



Taxa fraude: 5.00%


#### 5.5 Salvando Dados Localmente 
###### Nesta etapa, salvo as bases de treino, teste e validação já tratadas no formato PKL para reduzir o tempo de processamento nas próximas execuções. Em seguida, libero da memória todos os DataFrames e Series utilizados durante as etapas anteriores, realizando a limpeza dos objetos carregados para reduzir o consumo de memória. Por fim, recarrego as bases salvas em disco para validar que os arquivos foram gravados corretamente e para continuar o projeto a partir de um ponto padronizado e reproduzível, sem precisar executar novamente todas as etapas anteriores de tratamento dos dados.

In [18]:
# ============================================================
# CRIAR PASTA BASES
# ============================================================

os.makedirs(
    "outputs/preprocessing",
    exist_ok=True
)

# ============================================================
# SALVAR EM PKL
# ============================================================

df_treino_tratado.to_pickle(
    "outputs/preprocessing/df_treino_tratado.pkl"
)

df_teste_tratado.to_pickle(
    "outputs/preprocessing/df_teste_tratado.pkl"
)

df_validacao_tratado.to_pickle(
    "outputs/preprocessing/df_validacao_tratado.pkl"
)

print("Arquivos salvos com sucesso.")

print(df_treino_tratado.shape)
print(df_teste_tratado.shape)
print(df_validacao_tratado.shape)

for nome in list(globals().keys()):

    obj = globals()[nome]

    if isinstance(
        obj,
        (pd.DataFrame, pd.Series)
    ):
        del globals()[nome]

gc.collect()


for nome, obj in globals().items():

    if isinstance(obj, pd.DataFrame):

        print(
            nome,
            obj.shape
        )

df_treino_tratado = pd.read_pickle(
    "outputs/preprocessing/df_treino_tratado.pkl"
)

df_teste_tratado = pd.read_pickle(
    "outputs/preprocessing/df_teste_tratado.pkl"
)

df_validacao_tratado = pd.read_pickle(
    "outputs/preprocessing/df_validacao_tratado.pkl"
)

Arquivos salvos com sucesso.
(105000, 44)
(22500, 44)
(22500, 44)


#### 5.6 Tipos de Variáveis
###### Nesta etapa, identifico automaticamente as variáveis numéricas e categóricas disponíveis na base de treino, excluindo apenas as colunas que não devem participar da modelagem, como o alvo de fraude e a data da transação. O objetivo é entender a composição dos dados após todas as etapas de tratamento e engenharia de atributos, facilitando a definição das estratégias de pré-processamento que serão aplicadas posteriormente a cada tipo de variável.

In [19]:
# Remover colunas que não entram no modelo
cols_excluir = [
    'fraude',
    'fecha',
]

# Numéricas
colunas_numericas = [
    c for c in df_treino_tratado.columns
    if c not in cols_excluir
    and pd.api.types.is_numeric_dtype(df_treino_tratado[c])
]

# Categóricas
colunas_categoricas = [
    c for c in df_treino_tratado.columns
    if c not in cols_excluir
    and not pd.api.types.is_numeric_dtype(df_treino_tratado[c])
]

print("Numéricas:", len(colunas_numericas))
print("Categóricas:", len(colunas_categoricas))

print("\nNuméricas:")
print(colunas_numericas)

print("\nCategóricas:")
print(colunas_categoricas)

Numéricas: 36
Categóricas: 6

Numéricas:
['a', 'b', 'c', 'd', 'e', 'f', 'h', 'k', 'l', 'm', 'n', 'monto', 'score', 'produto_tamanho', 'produto_qtd_palavras', 'produto_tem_numero', 'produto_freq', 'produto_risco', 'produto_raro', 'produto_muito_raro', 'freq_categoria', 'categoria_risco', 'categoria_rara', 'pais_risco', 'log_monto', 'monto_baixo', 'monto_alto', 'monto_por_palavra', 'dia', 'mes', 'hora', 'dia_semana', 'fim_semana', 'madrugada', 'hora_comercial', 'risco_medio_historico']

Categóricas:
['g', 'i', 'j', 'o', 'p', 'produto_limpo']


### 6. Grau de Importância & Seleção de Features & Clusterização

#### 6.1 Seleção supervisionada de variáveis
###### Aqui realizo a seleção automática de variáveis combinando relevância estatística e capacidade preditiva. Para evitar vazamento de informação, ajusto todo o pré-processamento utilizando apenas a base de treino e aplico exatamente o mesmo tratamento na base de teste. Em seguida, avalio a importância das variáveis por dois critérios complementares: Mutual Information, que mede a relação entre cada variável e o alvo de fraude, e LightGBM, que mede o ganho de informação proporcionado por cada variável durante o treinamento do modelo. Os dois critérios são normalizados e combinados sem a utilização de pesos arbitrários, gerando um ranking consolidado de importância. A partir desse ranking, seleciono automaticamente as variáveis mais relevantes e avalio seu desempenho na base de teste. Ao final, obtenho o ranking completo das variáveis, o conjunto de atributos selecionados, as métricas de desempenho do modelo e os artefatos necessários para reutilização nas próximas etapas do projeto.

In [21]:
# ============================================================
# SELEÇÃO DE VARIÁVEIS:
# PRÉ-PROCESSAMENTO + MUTUAL INFORMATION + LIGHTGBM
#
# 1. Ajusta o pré-processamento somente no treino;
# 2. Aplica o mesmo tratamento na teste;
# 3. Calcula Mutual Information nas variáveis originais;
# 4. Calcula importância por ganho usando LightGBM;
# 5. Combina os dois critérios sem pesos arbitrários;
# 6. Seleciona variáveis acima da mediana do score combinado.
# ============================================================


# ============================================================
# 1. CONFIGURAÇÕES
# ============================================================

TARGET = "fraude"

CATEGORICAS_INFORMADAS = [
    "a",
    "g",
    "h",
    "o"
]

BINARIAS_INFORMADAS = [
    "produto_tem_numero",
    "produto_raro",
    "produto_muito_raro",
    "categoria_rara",
    "monto_baixo",
    "monto_alto",
    "fim_semana",
    "madrugada",
    "hora_comercial",
    "n",
    "p"
]

COLUNAS_EXCLUIR = [
    "fraude",
    "fecha",
    "score",
    "monto_score",
    "score_x_monto",
    "i",
    "j",
    "produto_limpo"
]


# ============================================================
# 15. EXECUTAR O PROCESSO COMPLETO
# ============================================================

resultado_ranking = gerar_ranking_completo(
    df_treino=df_treino_tratado,
    df_teste=df_teste_tratado,
    target=TARGET,
    colunas_excluir=COLUNAS_EXCLUIR,
    categoricas_informadas=CATEGORICAS_INFORMADAS,
    binarias_informadas=BINARIAS_INFORMADAS,
    top_n=50,
    salvar_objetos=True
)


# ============================================================
# 16. RECUPERAR OS RESULTADOS
# ============================================================

ranking = resultado_ranking["ranking"]

variaveis_selecionadas = (
    resultado_ranking[
        "variaveis_selecionadas"
    ]
)

metricas_teste = (
    resultado_ranking[
        "metricas_teste"
    ]
)

importancia_transformada = (
    resultado_ranking[
        "importancia_transformada"
    ]
)

modelo_lgbm = (
    resultado_ranking[
        "modelo_lgbm"
    ]
)

preprocessador = (
    resultado_ranking[
        "preprocessador"
    ]
)

X_treino_processado = (
    resultado_ranking[
        "X_treino_processado"
    ]
)

X_teste_processado = (
    resultado_ranking[
        "X_teste_processado"
    ]
)

y_treino = resultado_ranking["y_treino"]

y_teste = (
    resultado_ranking[
        "y_teste"
    ]
)


# ============================================================
# 17. MOSTRAR O RANKING FINAL
# ============================================================

display(
    ranking[
        [
            "variavel",
            "mutual_information",
            "importancia",
            "qtd_features_geradas",
            "rank_mi",
            "rank_importancia",
            "ranking_medio",
            "mi_norm",
            "imp_norm",
            "score_combinado",
            "score_combinado_duplo_impacto"
        ]
    ]
)


# ============================================================
# 18. MOSTRAR VARIÁVEIS SELECIONADAS
# ============================================================

print(
    "\nQuantidade de variáveis selecionadas:",
    len(variaveis_selecionadas)
)

print(
    "\nVariáveis selecionadas:"
)

print(variaveis_selecionadas)


Colunas excluídas:
['fraude', 'fecha', 'score', 'monto_score', 'score_x_monto', 'i', 'j', 'produto_limpo']

Colunas faltantes na teste:
[]

Colunas extras na teste:
[]

Numéricas:
['b', 'c', 'd', 'e', 'f', 'k', 'l', 'm', 'monto', 'produto_tamanho', 'produto_qtd_palavras', 'produto_freq', 'produto_risco', 'freq_categoria', 'categoria_risco', 'pais_risco', 'log_monto', 'monto_por_palavra', 'dia', 'mes', 'hora', 'dia_semana', 'risco_medio_historico']

Binárias:
['produto_tem_numero', 'produto_raro', 'produto_muito_raro', 'categoria_rara', 'monto_baixo', 'monto_alto', 'fim_semana', 'madrugada', 'hora_comercial', 'n', 'p']

Categóricas:
['a', 'g', 'h', 'o']

Não classificadas e descartadas:
[]

Pré-processamento concluído.
Dimensão original do treino: (105000, 38)
Dimensão processada do treino: (105000, 117)
Dimensão processada da teste: (22500, 117)

DESEMPENHO LIGHTGBM NA teste
Precisão : 0.5876
Cobertura: 0.1520
F1       : 0.2415
AUC      : 0.7369

TOP 50 VARIÁVEIS ORIGINAIS
           

,variavel,mutual_information,importancia,qtd_features_geradas,rank_mi,rank_importancia,ranking_medio,mi_norm,imp_norm,score_combinado,score_combinado_duplo_impacto
0,risco_medio_historico,4.040189e-02,56905.180632,1,1.0,1.0,1.0,1.000000,1.000000,1.000000,1.000000e+00
1,o,2.763400e-02,50820.018352,3,3.0,2.0,2.5,0.683978,0.893065,0.788521,6.108366e-01
2,categoria_risco,3.836738e-02,29904.670071,1,2.0,3.0,2.5,0.949643,0.525518,0.737580,4.990542e-01
3,l,1.392557e-02,6888.958514,1,6.0,8.0,7.0,0.344676,0.121060,0.232868,4.172661e-02
4,m,9.251263e-03,10509.390769,1,11.0,4.0,7.5,0.228981,0.184682,0.206832,4.228877e-02
5,produto_risco,1.662992e-02,40.242569,1,4.0,37.0,20.5,0.411612,0.000707,0.206160,2.910867e-04
6,pais_risco,1.601995e-02,862.332250,1,5.0,26.0,15.5,0.396515,0.015154,0.205834,6.008725e-03
7,f,1.030215e-02,4712.264150,1,8.0,14.0,11.0,0.254992,0.082809,0.168900,2.111562e-02
8,freq_categoria,8.482786e-03,6518.262050,1,12.0,11.0,11.5,0.209960,0.114546,0.162253,2.405010e-02
9,mes,1.249348e-02,708.847600,1,7.0,27.0,17.0,0.309230,0.012457,0.160843,3.851969e-03



Quantidade de variáveis selecionadas: 19

Variáveis selecionadas:
['risco_medio_historico', 'o', 'categoria_risco', 'l', 'm', 'produto_risco', 'pais_risco', 'f', 'freq_categoria', 'mes', 'd', 'n', 'monto', 'b', 'monto_por_palavra', 'c', 'hora', 'produto_freq', 'log_monto']


#### 6.2 Criação dos Ranking dos Top's
###### Aqui utilizo o ranking de importância das variáveis para criar diferentes conjuntos de atributos candidatos à modelagem. São gerados cenários contendo os Top 5, Top 10, Top 15, Top 20 e o conjunto completo de variáveis, considerando tanto o score combinado quanto o score combinado de duplo impacto. Em seguida, identifico e removo automaticamente os conjuntos duplicados, garantindo que apenas combinações realmente distintas sejam avaliadas. O objetivo é reduzir cenários redundantes e concentrar os testes em subconjuntos de variáveis potencialmente mais relevantes para a identificação de fraudes.

In [22]:
numericas = ['b', 'c', 'd', 'e', 'f', 'k', 'l', 'm', 'monto', 'produto_tamanho', 'produto_qtd_palavras', 'produto_freq', 'produto_risco', 'freq_categoria', 'categoria_risco', 'pais_risco', 'log_monto', 'monto_por_palavra', 'dia', 'mes', 'hora', 'dia_semana', 'risco_medio_historico']

feature_sets = {

    'top5': ranking.sort_values(
        'score_combinado',
        ascending=False
    )['variavel'].head(5).tolist(),

    'top10': ranking.sort_values(
        'score_combinado',
        ascending=False
    )['variavel'].head(10).tolist(),

    'top15': ranking.sort_values(
        'score_combinado',
        ascending=False
    )['variavel'].head(15).tolist(),

    'top20': ranking.sort_values(
        'score_combinado',
        ascending=False
    )['variavel'].head(20).tolist(),

    'di_top5': ranking.sort_values(
        'score_combinado_duplo_impacto',
        ascending=False
    )['variavel'].head(5).tolist(),

    'di_top10': ranking.sort_values(
        'score_combinado_duplo_impacto',
        ascending=False
    )['variavel'].head(10).tolist(),

    'di_top15': ranking.sort_values(
        'score_combinado_duplo_impacto',
        ascending=False
    )['variavel'].head(15).tolist(),

    'di_top20': ranking.sort_values(
        'score_combinado_duplo_impacto',
        ascending=False
    )['variavel'].head(20).tolist(),

    'top_total': ranking['variavel'].tolist()
}

# ==========================================================
# REMOVER FEATURE SETS DUPLICADOS
# ==========================================================

feature_sets_unicos = {}
assinaturas = {}

for nome, features in feature_sets.items():

    assinatura = tuple(features)

    if assinatura not in assinaturas:

        assinaturas[assinatura] = nome
        feature_sets_unicos[nome] = features

    else:

        print(
            f'Removendo {nome} '
            f'(igual a {assinaturas[assinatura]})'
        )

feature_sets = feature_sets_unicos

#### 6.3 Seleção de Variáveis para Clusterização

###### Nesta etapa, preparo a base que será utilizada na clusterização considerando apenas os registros classificados como fraude. A partir do ranking de importância das variáveis, seleciono automaticamente os atributos com maior relevância estatística e preditiva, utilizando como critério a mediana do score combinado de duplo impacto. Em seguida, mantenho apenas as variáveis numéricas para compor o conjunto de atributos da clusterização. O objetivo é reduzir ruído, eliminar variáveis pouco relevantes e concentrar a análise nos fatores mais importantes para identificar padrões e perfis distintos de fraude.

In [23]:
df_treino_clusterizacao = df_treino_tratado[df_treino_tratado['fraude']==1].copy()

# calcula a mediana
limite = ranking['score_combinado_duplo_impacto'].median()

# seleciona variáveis acima da mediana
variaveis_cluster = (
    ranking.loc[
        ranking['score_combinado_duplo_impacto'] >= limite,
        'variavel'
    ]
    .tolist()
)

print(f'Mediana: {limite:.4f}')
print(f'Variáveis selecionadas: {len(variaveis_cluster)}')


features_cluster = [
    v
    for v in variaveis_selecionadas
    if v in numericas
]

print(features_cluster)
print(len(features_cluster))

Mediana: 0.0051
Variáveis selecionadas: 19
['risco_medio_historico', 'categoria_risco', 'l', 'm', 'produto_risco', 'pais_risco', 'f', 'freq_categoria', 'mes', 'd', 'monto', 'b', 'monto_por_palavra', 'c', 'hora', 'produto_freq', 'log_monto']
17


#### 6.4 Clusterização

###### Nesta etapa, construo e avalio uma solução de clusterização voltada para a identificação de perfis distintos de fraude. Inicialmente, realizo o tratamento das variáveis selecionadas, tratando valores ausentes, reduzindo o impacto de valores extremos e padronizando as escalas para garantir comparabilidade entre os atributos. Em seguida, avalio diferentes configurações de clusterização utilizando algoritmos como KMeans e, quando disponível, KMedoids, comparando os resultados por métricas de qualidade e estabilidade dos agrupamentos. A melhor configuração é selecionada com base no desempenho conjunto dessas métricas. Posteriormente, aplico a clusterização de forma consistente nas bases de treino, teste e validação, gerando informações como cluster atribuído e distância aos centroides. Por fim, salvo os artefatos necessários para reutilização do processo e produzo um resumo dos clusters identificados, permitindo analisar seu tamanho, representatividade e características médias, com o objetivo de compreender diferentes perfis de comportamento fraudulento presentes na base.

In [24]:
K_MIN = 2
K_MAX = 5

SEMENTES = [
    11,
    23,
    42,
    67,
    89
]

X_original = (
    df_treino_clusterizacao[
        features_cluster
    ]
    .copy()
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
)

imputer = SimpleImputer(
    strategy="median"
)

X_imputado = pd.DataFrame(
    imputer.fit_transform(
        X_original
    ),
    columns=features_cluster,
    index=X_original.index
)

limites_winsor = {}

for coluna in X_imputado.columns:

    p1 = X_imputado[coluna].quantile(0.01)
    p99 = X_imputado[coluna].quantile(0.99)

    limites_winsor[coluna] = {
        "p1": p1,
        "p99": p99
    }

    X_imputado[coluna] = (
        X_imputado[coluna]
        .clip(
            lower=p1,
            upper=p99
        )
    )

scaler = StandardScaler()

X_cluster = scaler.fit_transform(
    X_imputado
)


algoritmos = [
    "KMeans"
]

if KMEDOIDS_DISPONIVEL:

    algoritmos.append(
        "KMedoids"
    )

resultados = []

for algoritmo in algoritmos:

    for k in range(
        K_MIN,
        K_MAX + 1
    ):

        try:

            if algoritmo == "KMeans":

                modelo = KMeans(
                    n_clusters=k,
                    random_state=42,
                    n_init=30,
                    max_iter=500
                )

            else:

                modelo = KMedoids(
                    n_clusters=k,
                    random_state=42,
                    init="k-medoids++"
                )

            labels = modelo.fit_predict(
                X_cluster
            )

            sil = silhouette_score(
                X_cluster,
                labels
            )

            db = davies_bouldin_score(
                X_cluster,
                labels
            )

            ch = calinski_harabasz_score(
                X_cluster,
                labels
            )

            est = calcular_estabilidade(
                algoritmo,
                X_cluster,
                k
            )

            menor_cluster = (
                pd.Series(labels)
                .value_counts()
                .min()
            )

            resultados.append(
                {
                    "algoritmo": algoritmo,
                    "k": k,
                    "silhouette": sil,
                    "davies": db,
                    "calinski": ch,
                    "estabilidade": est,
                    "menor_cluster": menor_cluster,
                    "modelo": modelo
                }
            )

            print(
                f"{algoritmo} | "
                f"K={k} | "
                f"SIL={sil:.4f} | "
                f"DB={db:.4f} | "
                f"CH={ch:.2f} | "
                f"EST={est:.4f}"
            )

        except Exception as erro:

            print(
                algoritmo,
                k,
                erro
            )

resultados_modelos = pd.DataFrame(
    resultados
)

resultados_modelos["r_sil"] = (
    resultados_modelos["silhouette"]
    .rank(
        ascending=False
    )
)

resultados_modelos["r_db"] = (
    resultados_modelos["davies"]
    .rank(
        ascending=True
    )
)

resultados_modelos["r_ch"] = (
    resultados_modelos["calinski"]
    .rank(
        ascending=False
    )
)

resultados_modelos["r_est"] = (
    resultados_modelos["estabilidade"]
    .rank(
        ascending=False
    )
)

resultados_modelos["ranking_medio"] = (
    resultados_modelos[
        [
            "r_sil",
            "r_db",
            "r_ch",
            "r_est"
        ]
    ]
    .mean(axis=1)
)

resultados_modelos["prioridade"] = (
    np.where(
        resultados_modelos["algoritmo"] == "KMeans",
        0,
        1
    )
)

resultados_modelos = (
    resultados_modelos
    .sort_values(
        [
            "ranking_medio",
            "prioridade"
        ]
    )
    .reset_index(drop=True)
)

display(
    resultados_modelos[
        [
            "algoritmo",
            "k",
            "silhouette",
            "davies",
            "calinski",
            "estabilidade",
            "ranking_medio"
        ]
    ]
)

melhor_modelo = (
    resultados_modelos
    .iloc[0]["modelo"]
)

print(
    resultados_modelos.iloc[0]
)


df_treino_tratado = aplicar_clusterizacao(
    df_treino_tratado
)

df_teste_tratado = aplicar_clusterizacao(
    df_teste_tratado
)

df_validacao_tratado = aplicar_clusterizacao(
    df_validacao_tratado
)

print(
    df_treino_tratado.shape,
    df_teste_tratado.shape,
    df_validacao_tratado.shape
)

features_distancia = [
    coluna
    for coluna
    in df_treino_tratado.columns
    if coluna.startswith(
        "distancia_cluster_"
    )
]


features_automl_cluster = (
    features_cluster
    +
    ["cluster"]
    +
    features_distancia
    +
    [
        "menor_distancia_cluster",
        "distancia_cluster_atribuido"
    ]
)

artefatos_clusterizacao = {
    "imputer": imputer,
    "limites_winsor": limites_winsor,
    "scaler": scaler,
    "modelo": melhor_modelo,
    "features_cluster": features_cluster
}

with open(
    "models/artefatos_clusterizacao.pkl",
    "wb"
) as f:
    pickle.dump(
        artefatos_clusterizacao,
        f
    )

# =====================================================
# RESUMO DOS CLUSTERS
# =====================================================

resumo_clusters = (
    df_treino_tratado
    .groupby("cluster")[features_cluster]
    .mean()
    .reset_index()
)

# quantidade por cluster
qtd_cluster = (
    df_treino_tratado
    .groupby("cluster")
    .size()
    .reset_index(name="qtd")
)

# percentual da base
qtd_cluster["pct_base"] = (
    qtd_cluster["qtd"]
    / qtd_cluster["qtd"].sum()
)

# taxa de fraude
taxa_fraude = (
    df_treino_tratado
    .groupby("cluster")["fraude"]
    .mean()
    .reset_index(name="taxa_fraude")
)

# merge final
resumo_clusters = (
    qtd_cluster
    .merge(
        taxa_fraude,
        on="cluster"
    )
    .merge(
        resumo_clusters,
        on="cluster"
    )
)

resumo_clusters.to_csv('outputs/analytics/resumo_cluster.csv', sep = ',')

display(
    resumo_clusters
)

KMeans | K=2 | SIL=0.3470 | DB=1.5875 | CH=858.44 | EST=1.0000
KMeans | K=3 | SIL=0.3214 | DB=1.5427 | CH=837.02 | EST=0.9997
KMeans | K=4 | SIL=0.3159 | DB=1.2197 | CH=860.06 | EST=0.9981
KMeans | K=5 | SIL=0.2035 | DB=1.4976 | CH=866.66 | EST=0.9971
KMedoids | K=2 | SIL=0.0844 | DB=3.6307 | CH=310.53 | EST=0.0116
KMedoids | K=3 | SIL=0.1176 | DB=2.6175 | CH=580.54 | EST=0.1211
KMedoids | K=4 | SIL=0.0878 | DB=2.5895 | CH=526.06 | EST=0.2150
KMedoids | K=5 | SIL=0.1118 | DB=2.2349 | CH=601.94 | EST=0.2709


,algoritmo,k,silhouette,davies,calinski,estabilidade,ranking_medio
0,KMeans,2,0.346958,1.587465,858.439632,1.000000,2.25
1,KMeans,4,0.315913,1.219659,860.063623,0.998106,2.25
2,KMeans,3,0.321423,1.542662,837.015905,0.999701,2.75
3,KMeans,5,0.203462,1.497600,866.659003,0.997074,2.75
4,KMedoids,5,0.111762,2.234903,601.942121,0.270910,5.25
5,KMedoids,3,0.117559,2.617457,580.540677,0.121115,6.25
6,KMedoids,4,0.087845,2.589458,526.057853,0.215020,6.50
7,KMedoids,2,0.084386,3.630671,310.526707,0.011556,8.00


algoritmo                                                   KMeans
k                                                                2
silhouette                                                0.346958
davies                                                    1.587465
calinski                                                858.439632
estabilidade                                                   1.0
menor_cluster                                                  606
modelo           KMeans(max_iter=500, n_clusters=2, n_init=30, ...
r_sil                                                          1.0
r_db                                                           4.0
r_ch                                                           3.0
r_est                                                          1.0
ranking_medio                                                 2.25
prioridade                                                       0
Name: 0, dtype: object
(105000, 49) (22500, 49) (22500, 49)


,cluster,qtd,pct_base,taxa_fraude,risco_medio_historico,categoria_risco,l,m,produto_risco,pais_risco,...,freq_categoria,mes,d,monto,b,monto_por_palavra,c,hora,produto_freq,log_monto
0,0,2073,0.019743,0.292330,0.122987,0.252966,78.910275,3.503618,0.064852,0.051144,...,995.839363,3.491558,1.229137,55.125485,0.739926,6.143427,307741.253734,14.277376,2.743367,3.267956
1,1,102927,0.980257,0.045119,0.050000,0.049975,2350.464135,305.286718,0.050074,0.049953,...,20.778600,3.487404,22.071886,43.099873,0.730360,4.745263,237810.682374,14.120211,1.584686,3.110789


#### 6.5  Ranking das Variáveis com Variáveis de Clusterização

###### Nesta etapa, realizo uma nova seleção de variáveis utilizando as bases enriquecidas com as informações geradas pela clusterização. Para isso, incorporo o cluster atribuído como uma nova variável explicativa e avalio conjuntamente os atributos originais e as variáveis derivadas dos agrupamentos. Em seguida, aplico novamente os critérios de Mutual Information e importância por ganho do LightGBM para medir a relevância de cada variável na predição de fraude. Com base nesses resultados, gero um ranking consolidado de importância, permitindo identificar o impacto das informações de clusterização na capacidade preditiva do modelo e destacar os atributos mais relevantes para as próximas etapas de modelagem.

In [26]:
# ============================================================
# 1. CONFIGURAÇÕES
# ============================================================

TARGET = "fraude"

CATEGORICAS_INFORMADAS = [
    "a",
    "g",
    "h",
    "o",
	"cluster"
]

BINARIAS_INFORMADAS = [
    "produto_tem_numero",
    "produto_raro",
    "produto_muito_raro",
    "categoria_rara",
    "monto_baixo",
    "monto_alto",
    "fim_semana",
    "madrugada",
    "hora_comercial",
    "n",
    "p"
]

COLUNAS_EXCLUIR = [
    "fraude",
    "fecha",
    "score",
    "monto_score",
    "score_x_monto",
    "i",
    "j",
    "produto_limpo",
    "distancia_cluster_atribuido",
    "menor_distancia_cluster"	
]




# ============================================================
# 12. EXECUTAR TUDO
# ============================================================

resultado_ranking = gerar_ranking_completo(
    df_treino=df_treino_tratado,
    df_teste=df_teste_tratado,
    target=TARGET,
    colunas_excluir=COLUNAS_EXCLUIR,
    categoricas_informadas=CATEGORICAS_INFORMADAS,
    binarias_informadas=BINARIAS_INFORMADAS,
    top_n=50,
    salvar_objetos=True
)

# ============================================================
# 13. RECUPERAR OBJETOS
# ============================================================

ranking = resultado_ranking["ranking"]

importancia_transformada = (
    resultado_ranking[
        "importancia_transformada"
    ]
)

modelo_lgbm = (
    resultado_ranking["modelo_lgbm"]
)

preprocessador = (
    resultado_ranking["preprocessador"]
)

X_treino_processado = (
    resultado_ranking[
        "X_treino_processado"
    ]
)

y_treino = (
    resultado_ranking["y_treino"]
)


# ============================================================
# 14. MOSTRAR RANKING SEM ENCODING
# ============================================================

display(
    ranking[
        [
            "variavel",
            "mutual_information",
            "importancia",
            "qtd_features_geradas",
            "rank_mi",
            "rank_importancia",
            "ranking_medio",
            "mi_norm",
            "imp_norm",
            "score_combinado",
            "score_combinado_duplo_impacto"
        ]
    ]
)


Colunas excluídas:
['fraude', 'fecha', 'score', 'monto_score', 'score_x_monto', 'i', 'j', 'produto_limpo', 'distancia_cluster_atribuido', 'menor_distancia_cluster']

Colunas faltantes na teste:
[]

Colunas extras na teste:
[]

Numéricas:
['b', 'c', 'd', 'e', 'f', 'k', 'l', 'm', 'monto', 'produto_tamanho', 'produto_qtd_palavras', 'produto_freq', 'produto_risco', 'freq_categoria', 'categoria_risco', 'pais_risco', 'log_monto', 'monto_por_palavra', 'dia', 'mes', 'hora', 'dia_semana', 'risco_medio_historico', 'distancia_cluster_0', 'distancia_cluster_1']

Binárias:
['produto_tem_numero', 'produto_raro', 'produto_muito_raro', 'categoria_rara', 'monto_baixo', 'monto_alto', 'fim_semana', 'madrugada', 'hora_comercial', 'n', 'p']

Categóricas:
['a', 'g', 'h', 'o', 'cluster']

Não classificadas e descartadas:
[]

Pré-processamento concluído.
Dimensão original do treino: (105000, 41)
Dimensão processada do treino: (105000, 121)
Dimensão processada da teste: (22500, 121)

DESEMPENHO LIGHTGBM NA te

,variavel,mutual_information,importancia,qtd_features_geradas,rank_mi,rank_importancia,ranking_medio,mi_norm,imp_norm,score_combinado,score_combinado_duplo_impacto
0,risco_medio_historico,4.125919e-02,55073.249729,1,1.0,1.0,1.0,1.000000,1.000000,1.000000,1.000000e+00
1,o,2.763400e-02,51216.294646,3,3.0,2.0,2.5,0.669766,0.929967,0.799866,6.228601e-01
2,categoria_risco,3.877885e-02,29444.713776,1,2.0,3.0,2.5,0.939884,0.534646,0.737265,5.025055e-01
3,distancia_cluster_0,1.870599e-02,5301.481748,1,4.0,15.0,9.5,0.453378,0.096262,0.274820,4.364320e-02
4,l,1.393820e-02,6692.064099,1,7.0,9.0,8.0,0.337821,0.121512,0.229666,4.104928e-02
5,produto_risco,1.686567e-02,55.489630,1,5.0,38.0,21.5,0.408774,0.001008,0.204891,4.118642e-04
6,m,9.321587e-03,8891.226358,1,12.0,4.0,8.0,0.225928,0.161444,0.193686,3.647457e-02
7,pais_risco,1.474456e-02,890.824236,1,6.0,28.0,17.0,0.357364,0.016175,0.186770,5.780461e-03
8,freq_categoria,9.529980e-03,6218.452666,1,11.0,11.0,11.0,0.230978,0.112912,0.171945,2.608032e-02
9,f,1.103214e-02,4159.128442,1,9.0,18.0,13.5,0.267386,0.075520,0.171453,2.019300e-02


#### 6.6  Seleção do Conjunto de Features Finais

###### Nesta etapa, utilizo o ranking de importância obtido após a inclusão das variáveis de clusterização para criar diferentes conjuntos de atributos candidatos à modelagem. São gerados cenários contendo os Top 5, Top 10, Top 15, Top 20 e o conjunto completo de variáveis mais relevantes segundo os critérios de score combinado e score combinado de duplo impacto. Em seguida, removo automaticamente os conjuntos duplicados e consolido os feature sets gerados antes e depois da clusterização em uma única estrutura. O objetivo é construir um conjunto abrangente de cenários de modelagem, eliminando redundâncias e permitindo avaliar de forma consistente o impacto das variáveis de clusterização na capacidade preditiva dos modelos de fraude.

In [28]:
feature_sets_cluster = {

    'top5_cluster': ranking.sort_values(
        'score_combinado',
        ascending=False
    )['variavel'].head(5).tolist(),

    'top10_cluster': ranking.sort_values(
        'score_combinado',
        ascending=False
    )['variavel'].head(10).tolist(),

    'top15_cluster': ranking.sort_values(
        'score_combinado',
        ascending=False
    )['variavel'].head(15).tolist(),

    'top20_cluster': ranking.sort_values(
        'score_combinado',
        ascending=False
    )['variavel'].head(20).tolist(),

    'di_top5_cluster': ranking.sort_values(
        'score_combinado_duplo_impacto',
        ascending=False
    )['variavel'].head(5).tolist(),

    'di_top10_cluster': ranking.sort_values(
        'score_combinado_duplo_impacto',
        ascending=False
    )['variavel'].head(10).tolist(),

    'di_top15_cluster': ranking.sort_values(
        'score_combinado_duplo_impacto',
        ascending=False
    )['variavel'].head(15).tolist(),

    'di_top20_cluster': ranking.sort_values(
        'score_combinado_duplo_impacto',
        ascending=False
    )['variavel'].head(20).tolist(),

    'top_total_cluster': ranking['variavel'].tolist()
}

# ==========================================================
# REMOVER FEATURE SETS DUPLICADOS
# ==========================================================

feature_sets_unicos = {}
assinaturas = {}

for nome, features in feature_sets_cluster.items():

    assinatura = tuple(features)

    if assinatura not in assinaturas:

        assinaturas[assinatura] = nome
        feature_sets_unicos[nome] = features

    else:

        print(
            f'Removendo {nome} '
            f'(igual a {assinaturas[assinatura]})'
        )

feature_sets_cluster = feature_sets_unicos

# ==========================================================
# JUNTA OS DOIS DICIONÁRIOS
# ==========================================================

feature_sets_final = {
    **feature_sets,
    **feature_sets_cluster
}

# ==========================================================
# REMOVER FEATURE SETS DUPLICADOS
# ==========================================================

feature_sets_unicos = {}
assinaturas = {}

for nome, features in feature_sets_final.items():

    assinatura = tuple(features)

    if assinatura not in assinaturas:

        feature_sets_unicos[nome] = features
        assinaturas[assinatura] = nome

    else:

        print(
            f"Removendo {nome} "
            f"(igual a {assinaturas[assinatura]})"
        )

feature_sets_final = feature_sets_unicos

Removendo di_top5_cluster (igual a top5_cluster)


#### 6.7 Salvando Dados

###### Nesta etapa, salvo as bases de treino, teste e validação já tratadas, juntamente com o conjunto final de feature sets definidos ao longo do processo de seleção de variáveis. O objetivo é garantir a reprodutibilidade da solução e facilitar futuras execuções, permitindo reutilizar diretamente os dados preparados e os conjuntos de atributos selecionados sem a necessidade de repetir as etapas de tratamento, engenharia de atributos, clusterização e seleção de variáveis.

In [29]:
# =====================================================
# Criar pasta
# =====================================================
os.makedirs("bases", exist_ok=True)

# =====================================================
# Salvar dataframes
# =====================================================
df_treino_tratado.to_pickle(
    "outputs/preprocessing/df_treino_tratado.pkl"
)

df_teste_tratado.to_pickle(
    "outputs/preprocessing/df_teste_tratado.pkl"
)

df_validacao_tratado.to_pickle(
    "outputs/preprocessing/df_validacao_tratado.pkl"
)

# =====================================================
# Salvar feature_sets
# =====================================================
with open(
    "outputs/modeling/feature_sets_final.pkl",
    "wb"
) as f:
    pickle.dump(
        feature_sets_final,
        f
    )

print("✅ Arquivos salvos:")
print("outputs/preprocessing/df_treino_tratado.pkl")
print("outputs/preprocessing/df_teste_tratado.pkl")
print("outputs/preprocessing/df_validacao_tratado.pkl")
print("outputs/modeling/feature_sets_final.pkl")

✅ Arquivos salvos:
outputs/preprocessing/df_treino_tratado.pkl
outputs/preprocessing/df_teste_tratado.pkl
outputs/preprocessing/df_validacao_tratado.pkl
outputs/modeling/feature_sets_final.pkl


### 7. Modelagem & Análise de Desempenho

#### 7.1 Carga de Dados

###### Nesta etapa, recarrego as bases de treino, teste e validação previamente tratadas, juntamente com o conjunto final de feature sets definidos ao longo do projeto. O objetivo é retomar a análise a partir dos dados já preparados e das configurações já consolidadas, evitando a necessidade de executar novamente todas as etapas de tratamento, engenharia de atributos, seleção de variáveis e clusterização. Dessa forma, garanto maior agilidade, reprodutibilidade e consistência nas próximas etapas da modelagem e avaliação.

In [31]:
df_treino_tratado = pd.read_pickle(
    "outputs/preprocessing/df_treino_tratado.pkl"
)

df_teste_tratado = pd.read_pickle(
    "outputs/preprocessing/df_teste_tratado.pkl"
)

df_validacao_tratado = pd.read_pickle(
    "outputs/preprocessing/df_validacao_tratado.pkl"
)

with open(
    "outputs/modeling/feature_sets_final.pkl",
    "rb"
) as f:
    feature_sets_final = pickle.load(f)

#### 7.2 Desempenho Atual & Seleção Aleatória
###### Nesta etapa, comparo o desempenho do score atual com um cenário totalmente aleatório para validar sua capacidade de identificar fraudes. Para isso, avalio ambos os cenários considerando apenas os 10% registros de maior risco da base de teste e calculo métricas como precisão, cobertura, F1, AUC, Lift e erros por acerto. Enquanto o score atual utiliza a priorização real definida pelo negócio, o cenário aleatório seleciona registros sem qualquer critério preditivo. Ao final, comparo os resultados obtidos para quantificar o ganho efetivo do score atual na concentração e identificação de fraudes em relação a uma seleção puramente aleatória.

In [32]:
resultado_score_teste = avaliar_top_percent(
    df_teste_tratado,
    coluna_score='score',
    target='fraude',
    pct=0.1
)

df_random_teste = df_teste_tratado.copy()

np.random.seed(42)

df_random_teste['score_random'] = np.random.rand(
    len(df_random_teste)
)

resultado_random_teste = avaliar_top_percent(
    df_random_teste,
    coluna_score='score_random',
    target='fraude',
    pct=0.1
)

comparativo = pd.DataFrame([
    {
        'Cenario':'Score',
        **resultado_score_teste
    },
    {
        'Cenario':'Aleatorio',
        **resultado_random_teste
    }
])

display(comparativo)

,Cenario,Total Base,Total Fraudes,Precisão,Cobertura,F1,AUC,Erro_por_Acerto,Lift,Selecionados,Fraudes Encontradas,Threshold
0,Score,22500,1125,0.2120,0.4240,0.2827,0.7226,3.72,4.24,2250,477,88.00000
1,Aleatorio,22500,1125,0.0596,0.1191,0.0794,0.4958,15.79,1.19,2250,134,0.90088


#### 7.3 Modelo Final de Classificação com GridSearch

###### Nesta etapa, preparo as bases de treino e teste, tratando as variáveis de acordo com seus tipos numéricos, categóricos e binários. Em seguida, treino e comparo diferentes modelos e conjuntos de variáveis, utilizando validação cruzada e otimização de hiperparâmetros com foco em Average Precision. Avalio o desempenho no teste considerando os 10% registros de maior risco e métricas como precisão, cobertura, F1, AUC, KS, Lift e erros por acerto. Ao final, seleciono a melhor combinação de modelo e features, apresento seus resultados e salvo os artefatos necessários para reutilização nas etapas seguintes.

In [33]:
# ============================================================
# 1. CONFIGURAÇÕES
# ============================================================

TARGET = "fraude"

RANDOM_STATE = 42
N_SPLITS_CV = 2
N_JOBS = -1

PROPORCAO_TOP = 0.10
SCORING_GRID = "average_precision"

INCLUIR_TODAS_FEATURES = False


CATEGORICAS_INFORMADAS = [
    "a",
    "g",
    "h",
    "o",
    "cluster"
]


BINARIAS_INFORMADAS = [
    "produto_tem_numero",
    "produto_raro",
    "produto_muito_raro",
    "categoria_rara",
    "monto_baixo",
    "monto_alto",
    "fim_semana",
    "madrugada",
    "hora_comercial",
    "n",
    "p"
]


COLUNAS_NAO_MODELAVEIS = [
    TARGET,
    "fecha",
    "i",
    "produto_limpo",
    "monto_score",
    "score_x_monto",
    "score"
]


# ============================================================
# 2. PREPARAR CÓPIAS DAS BASES
# ============================================================

treino = df_treino_tratado.copy()
teste = df_teste_tratado.copy()


# ============================================================
# 3. PADRONIZAR AS BASES
# ============================================================

for nome_base, df_aux in [
    ("treino", treino),
    ("teste", teste)
]:

    df_aux.columns = (
        df_aux.columns
        .astype(str)
        .str.strip()
    )

    if TARGET not in df_aux.columns:
        raise ValueError(
            f"A coluna '{TARGET}' não foi encontrada na base {nome_base}."
        )

    # --------------------------------------------------------
    # Target
    # --------------------------------------------------------

    df_aux[TARGET] = pd.to_numeric(
        df_aux[TARGET],
        errors="coerce"
    )

    # --------------------------------------------------------
    # p: binária N/Y ou 0/1
    # --------------------------------------------------------

    if "p" in df_aux.columns:

        p_original = (
            df_aux["p"]
            .astype("string")
            .str.strip()
            .str.upper()
        )

        valores_validos = {"N", "Y", "0", "1"}

        valores_invalidos = (
            set(p_original.dropna().unique())
            - valores_validos
        )

        if valores_invalidos:
            raise ValueError(
                f"Valores inválidos em 'p' na base {nome_base}: "
                f"{sorted(valores_invalidos)}"
            )

        df_aux["p"] = p_original.map({
            "N": 0,
            "Y": 1,
            "0": 0,
            "1": 1
        })

    # --------------------------------------------------------
    # o: categórica nominal
    # --------------------------------------------------------

    if "o" in df_aux.columns:

        df_aux["o"] = (
            df_aux["o"]
            .astype("string")
            .str.strip()
            .str.upper()
            .fillna("NA")
        )

    # --------------------------------------------------------
    # Cluster: categórica
    # --------------------------------------------------------

    if "cluster" in df_aux.columns:

        df_aux["cluster"] = (
            df_aux["cluster"]
            .astype("string")
            .fillna("DESCONHECIDO")
        )


# ============================================================
# 4. REMOVER LINHAS SEM TARGET
# ============================================================

treino = (
    treino
    .dropna(subset=[TARGET])
    .reset_index(drop=True)
)

teste = (
    teste
    .dropna(subset=[TARGET])
    .reset_index(drop=True)
)

treino[TARGET] = treino[TARGET].astype(int)
teste[TARGET] = teste[TARGET].astype(int)


# ============================================================
# 5. SEPARAR X E y
# ============================================================

colunas_remover_features = [
    coluna
    for coluna in COLUNAS_NAO_MODELAVEIS
    if coluna != TARGET
]


X_treino = treino.drop(
    columns=colunas_remover_features + [TARGET],
    errors="ignore"
).copy()

y_treino = treino[TARGET].copy()


X_teste = teste.drop(
    columns=colunas_remover_features + [TARGET],
    errors="ignore"
).copy()

y_teste = teste[TARGET].copy()


# Usar o treino como referência de colunas
X_teste = X_teste.reindex(
    columns=X_treino.columns
)


print("=" * 80)
print("BASES")
print("=" * 80)

print("Treino:", X_treino.shape)
print("Teste:", X_teste.shape)

print(
    f"Taxa de fraude no treino: "
    f"{y_treino.mean():.2%}"
)

print(
    f"Taxa de fraude no teste: "
    f"{y_teste.mean():.2%}"
)



feature_sets_validos = validar_feature_sets(
    feature_sets=feature_sets_final,
    X=X_treino,
    incluir_todas=INCLUIR_TODAS_FEATURES
)


print("\n" + "=" * 80)
print("FEATURE SETS VÁLIDOS")
print("=" * 80)

for nome_set, features in feature_sets_validos.items():

    print(
        f"{nome_set}: {len(features)} variáveis"
    )


# ============================================================
# 13. EXECUTAR
# ============================================================

(
    comparativo_desempenho,
    modelos_treinados,
    melhor_resultado
) = executar_classificacao(
    X_treino=X_treino,
    y_treino=y_treino,
    X_teste=X_teste,
    y_teste=y_teste,
    feature_sets=feature_sets_validos
)


# ============================================================
# 14. RESULTADO COMPLETO
# ============================================================

print("\n" + "=" * 100)
print("COMPARATIVO FINAL NO TESTE")
print("=" * 100)

display(
    comparativo_desempenho[
        [
            "Feature Set",
            "Modelo",
            "Qtd Features Originais",
            "Qtd Numéricas",
            "Qtd Binárias",
            "Qtd Categóricas",
            "CV Average Precision",
            "Total Base",
            "Total Fraudes",
            "Selecionados",
            "TP",
            "FP",
            "FN",
            "TN",
            "Precisão",
            "Cobertura",
            "F1",
            "AUC",
            "Average Precision",
            "KS",
            "Balanced Accuracy",
            "Erro_por_Acerto",
            "Lift",
            "Threshold",
            "Melhores Parâmetros"
        ]
    ]
)


# ============================================================
# 15. MELHOR RESULTADO
# ============================================================

melhor_modelo = melhor_resultado["modelo"]
melhores_features = melhor_resultado["features"]
melhor_feature_set = melhor_resultado["feature_set"]

print("\n" + "=" * 100)
print("MELHOR MODELO")
print("=" * 100)

print(
    "Feature set:",
    melhor_feature_set
)

print(
    "Modelo:",
    melhor_modelo.named_steps["clf"].__class__.__name__
)

print(
    "Quantidade de features:",
    len(melhores_features)
)

print(
    "Features:",
    melhores_features
)

print(
    "Melhores parâmetros:",
    melhor_resultado["melhores_parametros"]
)

print(
    "\nMétricas no teste:"
)

for metrica, valor in (
    melhor_resultado["metricas_teste"].items()
):

    if isinstance(valor, float):
        print(f"{metrica}: {valor:.4f}")
    else:
        print(f"{metrica}: {valor}")


# ============================================================
# 16. SALVAR O MELHOR MODELO
# ============================================================

joblib.dump(
    melhor_modelo,
    "models/melhor_modelo_feature_sets.pkl"
)

joblib.dump(
    melhores_features,
    "models/melhores_features_feature_sets.pkl"
)

comparativo_desempenho.to_csv(
    "outputs/modeling/comparativo_feature_sets.csv",
    index=False,
    sep=";"
)

print("\nArquivos salvos:")
print("melhor_modelo_feature_sets.pkl")
print("melhores_features_feature_sets.pkl")
print("comparativo_feature_sets.csv")

BASES
Treino: (105000, 44)
Teste: (22500, 44)
Taxa de fraude no treino: 5.00%
Taxa de fraude no teste: 5.00%

FEATURE SETS VÁLIDOS
top5: 5 variáveis
top10: 10 variáveis
top15: 15 variáveis
top20: 20 variáveis
di_top5: 5 variáveis
di_top10: 10 variáveis
di_top15: 15 variáveis
di_top20: 20 variáveis
top_total: 38 variáveis
top5_cluster: 5 variáveis
top10_cluster: 10 variáveis
top15_cluster: 15 variáveis
top20_cluster: 20 variáveis
di_top10_cluster: 10 variáveis
di_top15_cluster: 15 variáveis
di_top20_cluster: 20 variáveis
top_total_cluster: 41 variáveis

FEATURE SET: top5
Total original: 5 | Numéricas: 4 | Binárias: 0 | Categóricas: 1
➡️ LogisticRegression | F1: 0.3153 | Precisão: 0.2364 | Cobertura: 0.4729 | AUC: 0.8184 | AP: 0.2445 | Erros/Acerto: 3.23
➡️ DecisionTree | F1: 0.2696 | Precisão: 0.2022 | Cobertura: 0.4044 | AUC: 0.6977 | AP: 0.2015 | Erros/Acerto: 3.95
➡️ RandomForest | F1: 0.2898 | Precisão: 0.2173 | Cobertura: 0.4347 | AUC: 0.7396 | AP: 0.2262 | Erros/Acerto: 3.60
➡️ XG

,Feature Set,Modelo,Qtd Features Originais,Qtd Numéricas,Qtd Binárias,Qtd Categóricas,CV Average Precision,Total Base,Total Fraudes,Selecionados,...,Cobertura,F1,AUC,Average Precision,KS,Balanced Accuracy,Erro_por_Acerto,Lift,Threshold,Melhores Parâmetros
0,top_total,LogisticRegression,38,23,11,4,0.359573,22500,1125,2250,...,0.502222,0.334815,0.829582,0.288097,0.509942,0.711696,2.982301,5.022222,0.124034,"{'clf__C': 0.1, 'clf__solver': 'liblinear'}"
1,top_total_cluster,RandomForest,41,25,11,5,0.435609,22500,1125,2250,...,0.488889,0.325926,0.791270,0.313537,0.449637,0.704678,3.090909,4.888889,0.150000,"{'clf__max_depth': None, 'clf__n_estimators': ..."
2,top_total_cluster,LogisticRegression,41,25,11,5,0.381361,22500,1125,2250,...,0.488000,0.325333,0.818648,0.277486,0.494690,0.704211,3.098361,4.880000,0.126626,"{'clf__C': 0.1, 'clf__solver': 'liblinear'}"
3,top_total,RandomForest,38,23,11,4,0.440196,22500,1125,2250,...,0.485333,0.323556,0.792708,0.315696,0.453848,0.702807,3.120879,4.853333,0.150000,"{'clf__max_depth': None, 'clf__n_estimators': ..."
4,di_top20,LogisticRegression,20,17,1,2,0.337494,22500,1125,2250,...,0.480889,0.320593,0.825496,0.266425,0.507322,0.700468,3.158965,4.808889,0.113791,"{'clf__C': 0.1, 'clf__solver': 'liblinear'}"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80,di_top10_cluster,DecisionTree,10,9,0,1,0.335901,22500,1125,2250,...,0.408889,0.272593,0.712864,0.210915,0.406830,0.662573,3.891304,4.088889,0.133185,"{'clf__max_depth': 5, 'clf__min_samples_split'..."
81,top10_cluster,DecisionTree,10,9,0,1,0.334082,22500,1125,2250,...,0.408000,0.272000,0.712228,0.200022,0.407298,0.662105,3.901961,4.080000,0.127617,"{'clf__max_depth': 5, 'clf__min_samples_split'..."
82,top5,DecisionTree,5,4,0,1,0.328798,22500,1125,2250,...,0.404444,0.269630,0.697709,0.201458,0.404772,0.660234,3.945055,4.044444,0.136101,"{'clf__max_depth': 5, 'clf__min_samples_split'..."
83,di_top5,DecisionTree,5,4,0,1,0.328798,22500,1125,2250,...,0.404444,0.269630,0.697709,0.201458,0.404772,0.660234,3.945055,4.044444,0.136101,"{'clf__max_depth': 5, 'clf__min_samples_split'..."



MELHOR MODELO
Feature set: top_total
Modelo: LogisticRegression
Quantidade de features: 38
Features: ['risco_medio_historico', 'o', 'categoria_risco', 'l', 'm', 'produto_risco', 'pais_risco', 'f', 'freq_categoria', 'mes', 'd', 'n', 'monto', 'b', 'monto_por_palavra', 'c', 'hora', 'produto_freq', 'log_monto', 'p', 'e', 'produto_tamanho', 'produto_qtd_palavras', 'h', 'a', 'k', 'dia_semana', 'dia', 'categoria_rara', 'g', 'monto_alto', 'madrugada', 'produto_muito_raro', 'produto_raro', 'hora_comercial', 'monto_baixo', 'produto_tem_numero', 'fim_semana']
Melhores parâmetros: {'clf__C': 0.1, 'clf__solver': 'liblinear'}

Métricas no teste:
Total Base: 22500
Total Fraudes: 1125
Selecionados: 2250
TP: 565
FP: 1685
FN: 560
TN: 19690
Precisão: 0.2511
Cobertura: 0.5022
F1: 0.3348
AUC: 0.8296
Average Precision: 0.2881
KS: 0.5099
Balanced Accuracy: 0.7117
Erro_por_Acerto: 2.9823
Lift: 5.0222
Threshold: 0.1240

Arquivos salvos:
melhor_modelo_feature_sets.pkl
melhores_features_feature_sets.pkl
compara

#### 7.4 Validação Out-of-Sample do Melhor Modelo de Fraude e Comparação com o Ambiente de Teste

###### Aplico o melhor modelo previamente treinado na base de validação, garantindo que as mesmas features utilizadas durante o treinamento estejam disponíveis e na mesma ordem. Em seguida, calculo a probabilidade de fraude de cada transação e classifico os registros de acordo com o Top 10% de maior risco. A partir dessas probabilidades, avalio o desempenho do modelo utilizando métricas como precisão, cobertura, F1, AUC, Average Precision, KS, Lift e erros por acerto. Também comparo os resultados obtidos na validação com aqueles observados durante o teste para verificar a estabilidade e a capacidade de generalização do modelo. Por fim, gero uma base detalhada contendo as probabilidades previstas, ranking de risco, indicação das transações selecionadas no Top 10% e os respectivos acertos e erros de classificação para apoiar análises posteriores.

In [34]:
# ------------------------------------------------------------
# 1. Carregar modelo e features salvas
# ------------------------------------------------------------

melhor_modelo_carregado = joblib.load(
    "models/melhor_modelo_feature_sets.pkl"
)

melhores_features_carregadas = joblib.load(
    "models/melhores_features_feature_sets.pkl"
)


# ------------------------------------------------------------
# 2. Preparar base de validação
# ------------------------------------------------------------

validacao = df_validacao_tratado.copy()

validacao.columns = (
    validacao.columns
    .astype(str)
    .str.strip()
)


# ------------------------------------------------------------
# 3. Validar target
# ------------------------------------------------------------

if TARGET not in validacao.columns:
    raise ValueError(
        f"A coluna target '{TARGET}' não foi encontrada "
        "em df_validacao_tratado."
    )

validacao[TARGET] = pd.to_numeric(
    validacao[TARGET],
    errors="coerce"
)


# ------------------------------------------------------------
# 4. Padronizar variável p
# ------------------------------------------------------------

if "p" in validacao.columns:

    p_original = (
        validacao["p"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    valores_validos = {"N", "Y", "0", "1"}

    valores_invalidos = (
        set(p_original.dropna().unique())
        - valores_validos
    )

    if valores_invalidos:
        raise ValueError(
            "Valores inválidos em 'p' na validação: "
            f"{sorted(valores_invalidos)}"
        )

    validacao["p"] = p_original.map({
        "N": 0,
        "Y": 1,
        "0": 0,
        "1": 1
    })


# ------------------------------------------------------------
# 5. Padronizar variável o
# ------------------------------------------------------------

if "o" in validacao.columns:

    validacao["o"] = (
        validacao["o"]
        .astype("string")
        .str.strip()
        .str.upper()
        .fillna("NA")
    )


# ------------------------------------------------------------
# 6. Padronizar cluster
# ------------------------------------------------------------

if "cluster" in validacao.columns:

    validacao["cluster"] = (
        validacao["cluster"]
        .astype("string")
        .fillna("DESCONHECIDO")
    )


# ------------------------------------------------------------
# 7. Remover registros sem target
# ------------------------------------------------------------

validacao = (
    validacao
    .dropna(subset=[TARGET])
    .reset_index(drop=True)
)

validacao[TARGET] = (
    validacao[TARGET]
    .astype(int)
)


# ------------------------------------------------------------
# 8. Separar X e y
# ------------------------------------------------------------

X_validacao = validacao.drop(
    columns=COLUNAS_NAO_MODELAVEIS,
    errors="ignore"
).copy()

y_validacao = validacao[TARGET].copy()


# ------------------------------------------------------------
# 9. Verificar features ausentes
# ------------------------------------------------------------

features_ausentes = [
    coluna
    for coluna in melhores_features_carregadas
    if coluna not in X_validacao.columns
]

if features_ausentes:

    raise ValueError(
        "As seguintes features utilizadas pelo melhor modelo "
        "não foram encontradas na validação:\n"
        f"{features_ausentes}"
    )


# Manter somente as features usadas pelo melhor modelo
# e exatamente na mesma ordem do treinamento

X_validacao_modelo = X_validacao.reindex(
    columns=melhores_features_carregadas
)


# ------------------------------------------------------------
# 10. Gerar probabilidades na validação
# ------------------------------------------------------------

probabilidades_validacao = (
    melhor_modelo_carregado
    .predict_proba(
        X_validacao_modelo
    )[:, 1]
)


# ------------------------------------------------------------
# 11. Gerar predição pelo Top 10%
# ------------------------------------------------------------

(
    predicao_validacao,
    quantidade_top_validacao,
    threshold_validacao
) = predicao_top_percentual(
    scores=probabilidades_validacao,
    proporcao_top=PROPORCAO_TOP
)


# ------------------------------------------------------------
# 12. Calcular métricas na validação
# ------------------------------------------------------------

metricas_validacao = calcular_metricas(
    y_real=y_validacao,
    scores=probabilidades_validacao,
    proporcao_top=PROPORCAO_TOP
)


# ------------------------------------------------------------
# 13. Criar base de validação com resultados
# ------------------------------------------------------------

resultado_validacao = validacao.copy()

resultado_validacao[
    "probabilidade_modelo"
] = probabilidades_validacao

resultado_validacao[
    "predicao_top_percentual"
] = predicao_validacao

resultado_validacao[
    "selecionado_top"
] = (
    resultado_validacao[
        "predicao_top_percentual"
    ] == 1
)

resultado_validacao[
    "acerto_fraude"
] = (
    (
        resultado_validacao[TARGET] == 1
    )
    & (
        resultado_validacao[
            "predicao_top_percentual"
        ] == 1
    )
).astype(int)

resultado_validacao[
    "erro_selecao"
] = (
    (
        resultado_validacao[TARGET] == 0
    )
    & (
        resultado_validacao[
            "predicao_top_percentual"
        ] == 1
    )
).astype(int)

resultado_validacao[
    "ranking_score"
] = (
    resultado_validacao[
        "probabilidade_modelo"
    ]
    .rank(
        method="first",
        ascending=False
    )
    .astype(int)
)

resultado_validacao = (
    resultado_validacao
    .sort_values(
        "probabilidade_modelo",
        ascending=False
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 14. Criar comparativo teste x validação
# ------------------------------------------------------------

comparativo_teste_validacao = pd.DataFrame([
    {
        "Base": "Teste",
        **melhor_resultado["metricas_teste"]
    },
    {
        "Base": "Validação",
        **metricas_validacao
    }
])


# ------------------------------------------------------------
# 15. Mostrar informações do modelo
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("MELHOR MODELO APLICADO NA VALIDAÇÃO")
print("=" * 100)

print(
    "Modelo:",
    melhor_modelo_carregado
    .named_steps["clf"]
    .__class__.__name__
)

print(
    "Quantidade de features:",
    len(melhores_features_carregadas)
)

print(
    "Quantidade de registros na validação:",
    len(resultado_validacao)
)

print(
    "Quantidade selecionada no Top "
    f"{PROPORCAO_TOP:.0%}:",
    quantidade_top_validacao
)

print(
    "Threshold da validação:",
    round(threshold_validacao, 6)
)

print(
    "Taxa de fraude na validação:",
    f"{y_validacao.mean():.2%}"
)


# ------------------------------------------------------------
# 16. Mostrar métricas da validação
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("MÉTRICAS NA VALIDAÇÃO")
print("=" * 100)

for metrica, valor in metricas_validacao.items():

    if isinstance(valor, float):
        print(f"{metrica}: {valor:.4f}")
    else:
        print(f"{metrica}: {valor}")


# ------------------------------------------------------------
# 17. Mostrar comparativo
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("COMPARATIVO TESTE X VALIDAÇÃO")
print("=" * 100)

display(
    comparativo_teste_validacao[
        [
            "Base",
            "Total Base",
            "Total Fraudes",
            "Selecionados",
            "TP",
            "FP",
            "FN",
            "TN",
            "Precisão",
            "Cobertura",
            "F1",
            "AUC",
            "Average Precision",
            "KS",
            "Balanced Accuracy",
            "Erro_por_Acerto",
            "Lift",
            "Threshold"
        ]
    ]
)


# ------------------------------------------------------------
# 18. Mostrar Top 10% da validação
# ------------------------------------------------------------

top_validacao = (
    resultado_validacao[
        resultado_validacao[
            "selecionado_top"
        ]
    ]
    .copy()
)

print("\n" + "=" * 100)
print(f"REGISTROS SELECIONADOS NO TOP {PROPORCAO_TOP:.0%}")
print("=" * 100)

display(
    top_validacao
)


# ------------------------------------------------------------
# 19. Salvar resultados
# ------------------------------------------------------------

resultado_validacao.to_csv(
    "outputs/predictions/resultado_validacao_melhor_modelo.csv",
    index=False,
    sep=";"
)

comparativo_teste_validacao.to_csv(
    "outputs/modeling/comparativo_teste_validacao.csv",
    index=False,
    sep=";"
)

print("\nArquivos salvos:")
print("resultado_validacao_melhor_modelo.csv")
print("comparativo_teste_validacao.csv")


MELHOR MODELO APLICADO NA VALIDAÇÃO
Modelo: LogisticRegression
Quantidade de features: 38
Quantidade de registros na validação: 22500
Quantidade selecionada no Top 10%: 2250
Threshold da validação: 0.12407
Taxa de fraude na validação: 5.00%

MÉTRICAS NA VALIDAÇÃO
Total Base: 22500
Total Fraudes: 1125
Selecionados: 2250
TP: 536
FP: 1714
FN: 589
TN: 19661
Precisão: 0.2382
Cobertura: 0.4764
F1: 0.3176
AUC: 0.8248
Average Precision: 0.2749
KS: 0.5163
Balanced Accuracy: 0.6981
Erro_por_Acerto: 3.1978
Lift: 4.7644
Threshold: 0.1241

COMPARATIVO TESTE X VALIDAÇÃO


,Base,Total Base,Total Fraudes,Selecionados,TP,FP,FN,TN,Precisão,Cobertura,F1,AUC,Average Precision,KS,Balanced Accuracy,Erro_por_Acerto,Lift,Threshold
0,Teste,22500,1125,2250,565,1685,560,19690,0.251111,0.502222,0.334815,0.829582,0.288097,0.509942,0.711696,2.982301,5.022222,0.124034
1,Validação,22500,1125,2250,536,1714,589,19661,0.238222,0.476444,0.317630,0.824826,0.274856,0.516304,0.698129,3.197761,4.764444,0.124070



REGISTROS SELECIONADOS NO TOP 10%


,a,b,c,d,e,f,g,h,i,j,...,distancia_cluster_0,distancia_cluster_1,menor_distancia_cluster,distancia_cluster_atribuido,probabilidade_modelo,predicao_top_percentual,selecionado_top,acerto_fraude,erro_selecao,ranking_score
0,1,0.6465,4957870.01,1.0,0.012729,0.0,BR,0,Notebook Aspire 5 A515-52-57b7 I5 4 Gb 1tb Hdd...,cat_e5ca240,...,7.488155,8.602528,7.488155,7.488155,0.995191,1,True,1,0,1
1,2,0.7554,43819.26,1.0,0.000000,21.0,BR,0,Tapete Ilustrativo Pista Carrinho Hot Wheels 0...,cat_c84c9c3,...,11.782224,13.002570,11.782224,11.782224,0.994002,1,True,1,0,2
2,2,1.0000,31.01,1.0,0.000000,0.0,BR,0,Tapete Ilustrativo Pista Carrinho Hot Wheels 0...,cat_c84c9c3,...,11.984991,13.179327,11.984991,11.984991,0.990322,1,True,1,0,3
3,3,1.0000,128.50,1.0,0.202020,26.0,BR,0,Bateria Moura 70 Amperes - Passat - Santana - ...,cat_440aeb1,...,11.890788,12.878197,11.890788,11.890788,0.987559,1,True,1,0,4
4,2,1.0000,25.83,3.0,0.000000,8.0,BR,0,Tapete Ilustrativo Pista Carrinho Hot Wheels 0...,cat_c84c9c3,...,12.139030,13.180402,12.139030,12.139030,0.986857,1,True,1,0,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2245,4,0.7388,51044.58,2.0,0.128986,0.0,BR,24,50 Cotonete De Limpeza De Cabecas Mimaki Rolan...,cat_1948bad,...,5.759981,2.029637,2.029637,2.029637,0.124464,1,True,1,0,2246
2246,4,0.7671,290600.34,10.0,0.265487,0.0,BR,42,"Cabo De Chupeta Transferência De Carga 2,5 Met...",cat_0b6fc56,...,4.946254,2.228584,2.228584,2.228584,0.124370,1,True,0,1,2247
2247,4,0.4239,131475.20,50.0,0.000000,4.0,AR,0,Varios Aros Kia Preggio Motor J2 K2700 Aros ...,cat_3de024b,...,7.629345,4.900171,4.900171,4.900171,0.124361,1,True,0,1,2248
2248,4,0.7593,46030.73,3.0,0.373041,5.0,BR,0,Extensor Elástico Exercícios Ginástica Musculação,cat_297d68e,...,5.397174,2.735454,2.735454,2.735454,0.124290,1,True,0,1,2249



Arquivos salvos:
resultado_validacao_melhor_modelo.csv
comparativo_teste_validacao.csv


#### 7.5 Comparação do Score Atual vs Melhor Modelo (Top 10%)
###### Aqui é avaliado o desempenho do score atual na base de validação considerando o bloqueio dos 10% registros com maior risco. Para esse grupo, calculo métricas de classificação como precisão, cobertura, F1, AUC, Average Precision, Lift e erros por acerto, medindo a capacidade do score em concentrar fraudes no segmento de maior risco. Por fim, comparo diretamente os resultados obtidos pelo score atual com os resultados do melhor modelo desenvolvido, permitindo avaliar os ganhos de performance proporcionados pela nova abordagem.

In [35]:
# ============================================================
# DESEMPENHO DO SCORE ATUAL (TOP 10%)
# ============================================================

y_validacao = df_validacao_tratado["fraude"].astype(int)

metricas_score_atual = calcular_metricas(
    y_real=y_validacao,
    scores=df_validacao_tratado["score"],
    proporcao_top=0.10
)

print("=" * 80)
print("DESEMPENHO SCORE ATUAL")
print("=" * 80)

for k, v in metricas_score_atual.items():

    if isinstance(v, float):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")
        
comparativo = pd.DataFrame([
    {
        "Modelo": "Score Atual",
        **metricas_score_atual
    },
    {
        "Modelo": "Melhor Modelo",
        **metricas_validacao
    }
])

display(comparativo)

DESEMPENHO SCORE ATUAL
Total Base: 22500
Total Fraudes: 1125
Selecionados: 2250
TP: 495
FP: 1755
FN: 630
TN: 19620
Precisão: 0.2200
Cobertura: 0.4400
F1: 0.2933
AUC: 0.7289
Average Precision: 0.1845
KS: 0.4418
Balanced Accuracy: 0.6789
Erro_por_Acerto: 3.5455
Lift: 4.4000
Threshold: 88.0000


,Modelo,Total Base,Total Fraudes,Selecionados,TP,FP,FN,TN,Precisão,Cobertura,F1,AUC,Average Precision,KS,Balanced Accuracy,Erro_por_Acerto,Lift,Threshold
0,Score Atual,22500,1125,2250,495,1755,630,19620,0.220000,0.440000,0.293333,0.728921,0.184515,0.441778,0.678947,3.545455,4.400000,88.00000
1,Melhor Modelo,22500,1125,2250,536,1714,589,19661,0.238222,0.476444,0.317630,0.824826,0.274856,0.516304,0.698129,3.197761,4.764444,0.12407


#### 7.6 Avaliação do Modelo por Decis e Comparação com Top 20%
###### Avalio o desempenho do modelo utilizando as bases de treino, teste e validação. Os thresholds dos decis são definidos exclusivamente no treino e aplicados no teste para medir capacidade preditiva e impacto financeiro em diferentes níveis de risco. A partir desses resultados, seleciono a estratégia que maximiza o lucro líquido, preservando pelo menos 85% do lucro máximo e priorizando a maior cobertura de fraude. Em seguida, aplico esse mesmo corte na validação para obter uma avaliação final sem recalibração. Como análise complementar, também avalio um cenário alternativo que bloqueia exatamente os 20% registros com maior probabilidade de fraude na validação, comparando desempenho operacional, captura de fraudes, falsos positivos e resultado financeiro com a estratégia originalmente escolhida.

In [37]:



# ============================================================
# 1. CONFIGURAÇÕES
# ============================================================

TARGET = "fraude"
COLUNA_VALOR = "monto"

TAXA_COMISSAO = 0.10
PERDA_FRAUDE = 1.00

ARQUIVO_MODELO = "models/melhor_modelo_feature_sets.pkl"
ARQUIVO_FEATURES = "models/melhores_features_feature_sets.pkl"


# ============================================================
# 2. CARREGAR MODELO E FEATURES
# ============================================================

melhor_modelo = joblib.load(
    ARQUIVO_MODELO
)

melhores_features = joblib.load(
    ARQUIVO_FEATURES
)

melhores_features = list(
    melhores_features
)

print("=" * 100)
print("MODELO E FEATURES CARREGADOS")
print("=" * 100)

print(
    f"Quantidade de features: "
    f"{len(melhores_features)}"
)


# ============================================================
# 3. PREPARAR TREINO, TESTE E VALIDAÇÃO
# ============================================================

treino = preparar_base_modelagem(
    df=df_treino_tratado,
    nome_base="treino"
)

teste = preparar_base_modelagem(
    df=df_teste_tratado,
    nome_base="teste"
)

validacao = preparar_base_modelagem(
    df=df_validacao_tratado,
    nome_base="validação"
)


print("\n" + "=" * 100)
print("BASES PREPARADAS")
print("=" * 100)

print(f"Treino: {treino.shape}")
print(f"Teste: {teste.shape}")
print(f"Validação: {validacao.shape}")

print(
    f"\nTaxa de fraude no treino: "
    f"{treino[TARGET].mean():.2%}"
)

print(
    f"Taxa de fraude no teste: "
    f"{teste[TARGET].mean():.2%}"
)

print(
    f"Taxa de fraude na validação: "
    f"{validacao[TARGET].mean():.2%}"
)


# ============================================================
# 4. VALIDAR FEATURES NECESSÁRIAS
# ============================================================

features_ausentes_treino = [
    coluna
    for coluna in melhores_features
    if coluna not in treino.columns
]

features_ausentes_teste = [
    coluna
    for coluna in melhores_features
    if coluna not in teste.columns
]

features_ausentes_validacao = [
    coluna
    for coluna in melhores_features
    if coluna not in validacao.columns
]

if features_ausentes_treino:

    raise ValueError(
        "Features ausentes no treino: "
        f"{features_ausentes_treino}"
    )

if features_ausentes_teste:

    raise ValueError(
        "Features ausentes no teste: "
        f"{features_ausentes_teste}"
    )

if features_ausentes_validacao:

    raise ValueError(
        "Features ausentes na validação: "
        f"{features_ausentes_validacao}"
    )



# ============================================================
# 8. APLICAR O MODELO AO TREINO
# ============================================================

X_treino_modelo = treino.reindex(
    columns=melhores_features
)

treino["score_modelo"] = (
    melhor_modelo
    .predict_proba(
        X_treino_modelo
    )[:, 1]
)


# ============================================================
# 9. DEFINIR OS THRESHOLDS DOS DECIS NO TREINO
# ============================================================
#
# P90 = entrada no Top 10%
# P80 = entrada no Top 20%
# P70 = entrada no Top 30%
# ...
# P00 = entrada no Top 100%
# ============================================================

percentuais_acumulados = np.arange(
    0.10,
    1.01,
    0.10
)

limites_decis = pd.DataFrame({
    "Decil Acumulado": np.arange(1, 11),
    "Percentual Acumulado": percentuais_acumulados,
    "Top Acumulado": [
        f"Top {percentual:.0%}"
        for percentual in percentuais_acumulados
    ],
    "Threshold Treino": [
        float(
            treino["score_modelo"].quantile(
                1 - percentual
            )
        )
        for percentual in percentuais_acumulados
    ]
})

# Garantir que o Top 100% inclua todos
limites_decis.loc[
    limites_decis["Decil Acumulado"] == 10,
    "Threshold Treino"
] = -np.inf


print("\n" + "=" * 100)
print("LIMITES DOS DECIS DEFINIDOS NO TREINO")
print("=" * 100)

display(limites_decis)


# ============================================================
# 10. APLICAR O MODELO AO TESTE
# ============================================================

X_teste_modelo = teste.reindex(
    columns=melhores_features
)

teste["score_modelo"] = (
    melhor_modelo
    .predict_proba(
        X_teste_modelo
    )[:, 1]
)


# ============================================================
# 11. MÉTRICAS GLOBAIS DO TESTE
# ============================================================

(
    auc_global_teste,
    average_precision_global_teste
) = calcular_metricas_globais(
    base=teste
)


# ============================================================
# 12. CENÁRIO BASE DO TESTE: APROVAR TUDO
# ============================================================

cenario_base_teste = calcular_cenario_aprovar_tudo(
    base=teste
)

lucro_base_teste = cenario_base_teste[
    "Lucro Aprovar Tudo"
]


print("\n" + "=" * 100)
print("CENÁRIO BASE NO TESTE")
print("=" * 100)

print(
    f"Valor legítimo total: "
    f"R$ {cenario_base_teste['Valor Legítimo Total']:,.2f}"
)

print(
    f"Valor de fraude total: "
    f"R$ {cenario_base_teste['Valor Fraude Total']:,.2f}"
)

print(
    f"Receita aprovando tudo: "
    f"R$ {cenario_base_teste['Receita Aprovar Tudo']:,.2f}"
)

print(
    f"Perda aprovando tudo: "
    f"R$ {cenario_base_teste['Perda Aprovar Tudo']:,.2f}"
)

print(
    f"Lucro aprovando tudo: "
    f"R$ {lucro_base_teste:,.2f}"
)


# ============================================================
# 13. APLICAR TODOS OS THRESHOLDS NO TESTE
# ============================================================

resultados_acumulados_teste = []

for _, linha in limites_decis.iterrows():

    resultado = calcular_resultado_corte_model(
        base=teste,
        threshold=float(
            linha["Threshold Treino"]
        ),
        nome_base="Teste",
        auc_global=auc_global_teste,
        average_precision_global=(
            average_precision_global_teste
        ),
        lucro_base_aprovar_tudo=(
            lucro_base_teste
        ),
        decil_acumulado=int(
            linha["Decil Acumulado"]
        ),
        percentual_teorico=float(
            linha["Percentual Acumulado"]
        ),
        top_acumulado=linha[
            "Top Acumulado"
        ]
    )

    resultados_acumulados_teste.append(
        resultado
    )


comparativo_decis_teste = pd.DataFrame(
    resultados_acumulados_teste
)


# ============================================================
# 14. ESCOLHER O PONTO DE MAIOR LUCRO NO TESTE
# ============================================================


lucro_maximo = (
    comparativo_decis_teste[
        "Lucro Líquido"
    ].max()
)

candidatos = comparativo_decis_teste[
    comparativo_decis_teste[
        "Lucro Líquido"
    ] >= lucro_maximo * 0.85
]

indice_melhor = candidatos[
    "Cobertura"
].idxmax()

melhor_corte_teste = (
    candidatos.loc[indice_melhor]
)


# Comparar o melhor corte com aprovar tudo
if (
    lucro_base_teste
    >= melhor_corte_teste["Lucro Líquido"]
):

    recomendacao = (
        "APROVAR TODAS AS TRANSAÇÕES"
    )

    lucro_maximo_teste = (
        lucro_base_teste
    )

    threshold_recomendado = np.nan

else:

    recomendacao = (
        melhor_corte_teste["Top Acumulado"]
    )

    lucro_maximo_teste = float(
        melhor_corte_teste["Lucro Líquido"]
    )

    threshold_recomendado = float(
        melhor_corte_teste["Threshold Treino"]
    )


# ============================================================
# 15. EXIBIR VISÃO ACUMULADA DO TESTE
# ============================================================

colunas_exibicao = [
    "Base Avaliada",
    "Decil Acumulado",
    "Top Acumulado",
    "Threshold Treino",
    "Selecionados para Bloqueio",
    "% Real Selecionado",
    "TP - Fraudes Bloqueadas",
    "FP - Legítimas Bloqueadas",
    "FN - Fraudes Aprovadas",
    "TN - Legítimas Aprovadas",
    "Precisão",
    "Cobertura",
    "F1",
    "AUC Global",
    "Average Precision Global",
    "Balanced Accuracy",
    "Erro_por_Acerto",
    "Lift",
    "Receita Comissão",
    "Perda com Fraude",
    "Comissão Perdida por Bloqueio",
    "Lucro Líquido",
    "Lucro Aprovar Tudo",
    "Ganho vs Aprovar Tudo",
    "Ganho Percentual vs Aprovar Tudo"
]


print("\n" + "=" * 120)
print("DESEMPENHO ACUMULADO POR DECIL NO TESTE")
print("=" * 120)

display(
    comparativo_decis_teste[
        colunas_exibicao
    ]
)


# ============================================================
# 16. EXIBIR O CORTE ESCOLHIDO NO TESTE
# ============================================================

print("\n" + "=" * 120)
print("PONTO DE CORTE ESCOLHIDO NO TESTE")
print("=" * 120)

print(
    f"AUC global no teste: "
    f"{auc_global_teste:.4f}"
)

print(
    f"Average Precision no teste: "
    f"{average_precision_global_teste:.4f}"
)

print(
    f"Lucro aprovando tudo: "
    f"R$ {lucro_base_teste:,.2f}"
)

print(
    f"Estratégia recomendada: "
    f"{recomendacao}"
)

print(
    f"Lucro máximo no teste: "
    f"R$ {lucro_maximo_teste:,.2f}"
)

print(
    f"Ganho sobre aprovar tudo: "
    f"R$ {lucro_maximo_teste - lucro_base_teste:,.2f}"
)

if recomendacao != "APROVAR TODAS AS TRANSAÇÕES":

    print(
        f"Threshold recomendado: "
        f"{threshold_recomendado:.8f}"
    )

    print(
        f"Decil escolhido: "
        f"{int(melhor_corte_teste['Decil Acumulado'])}"
    )

    print(
        f"Top escolhido: "
        f"{melhor_corte_teste['Top Acumulado']}"
    )

    print(
        f"Percentual real bloqueado no teste: "
        f"{melhor_corte_teste['% Real Selecionado']:.2%}"
    )

    print(
        f"Fraudes bloqueadas no teste: "
        f"{int(melhor_corte_teste['TP - Fraudes Bloqueadas'])}"
    )

    print(
        f"Fraudes aprovadas no teste: "
        f"{int(melhor_corte_teste['FN - Fraudes Aprovadas'])}"
    )

    print(
        f"Legítimas bloqueadas no teste: "
        f"{int(melhor_corte_teste['FP - Legítimas Bloqueadas'])}"
    )

    print(
        f"Precisão no teste: "
        f"{melhor_corte_teste['Precisão']:.2%}"
    )

    print(
        f"Cobertura no teste: "
        f"{melhor_corte_teste['Cobertura']:.2%}"
    )

    print(
        f"Erros por acerto no teste: "
        f"{melhor_corte_teste['Erro_por_Acerto']:.2f}"
    )


# ============================================================
# 17. ATRIBUIR DECIL AOS REGISTROS DO TESTE
# ============================================================

thresholds_decis = (
    limites_decis
    .sort_values("Decil Acumulado")
    ["Threshold Treino"]
    .tolist()
)



teste["decil_modelo"] = (
    teste["score_modelo"]
    .apply(atribuir_decil_por_threshold)
)


if recomendacao == "APROVAR TODAS AS TRANSAÇÕES":

    teste["decisao_ponto_otimo"] = (
        "APROVAR"
    )

else:

    teste["decisao_ponto_otimo"] = np.where(
        teste["score_modelo"]
        >= threshold_recomendado,
        "BLOQUEAR",
        "APROVAR"
    )


# ============================================================
# 18. APLICAR O MODELO À VALIDAÇÃO
# ============================================================

X_validacao_modelo = validacao.reindex(
    columns=melhores_features
)

validacao["score_modelo"] = (
    melhor_modelo
    .predict_proba(
        X_validacao_modelo
    )[:, 1]
)


# ============================================================
# 19. MÉTRICAS GLOBAIS DA VALIDAÇÃO
# ============================================================

(
    auc_global_validacao,
    average_precision_global_validacao
) = calcular_metricas_globais(
    base=validacao
)


# ============================================================
# 20. CENÁRIO BASE DA VALIDAÇÃO
# ============================================================

cenario_base_validacao = calcular_cenario_aprovar_tudo(
    base=validacao
)

lucro_base_validacao = cenario_base_validacao[
    "Lucro Aprovar Tudo"
]


# ============================================================
# 21. APLICAR SOMENTE A ESTRATÉGIA ESCOLHIDA NO TESTE
# ============================================================
#
# Nesta etapa não procuramos outro ponto de corte.
# A validação recebe somente a decisão tomada no teste.
# ============================================================

if recomendacao == "APROVAR TODAS AS TRANSAÇÕES":

    total_validacao = len(validacao)

    total_fraudes_validacao = int(
        validacao[TARGET].sum()
    )

    total_legitimas_validacao = int(
        total_validacao
        - total_fraudes_validacao
    )

    resultado_final_validacao = {
        "Base Avaliada": "Validação",
        "Decil Acumulado": np.nan,
        "Top Acumulado": (
            "Aprovar todas as transações"
        ),
        "Percentual Teórico": 0.0,
        "Threshold Treino": np.nan,
        "Total Base": total_validacao,
        "Selecionados para Bloqueio": 0,
        "% Real Selecionado": 0.0,
        "TP - Fraudes Bloqueadas": 0,
        "FP - Legítimas Bloqueadas": 0,
        "FN - Fraudes Aprovadas": (
            total_fraudes_validacao
        ),
        "TN - Legítimas Aprovadas": (
            total_legitimas_validacao
        ),
        "Precisão": 0.0,
        "Cobertura": 0.0,
        "F1": 0.0,
        "AUC Global": auc_global_validacao,
        "Average Precision Global": (
            average_precision_global_validacao
        ),
        "Balanced Accuracy": 0.5,
        "Erro_por_Acerto": np.nan,
        "Lift": 0.0,
        "Receita Comissão": (
            cenario_base_validacao[
                "Receita Aprovar Tudo"
            ]
        ),
        "Perda com Fraude": (
            cenario_base_validacao[
                "Perda Aprovar Tudo"
            ]
        ),
        "Comissão Perdida por Bloqueio": 0.0,
        "Lucro Líquido": lucro_base_validacao,
        "Lucro Aprovar Tudo": (
            lucro_base_validacao
        ),
        "Ganho vs Aprovar Tudo": 0.0,
        "Ganho Percentual vs Aprovar Tudo": 0.0
    }

    validacao["decisao_ponto_otimo"] = (
        "APROVAR"
    )

else:

    resultado_final_validacao = (
        calcular_resultado_corte_model(
            base=validacao,
            threshold=threshold_recomendado,
            nome_base="Validação",
            auc_global=auc_global_validacao,
            average_precision_global=(
                average_precision_global_validacao
            ),
            lucro_base_aprovar_tudo=(
                lucro_base_validacao
            ),
            decil_acumulado=int(
                melhor_corte_teste[
                    "Decil Acumulado"
                ]
            ),
            percentual_teorico=float(
                melhor_corte_teste[
                    "Percentual Teórico"
                ]
            ),
            top_acumulado=(
                melhor_corte_teste[
                    "Top Acumulado"
                ]
            )
        )
    )

    validacao["decisao_ponto_otimo"] = np.where(
        validacao["score_modelo"]
        >= threshold_recomendado,
        "BLOQUEAR",
        "APROVAR"
    )


validacao["decil_modelo"] = (
    validacao["score_modelo"]
    .apply(atribuir_decil_por_threshold)
)

resultado_final_validacao = pd.DataFrame([
    resultado_final_validacao
])


# ============================================================
# 22. EXIBIR RESULTADO FINAL NA VALIDAÇÃO
# ============================================================

print("\n" + "=" * 120)
print("RESULTADO FINAL NA VALIDAÇÃO")
print("=" * 120)

print(
    f"Estratégia definida no teste: "
    f"{recomendacao}"
)

if recomendacao != "APROVAR TODAS AS TRANSAÇÕES":

    print(
        f"Threshold definido no teste e aplicado "
        f"na validação: {threshold_recomendado:.8f}"
    )

print(
    f"AUC global na validação: "
    f"{auc_global_validacao:.4f}"
)

print(
    f"Average Precision na validação: "
    f"{average_precision_global_validacao:.4f}"
)

print(
    f"Lucro aprovando tudo na validação: "
    f"R$ {lucro_base_validacao:,.2f}"
)

print(
    f"Lucro da estratégia na validação: "
    f"R$ "
    f"{resultado_final_validacao.loc[0, 'Lucro Líquido']:,.2f}"
)

print(
    f"Ganho vs aprovar tudo na validação: "
    f"R$ "
    f"{resultado_final_validacao.loc[0, 'Ganho vs Aprovar Tudo']:,.2f}"
)

display(
    resultado_final_validacao[
        colunas_exibicao
    ]
)


# ============================================================
# 23. RESUMO FINAL: TESTE X VALIDAÇÃO
# ============================================================

if recomendacao == "APROVAR TODAS AS TRANSAÇÕES":

    precisao_teste = 0.0
    cobertura_teste = 0.0
    f1_teste = 0.0
    erro_por_acerto_teste = np.nan
    percentual_bloqueado_teste = 0.0

else:

    precisao_teste = melhor_corte_teste[
        "Precisão"
    ]

    cobertura_teste = melhor_corte_teste[
        "Cobertura"
    ]

    f1_teste = melhor_corte_teste[
        "F1"
    ]

    erro_por_acerto_teste = melhor_corte_teste[
        "Erro_por_Acerto"
    ]

    percentual_bloqueado_teste = melhor_corte_teste[
        "% Real Selecionado"
    ]


resumo_final = pd.DataFrame([
    {
        "Base": "Teste",
        "Uso da Base": "Escolha do corte",
        "Estratégia": recomendacao,
        "Threshold": threshold_recomendado,
        "% Real Bloqueado": (
            percentual_bloqueado_teste
        ),
        "Precisão": precisao_teste,
        "Cobertura": cobertura_teste,
        "F1": f1_teste,
        "AUC Global": auc_global_teste,
        "Average Precision Global": (
            average_precision_global_teste
        ),
        "Erro_por_Acerto": (
            erro_por_acerto_teste
        ),
        "Lucro Aprovar Tudo": (
            lucro_base_teste
        ),
        "Lucro da Estratégia": (
            lucro_maximo_teste
        ),
        "Ganho vs Aprovar Tudo": (
            lucro_maximo_teste
            - lucro_base_teste
        )
    },
    {
        "Base": "Validação",
        "Uso da Base": "Avaliação final",
        "Estratégia": recomendacao,
        "Threshold": threshold_recomendado,
        "% Real Bloqueado": (
            resultado_final_validacao.loc[
                0,
                "% Real Selecionado"
            ]
        ),
        "Precisão": (
            resultado_final_validacao.loc[
                0,
                "Precisão"
            ]
        ),
        "Cobertura": (
            resultado_final_validacao.loc[
                0,
                "Cobertura"
            ]
        ),
        "F1": (
            resultado_final_validacao.loc[
                0,
                "F1"
            ]
        ),
        "AUC Global": auc_global_validacao,
        "Average Precision Global": (
            average_precision_global_validacao
        ),
        "Erro_por_Acerto": (
            resultado_final_validacao.loc[
                0,
                "Erro_por_Acerto"
            ]
        ),
        "Lucro Aprovar Tudo": (
            lucro_base_validacao
        ),
        "Lucro da Estratégia": (
            resultado_final_validacao.loc[
                0,
                "Lucro Líquido"
            ]
        ),
        "Ganho vs Aprovar Tudo": (
            resultado_final_validacao.loc[
                0,
                "Ganho vs Aprovar Tudo"
            ]
        )
    }
])


print("\n" + "=" * 120)
print("RESUMO FINAL: TESTE X VALIDAÇÃO")
print("=" * 120)

display(resumo_final)


# ============================================================
# 24. SALVAR RESULTADOS
# ============================================================

limites_decis.to_csv(
    "outputs/modeling/limites_decis_treino.csv",
    index=False,
    sep=";"
)

comparativo_decis_teste.to_csv(
    "outputs/modeling/desempenho_acumulado_decis_teste.csv",
    index=False,
    sep=";"
)

teste.to_csv(
    "outputs/predictions/teste_com_decil_e_decisao.csv",
    index=False,
    sep=";"
)

resultado_final_validacao.to_csv(
    "outputs/modeling/resultado_final_validacao.csv",
    index=False,
    sep=";"
)

validacao.to_csv(
    "outputs/predictions/validacao_com_decil_e_decisao.csv",
    index=False,
    sep=";"
)

resumo_final.to_csv(
    "outputs/modeling/resumo_final_teste_validacao.csv",
    index=False,
    sep=";"
)


print("\nArquivos salvos:")

print(
    "limites_decis_treino.csv"
)

print(
    "desempenho_acumulado_decis_teste.csv"
)

print(
    "teste_com_decil_e_decisao.csv"
)

print(
    "resultado_final_validacao.csv"
)

print(
    "validacao_com_decil_e_decisao.csv"
)

print(
    "resumo_final_teste_validacao.csv"
)

# ============================================================
# 21.1 CENÁRIO COMPARATIVO:
# TOP 20% EXATO DO MODELO NA VALIDAÇÃO
#
# IMPORTANTE:
# - Seleciona exatamente 20% dos registros da validação.
# - Usa score_modelo em ordem decrescente.
# - Serve como análise comparativa.
# - Não altera o corte escolhido anteriormente no teste.
# ============================================================

PROPORCAO_TOP20_VALIDACAO = 0.20

validacao_top20_modelo = validacao.copy()

total_validacao_top20 = len(
    validacao_top20_modelo
)

if total_validacao_top20 == 0:
    raise ValueError(
        "A base de validação está vazia."
    )

quantidade_top20_validacao = int(
    np.ceil(
        total_validacao_top20
        * PROPORCAO_TOP20_VALIDACAO
    )
)


# ------------------------------------------------------------
# Selecionar exatamente os 20% de maior probabilidade
#
# O índice original é usado como desempate para garantir
# exatamente a quantidade desejada.
# ------------------------------------------------------------

validacao_top20_modelo[
    "_indice_desempate"
] = np.arange(
    len(validacao_top20_modelo)
)

indices_top20_modelo = (
    validacao_top20_modelo
    .sort_values(
        by=[
            "score_modelo",
            "_indice_desempate"
        ],
        ascending=[
            False,
            True
        ],
        kind="mergesort"
    )
    .head(
        quantidade_top20_validacao
    )
    .index
)


# ------------------------------------------------------------
# Marcar decisão Top 20%
# ------------------------------------------------------------

validacao_top20_modelo[
    "bloqueio_top20_modelo"
] = 0

validacao_top20_modelo.loc[
    indices_top20_modelo,
    "bloqueio_top20_modelo"
] = 1

validacao_top20_modelo[
    "decisao_top20_modelo"
] = np.where(
    validacao_top20_modelo[
        "bloqueio_top20_modelo"
    ] == 1,
    "BLOQUEAR",
    "APROVAR"
)


# ------------------------------------------------------------
# Threshold observado do Top 20%
#
# Atenção:
# devido a empates, o threshold sozinho pode selecionar mais
# registros. A coluna bloqueio_top20_modelo é a referência
# para a seleção exata.
# ------------------------------------------------------------

threshold_top20_modelo = float(
    validacao_top20_modelo.loc[
        indices_top20_modelo,
        "score_modelo"
    ].min()
)


# ------------------------------------------------------------
# Matriz de confusão
# ------------------------------------------------------------

y_real_top20 = (
    validacao_top20_modelo[
        TARGET
    ]
    .astype(int)
)

y_pred_top20 = (
    validacao_top20_modelo[
        "bloqueio_top20_modelo"
    ]
    .astype(int)
)

tn_top20, fp_top20, fn_top20, tp_top20 = (
    confusion_matrix(
        y_real_top20,
        y_pred_top20,
        labels=[0, 1]
    )
    .ravel()
)


# ------------------------------------------------------------
# Métricas do Top 20%
# ------------------------------------------------------------

precisao_top20 = precision_score(
    y_real_top20,
    y_pred_top20,
    zero_division=0
)

cobertura_top20 = recall_score(
    y_real_top20,
    y_pred_top20,
    zero_division=0
)

f1_top20 = f1_score(
    y_real_top20,
    y_pred_top20,
    zero_division=0
)

balanced_accuracy_top20 = (
    balanced_accuracy_score(
        y_real_top20,
        y_pred_top20
    )
)

erro_por_acerto_top20 = (
    fp_top20 / tp_top20
    if tp_top20 > 0
    else np.nan
)

taxa_fraude_validacao = float(
    y_real_top20.mean()
)

lift_top20 = (
    precisao_top20
    / taxa_fraude_validacao
    if taxa_fraude_validacao > 0
    else np.nan
)

quantidade_real_top20 = int(
    y_pred_top20.sum()
)

percentual_real_top20 = (
    quantidade_real_top20
    / total_validacao_top20
)


# ------------------------------------------------------------
# Valores financeiros
# ------------------------------------------------------------

mascara_bloqueada_top20 = (
    y_pred_top20 == 1
)

mascara_aprovada_top20 = (
    y_pred_top20 == 0
)

valor_legitimo_aprovado_top20 = float(
    validacao_top20_modelo.loc[
        mascara_aprovada_top20
        & (
            validacao_top20_modelo[TARGET]
            == 0
        ),
        COLUNA_VALOR
    ].sum()
)

valor_fraude_aprovada_top20 = float(
    validacao_top20_modelo.loc[
        mascara_aprovada_top20
        & (
            validacao_top20_modelo[TARGET]
            == 1
        ),
        COLUNA_VALOR
    ].sum()
)

valor_legitimo_bloqueado_top20 = float(
    validacao_top20_modelo.loc[
        mascara_bloqueada_top20
        & (
            validacao_top20_modelo[TARGET]
            == 0
        ),
        COLUNA_VALOR
    ].sum()
)

valor_fraude_bloqueada_top20 = float(
    validacao_top20_modelo.loc[
        mascara_bloqueada_top20
        & (
            validacao_top20_modelo[TARGET]
            == 1
        ),
        COLUNA_VALOR
    ].sum()
)


# ------------------------------------------------------------
# Resultado financeiro
# ------------------------------------------------------------

receita_comissao_top20 = (
    TAXA_COMISSAO
    * valor_legitimo_aprovado_top20
)

perda_com_fraude_top20 = (
    PERDA_FRAUDE
    * valor_fraude_aprovada_top20
)

comissao_perdida_top20 = (
    TAXA_COMISSAO
    * valor_legitimo_bloqueado_top20
)

perda_fraude_evitada_top20 = (
    PERDA_FRAUDE
    * valor_fraude_bloqueada_top20
)

lucro_liquido_top20 = (
    receita_comissao_top20
    - perda_com_fraude_top20
)

ganho_top20_vs_aprovar_tudo = (
    lucro_liquido_top20
    - lucro_base_validacao
)

ganho_percentual_top20 = (
    ganho_top20_vs_aprovar_tudo
    / abs(lucro_base_validacao)
    if lucro_base_validacao != 0
    else np.nan
)


# ------------------------------------------------------------
# Consolidar resultado do Top 20%
# ------------------------------------------------------------

resultado_top20_modelo_validacao = pd.DataFrame([
    {
        "Base Avaliada": "Validação",
        "Tipo de Avaliação": (
            "Top 20% exato pela probabilidade do modelo"
        ),
        "Decil Acumulado": 2,
        "Top Acumulado": "Top 20%",
        "Percentual Teórico": (
            PROPORCAO_TOP20_VALIDACAO
        ),
        "Threshold Treino": (
            threshold_top20_modelo
        ),
        "Total Base": (
            total_validacao_top20
        ),
        "Selecionados para Bloqueio": (
            quantidade_real_top20
        ),
        "% Real Selecionado": (
            percentual_real_top20
        ),
        "TP - Fraudes Bloqueadas": int(
            tp_top20
        ),
        "FP - Legítimas Bloqueadas": int(
            fp_top20
        ),
        "FN - Fraudes Aprovadas": int(
            fn_top20
        ),
        "TN - Legítimas Aprovadas": int(
            tn_top20
        ),
        "Precisão": precisao_top20,
        "Cobertura": cobertura_top20,
        "F1": f1_top20,
        "AUC Global": (
            auc_global_validacao
        ),
        "Average Precision Global": (
            average_precision_global_validacao
        ),
        "Balanced Accuracy": (
            balanced_accuracy_top20
        ),
        "Erro_por_Acerto": (
            erro_por_acerto_top20
        ),
        "Lift": lift_top20,
        "Valor Fraude Bloqueada": (
            valor_fraude_bloqueada_top20
        ),
        "Perda de Fraude Evitada": (
            perda_fraude_evitada_top20
        ),
        "Receita Comissão": (
            receita_comissao_top20
        ),
        "Perda com Fraude": (
            perda_com_fraude_top20
        ),
        "Comissão Perdida por Bloqueio": (
            comissao_perdida_top20
        ),
        "Lucro Líquido": (
            lucro_liquido_top20
        ),
        "Lucro Aprovar Tudo": (
            lucro_base_validacao
        ),
        "Ganho vs Aprovar Tudo": (
            ganho_top20_vs_aprovar_tudo
        ),
        "Ganho Percentual vs Aprovar Tudo": (
            ganho_percentual_top20
        )
    }
])


# ============================================================
# 21.2 COMPARAR CORTE ESCOLHIDO X TOP 20% EXATO
# ============================================================

comparacao_corte_vs_top20_modelo = pd.DataFrame([
    {
        "Cenário": (
            f"Corte escolhido no teste: "
            f"{recomendacao}"
        ),
        "Threshold": (
            threshold_recomendado
        ),
        "% Real Bloqueado": (
            resultado_final_validacao.loc[
                0,
                "% Real Selecionado"
            ]
        ),
        "TP - Fraudes Bloqueadas": (
            resultado_final_validacao.loc[
                0,
                "TP - Fraudes Bloqueadas"
            ]
        ),
        "FP - Legítimas Bloqueadas": (
            resultado_final_validacao.loc[
                0,
                "FP - Legítimas Bloqueadas"
            ]
        ),
        "FN - Fraudes Aprovadas": (
            resultado_final_validacao.loc[
                0,
                "FN - Fraudes Aprovadas"
            ]
        ),
        "TN - Legítimas Aprovadas": (
            resultado_final_validacao.loc[
                0,
                "TN - Legítimas Aprovadas"
            ]
        ),
        "Precisão": (
            resultado_final_validacao.loc[
                0,
                "Precisão"
            ]
        ),
        "Cobertura": (
            resultado_final_validacao.loc[
                0,
                "Cobertura"
            ]
        ),
        "F1": (
            resultado_final_validacao.loc[
                0,
                "F1"
            ]
        ),
        "AUC Global": (
            auc_global_validacao
        ),
        "Average Precision Global": (
            average_precision_global_validacao
        ),
        "Balanced Accuracy": (
            resultado_final_validacao.loc[
                0,
                "Balanced Accuracy"
            ]
        ),
        "Erro_por_Acerto": (
            resultado_final_validacao.loc[
                0,
                "Erro_por_Acerto"
            ]
        ),
        "Lift": (
            resultado_final_validacao.loc[
                0,
                "Lift"
            ]
        ),
        "Receita Comissão": (
            resultado_final_validacao.loc[
                0,
                "Receita Comissão"
            ]
        ),
        "Perda com Fraude": (
            resultado_final_validacao.loc[
                0,
                "Perda com Fraude"
            ]
        ),
        "Comissão Perdida por Bloqueio": (
            resultado_final_validacao.loc[
                0,
                "Comissão Perdida por Bloqueio"
            ]
        ),
        "Lucro Líquido": (
            resultado_final_validacao.loc[
                0,
                "Lucro Líquido"
            ]
        ),
        "Lucro Aprovar Tudo": (
            lucro_base_validacao
        ),
        "Ganho vs Aprovar Tudo": (
            resultado_final_validacao.loc[
                0,
                "Ganho vs Aprovar Tudo"
            ]
        )
    },
    {
        "Cenário": (
            "Top 20% exato na validação"
        ),
        "Threshold": (
            threshold_top20_modelo
        ),
        "% Real Bloqueado": (
            percentual_real_top20
        ),
        "TP - Fraudes Bloqueadas": int(
            tp_top20
        ),
        "FP - Legítimas Bloqueadas": int(
            fp_top20
        ),
        "FN - Fraudes Aprovadas": int(
            fn_top20
        ),
        "TN - Legítimas Aprovadas": int(
            tn_top20
        ),
        "Precisão": precisao_top20,
        "Cobertura": cobertura_top20,
        "F1": f1_top20,
        "AUC Global": (
            auc_global_validacao
        ),
        "Average Precision Global": (
            average_precision_global_validacao
        ),
        "Balanced Accuracy": (
            balanced_accuracy_top20
        ),
        "Erro_por_Acerto": (
            erro_por_acerto_top20
        ),
        "Lift": lift_top20,
        "Receita Comissão": (
            receita_comissao_top20
        ),
        "Perda com Fraude": (
            perda_com_fraude_top20
        ),
        "Comissão Perdida por Bloqueio": (
            comissao_perdida_top20
        ),
        "Lucro Líquido": (
            lucro_liquido_top20
        ),
        "Lucro Aprovar Tudo": (
            lucro_base_validacao
        ),
        "Ganho vs Aprovar Tudo": (
            ganho_top20_vs_aprovar_tudo
        )
    }
])


# ============================================================
# 21.3 EXIBIR RESULTADOS
# ============================================================

print("\n" + "=" * 120)
print("RESULTADO DO MODELO NO TOP 20% EXATO DA VALIDAÇÃO")
print("=" * 120)

display(
    resultado_top20_modelo_validacao
)


print("\n" + "=" * 120)
print("CORTE ESCOLHIDO X TOP 20% EXATO DO MODELO")
print("=" * 120)

display(
    comparacao_corte_vs_top20_modelo
)


print("\nResumo do Top 20% exato:")

print(
    f"Quantidade bloqueada: "
    f"{quantidade_real_top20:,}"
)

print(
    f"Percentual bloqueado: "
    f"{percentual_real_top20:.2%}"
)

print(
    f"Threshold observado: "
    f"{threshold_top20_modelo:.8f}"
)

print(
    f"Precisão: "
    f"{precisao_top20:.2%}"
)

print(
    f"Cobertura: "
    f"{cobertura_top20:.2%}"
)

print(
    f"F1: "
    f"{f1_top20:.4f}"
)

print(
    f"Erros por acerto: "
    f"{erro_por_acerto_top20:.2f}"
)

print(
    f"Lucro líquido: "
    f"R$ {lucro_liquido_top20:,.2f}"
)

print(
    f"Ganho vs aprovar tudo: "
    f"R$ {ganho_top20_vs_aprovar_tudo:,.2f}"
)


# ============================================================
# 21.4 SALVAR RESULTADOS ADICIONAIS
# ============================================================

resultado_top20_modelo_validacao.to_csv(
    "outputs/modeling/resultado_top20_modelo_validacao.csv",
    index=False,
    sep=";"
)

comparacao_corte_vs_top20_modelo.to_csv(
    "outputs/modeling/comparacao_corte_escolhido_vs_top20_modelo_validacao.csv",
    index=False,
    sep=";"
)

validacao_top20_modelo.to_csv(
    "outputs/predictions/validacao_top20_modelo_detalhada.csv",
    index=False,
    sep=";"
)

print("\nArquivos adicionais salvos:")

print(
    "resultado_top20_modelo_validacao.csv"
)

print(
    "comparacao_corte_escolhido_vs_top20_modelo_validacao.csv"
)

print(
    "validacao_top20_modelo_detalhada.csv"
)

MODELO E FEATURES CARREGADOS
Quantidade de features: 38

BASES PREPARADAS
Treino: (105000, 49)
Teste: (22500, 49)
Validação: (22500, 49)

Taxa de fraude no treino: 5.00%
Taxa de fraude no teste: 5.00%
Taxa de fraude na validação: 5.00%

LIMITES DOS DECIS DEFINIDOS NO TREINO


,Decil Acumulado,Percentual Acumulado,Top Acumulado,Threshold Treino
0,1,0.1,Top 10%,0.115833
1,2,0.2,Top 20%,0.057323
2,3,0.3,Top 30%,0.033999
3,4,0.4,Top 40%,0.022588
4,5,0.5,Top 50%,0.016084
5,6,0.6,Top 60%,0.011960
6,7,0.7,Top 70%,0.009171
7,8,0.8,Top 80%,0.007040
8,9,0.9,Top 90%,0.005031
9,10,1.0,Top 100%,-inf



CENÁRIO BASE NO TESTE
Valor legítimo total: R$ 895,824.57
Valor de fraude total: R$ 94,386.58
Receita aprovando tudo: R$ 89,582.46
Perda aprovando tudo: R$ 94,386.58
Lucro aprovando tudo: R$ -4,804.12

DESEMPENHO ACUMULADO POR DECIL NO TESTE


,Base Avaliada,Decil Acumulado,Top Acumulado,Threshold Treino,Selecionados para Bloqueio,% Real Selecionado,TP - Fraudes Bloqueadas,FP - Legítimas Bloqueadas,FN - Fraudes Aprovadas,TN - Legítimas Aprovadas,...,Balanced Accuracy,Erro_por_Acerto,Lift,Receita Comissão,Perda com Fraude,Comissão Perdida por Bloqueio,Lucro Líquido,Lucro Aprovar Tudo,Ganho vs Aprovar Tudo,Ganho Percentual vs Aprovar Tudo
0,Teste,1,Top 10%,0.115833,2422,0.107644,581,1841,544,19534,...,0.715158,3.168675,4.797688,75455.311,25527.56,14127.146,49927.751,-4804.123,54731.874,11.392688
1,Teste,2,Top 20%,0.057323,4916,0.218489,786,4130,339,17245,...,0.752725,5.254453,3.197722,63640.277,15486.40,25942.180,48153.877,-4804.123,52958.000,11.023448
2,Teste,3,Top 30%,0.033999,7373,0.327689,900,6473,225,14902,...,0.748585,7.192222,2.441340,51119.157,9251.18,38463.300,41867.977,-4804.123,46672.100,9.715009
3,Teste,4,Top 40%,0.022588,9813,0.436133,982,8831,143,12544,...,0.729871,8.992872,2.001427,40012.668,6433.34,49569.789,33579.328,-4804.123,38383.451,7.989689
4,Teste,5,Top 50%,0.016084,12211,0.542711,1021,11190,104,10185,...,0.692023,10.959843,1.672263,30914.919,4621.60,58667.538,26293.319,-4804.123,31097.442,6.473074
5,Teste,6,Top 60%,0.011960,14659,0.651511,1064,13595,61,7780,...,0.654877,12.777256,1.451668,21757.909,2418.69,67824.548,19339.219,-4804.123,24143.342,5.025546
6,Teste,7,Top 70%,0.009171,16917,0.751867,1090,15827,35,5548,...,0.614222,14.520183,1.288645,14540.424,1452.30,75042.033,13088.124,-4804.123,17892.247,3.724352
7,Teste,8,Top 80%,0.007040,19014,0.845067,1109,17905,16,3470,...,0.574058,16.145176,1.166509,8622.119,1048.76,80960.338,7573.359,-4804.123,12377.482,2.576429
8,Teste,9,Top 90%,0.005031,20853,0.926800,1117,19736,8,1639,...,0.534784,17.668756,1.071309,4004.977,854.69,85577.480,3150.287,-4804.123,7954.410,1.655747
9,Teste,10,Top 100%,-inf,22500,1.000000,1125,21375,0,0,...,0.500000,19.000000,1.000000,0.000,0.00,89582.457,0.000,-4804.123,4804.123,1.000000



PONTO DE CORTE ESCOLHIDO NO TESTE
AUC global no teste: 0.8296
Average Precision no teste: 0.2881
Lucro aprovando tudo: R$ -4,804.12
Estratégia recomendada: Top 20%
Lucro máximo no teste: R$ 48,153.88
Ganho sobre aprovar tudo: R$ 52,958.00
Threshold recomendado: 0.05732306
Decil escolhido: 2
Top escolhido: Top 20%
Percentual real bloqueado no teste: 21.85%
Fraudes bloqueadas no teste: 786
Fraudes aprovadas no teste: 339
Legítimas bloqueadas no teste: 4130
Precisão no teste: 15.99%
Cobertura no teste: 69.87%
Erros por acerto no teste: 5.25

RESULTADO FINAL NA VALIDAÇÃO
Estratégia definida no teste: Top 20%
Threshold definido no teste e aplicado na validação: 0.05732306
AUC global na validação: 0.8248
Average Precision na validação: 0.2749
Lucro aprovando tudo na validação: R$ 19,262.27
Lucro da estratégia na validação: R$ 47,577.21
Ganho vs aprovar tudo na validação: R$ 28,314.94


...


Arquivos salvos:
limites_decis_treino.csv
desempenho_acumulado_decis_teste.csv
teste_com_decil_e_decisao.csv
resultado_final_validacao.csv
validacao_com_decil_e_decisao.csv
resumo_final_teste_validacao.csv

RESULTADO DO MODELO NO TOP 20% EXATO DA VALIDAÇÃO


,Base Avaliada,Tipo de Avaliação,Decil Acumulado,Top Acumulado,Percentual Teórico,Threshold Treino,Total Base,Selecionados para Bloqueio,% Real Selecionado,TP - Fraudes Bloqueadas,...,Lift,Valor Fraude Bloqueada,Perda de Fraude Evitada,Receita Comissão,Perda com Fraude,Comissão Perdida por Bloqueio,Lucro Líquido,Lucro Aprovar Tudo,Ganho vs Aprovar Tudo,Ganho Percentual vs Aprovar Tudo
0,Validação,Top 20% exato pela probabilidade do modelo,2,Top 20%,0.2,0.062763,22500,4500,0.2,749,...,3.328889,52247.91,52247.91,66793.783,20044.87,24761.265,46748.913,19262.268,27486.645,1.426968



CORTE ESCOLHIDO X TOP 20% EXATO DO MODELO


,Cenário,Threshold,% Real Bloqueado,TP - Fraudes Bloqueadas,FP - Legítimas Bloqueadas,FN - Fraudes Aprovadas,TN - Legítimas Aprovadas,Precisão,Cobertura,F1,...,Average Precision Global,Balanced Accuracy,Erro_por_Acerto,Lift,Receita Comissão,Perda com Fraude,Comissão Perdida por Bloqueio,Lucro Líquido,Lucro Aprovar Tudo,Ganho vs Aprovar Tudo
0,Corte escolhido no teste: Top 20%,0.057323,0.215733,781,4073,344,17302,0.160898,0.694222,0.261248,...,0.274856,0.751836,5.215109,3.217965,64886.115,17308.91,26668.933,47577.205,19262.268,28314.937
1,Top 20% exato na validação,0.062763,0.200000,749,3751,376,17624,0.166444,0.665778,0.266311,...,0.274856,0.745146,5.008011,3.328889,66793.783,20044.87,24761.265,46748.913,19262.268,27486.645



Resumo do Top 20% exato:
Quantidade bloqueada: 4,500
Percentual bloqueado: 20.00%
Threshold observado: 0.06276306
Precisão: 16.64%
Cobertura: 66.58%
F1: 0.2663
Erros por acerto: 5.01
Lucro líquido: R$ 46,748.91
Ganho vs aprovar tudo: R$ 27,486.64

Arquivos adicionais salvos:
resultado_top20_modelo_validacao.csv
comparacao_corte_escolhido_vs_top20_modelo_validacao.csv
validacao_top20_modelo_detalhada.csv


#### 7.7 Avaliação do Score Atual por Decis e Comparação com Top 20%###### Aqui é avaliado o desempenho do score atual nas bases de treino, teste e validação. Os limites dos decis são definidos no treino e aplicados no teste, onde são calculadas métricas preditivas e financeiras para diferentes percentuais de bloqueio. O corte final é escolhido entre as estratégias que preservam pelo menos 85% do lucro máximo, priorizando a maior cobertura de fraudes, e depois é aplicado sem alteração na validação. Como análise complementar, o código também avalia o bloqueio exato dos 20% de maior risco na validação, comparando esse cenário com a estratégia escolhida em termos de fraudes capturadas, erros, lucro líquido e ganho em relação à aprovação de todas as transações.e o resumo final para análise e acompanhamento.

In [39]:
# ============================================================
# AVALIAÇÃO COMPLETA DO SCORE ATUAL
#
# 1. TREINO:
#    Define os thresholds acumulados dos decis.
#
# 2. TESTE:
#    Aplica todos os thresholds definidos no treino.
#    Escolhe o corte que maximiza o lucro líquido.
#
# 3. VALIDAÇÃO:
#    Aplica somente o corte escolhido no teste.
#    Calcula o desempenho final fora da seleção do corte.
#
# IMPORTANTE:
# Este código usa a coluna "score" existente nas bases.
# Não usa predict_proba nem modelo treinado.
# ============================================================


# ============================================================
# 0. IMPORTAÇÕES
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix
)

from scipy.stats import ks_2samp


# ============================================================
# 1. CONFIGURAÇÕES
# ============================================================

TARGET = "fraude"
COLUNA_SCORE = "score"
COLUNA_VALOR = "monto"

TAXA_COMISSAO = 0.10
PERDA_FRAUDE = 1.00

# True:
# quanto maior o score, maior o risco de fraude.
#
# False:
# quanto menor o score, maior o risco de fraude.
SCORE_MAIOR_INDICA_MAIS_RISCO = True



# ============================================================
# 6. PREPARAR TREINO, TESTE E VALIDAÇÃO
# ============================================================

treino_score = preparar_base_score_original(
    df=df_treino_tratado,
    nome_base="Treino"
)

teste_score = preparar_base_score_original(
    df=df_teste_tratado,
    nome_base="Teste"
)

validacao_score = preparar_base_score_original(
    df=df_validacao_tratado,
    nome_base="Validação"
)


print("=" * 100)
print("BASES PREPARADAS")
print("=" * 100)

print(f"Treino: {treino_score.shape}")
print(f"Teste: {teste_score.shape}")
print(f"Validação: {validacao_score.shape}")

print(
    f"\nTaxa de fraude no treino: "
    f"{treino_score[TARGET].mean():.2%}"
)

print(
    f"Taxa de fraude no teste: "
    f"{teste_score[TARGET].mean():.2%}"
)

print(
    f"Taxa de fraude na validação: "
    f"{validacao_score[TARGET].mean():.2%}"
)


# ============================================================
# 7. VALIDAR O SENTIDO DO SCORE NO TREINO
# ============================================================

resumo_score_target = (
    treino_score
    .groupby(TARGET)
    .agg(
        Quantidade=(TARGET, "size"),
        Score_Médio=(COLUNA_SCORE, "mean"),
        Score_Mediano=(COLUNA_SCORE, "median"),
        Score_Mínimo=(COLUNA_SCORE, "min"),
        Score_Máximo=(COLUNA_SCORE, "max")
    )
    .reset_index()
)

print("\n" + "=" * 100)
print("VALIDAÇÃO DO SENTIDO DO SCORE NO TREINO")
print("=" * 100)

display(resumo_score_target)

auc_treino, ap_treino, ks_treino = (
    calcular_metricas_globais_original(
        treino_score
    )
)

print(f"AUC do score no treino: {auc_treino:.4f}")
print(f"Average Precision no treino: {ap_treino:.4f}")
print(f"KS no treino: {ks_treino:.4f}")

if not pd.isna(auc_treino) and auc_treino < 0.50:

    print(
        "\nATENÇÃO: AUC menor que 0,50. "
        "O sentido do score pode estar invertido."
    )

    print(
        "Verifique SCORE_MAIOR_INDICA_MAIS_RISCO."
    )


# ============================================================
# 8. DEFINIR THRESHOLDS DOS DECIS NO TREINO
# ============================================================

percentuais_acumulados = np.arange(
    0.10,
    1.01,
    0.10
)

thresholds_treino = []

for percentual in percentuais_acumulados:

    threshold = float(
        treino_score["score_risco"].quantile(
            1 - percentual
        )
    )

    thresholds_treino.append(threshold)


limites_decis_score = pd.DataFrame({
    "Decil Acumulado": np.arange(1, 11),
    "Percentual Acumulado": percentuais_acumulados,
    "Top Acumulado": [
        f"Top {percentual:.0%}"
        for percentual in percentuais_acumulados
    ],
    "Threshold Score Risco Treino": thresholds_treino
})


if SCORE_MAIOR_INDICA_MAIS_RISCO:

    limites_decis_score[
        "Threshold Score Original"
    ] = limites_decis_score[
        "Threshold Score Risco Treino"
    ]

else:

    limites_decis_score[
        "Threshold Score Original"
    ] = (
        -limites_decis_score[
            "Threshold Score Risco Treino"
        ]
    )


# Top 100% deve incluir todos os registros
limites_decis_score.loc[
    limites_decis_score["Decil Acumulado"] == 10,
    "Threshold Score Risco Treino"
] = -np.inf

if SCORE_MAIOR_INDICA_MAIS_RISCO:

    limites_decis_score.loc[
        limites_decis_score["Decil Acumulado"] == 10,
        "Threshold Score Original"
    ] = -np.inf

else:

    limites_decis_score.loc[
        limites_decis_score["Decil Acumulado"] == 10,
        "Threshold Score Original"
    ] = np.inf


print("\n" + "=" * 100)
print("THRESHOLDS DOS DECIS DEFINIDOS NO TREINO")
print("=" * 100)

display(limites_decis_score)


# ============================================================
# 9. MÉTRICAS GLOBAIS DO TESTE
# ============================================================

auc_global_teste, average_precision_global_teste, ks_global_teste = (
    calcular_metricas_globais_original(
        teste_score
    )
)


# ============================================================
# 10. CENÁRIO BASE DO TESTE
# ============================================================

cenario_base_teste = calcular_cenario_aprovar_tudo_original(
    teste_score
)

lucro_base_teste = cenario_base_teste[
    "Lucro Aprovar Tudo"
]


print("\n" + "=" * 100)
print("CENÁRIO BASE NO TESTE")
print("=" * 100)

print(
    f"Valor legítimo total: "
    f"R$ {cenario_base_teste['Valor Legítimo Total']:,.2f}"
)

print(
    f"Valor de fraude total: "
    f"R$ {cenario_base_teste['Valor de Fraude Total']:,.2f}"
)

print(
    f"Receita aprovando tudo: "
    f"R$ {cenario_base_teste['Receita Aprovar Tudo']:,.2f}"
)

print(
    f"Perda aprovando tudo: "
    f"R$ {cenario_base_teste['Perda Aprovar Tudo']:,.2f}"
)

print(
    f"Lucro aprovando tudo: "
    f"R$ {lucro_base_teste:,.2f}"
)


# ============================================================
# 11. APLICAR TODOS OS THRESHOLDS NO TESTE
# ============================================================

resultados_teste = []

for _, linha in limites_decis_score.iterrows():

    resultado = avaliar_threshold_original(
        base=teste_score,
        threshold_score_risco=float(
            linha["Threshold Score Risco Treino"]
        ),
        nome_base="Teste",
        auc_global=auc_global_teste,
        average_precision_global=(
            average_precision_global_teste
        ),
        ks_global=ks_global_teste,
        lucro_base_aprovar_tudo=lucro_base_teste,
        decil_acumulado=int(
            linha["Decil Acumulado"]
        ),
        percentual_teorico=float(
            linha["Percentual Acumulado"]
        ),
        top_acumulado=linha["Top Acumulado"]
    )

    resultados_teste.append(resultado)


comparativo_teste = pd.DataFrame(
    resultados_teste
)


# ============================================================
# 12. ESCOLHER O MELHOR CORTE NO TESTE
#
# Critério:
# 1. Identifica o maior lucro líquido.
# 2. Mantém os cortes com pelo menos 85% do lucro máximo.
# 3. Entre os candidatos, escolhe a maior cobertura.
# 4. Compara o corte escolhido com aprovar tudo.
# ============================================================

PERCENTUAL_MINIMO_LUCRO = 0.85

# ------------------------------------------------------------
# Verificar se as colunas necessárias existem
# ------------------------------------------------------------

colunas_necessarias_selecao = [
    "Lucro Líquido",
    "Cobertura",
    "Threshold Score Risco",
    "Threshold Score Original",
    "Top Acumulado",
    "Decil Acumulado"
]

colunas_ausentes_selecao = [
    coluna
    for coluna in colunas_necessarias_selecao
    if coluna not in comparativo_teste.columns
]

if colunas_ausentes_selecao:
    raise ValueError(
        "Colunas ausentes em comparativo_teste: "
        f"{colunas_ausentes_selecao}. "
        f"Colunas disponíveis: "
        f"{comparativo_teste.columns.tolist()}"
    )


# ------------------------------------------------------------
# Maior lucro entre todos os cortes avaliados
# ------------------------------------------------------------

lucro_maximo_disponivel = float(
    comparativo_teste[
        "Lucro Líquido"
    ].max()
)


# ------------------------------------------------------------
# Limite mínimo de lucro aceito
#
# Se o lucro máximo for positivo:
# mantém cortes com pelo menos 85% do lucro máximo.
#
# Se o lucro máximo for negativo:
# utiliza somente o corte de maior lucro, evitando que
# a multiplicação por 85% distorça a seleção.
# ------------------------------------------------------------

if lucro_maximo_disponivel > 0:

    limite_minimo_lucro = (
        lucro_maximo_disponivel
        * PERCENTUAL_MINIMO_LUCRO
    )

    candidatos = comparativo_teste.loc[
        comparativo_teste["Lucro Líquido"]
        >= limite_minimo_lucro
    ].copy()

else:

    limite_minimo_lucro = lucro_maximo_disponivel

    candidatos = comparativo_teste.loc[
        comparativo_teste["Lucro Líquido"]
        == lucro_maximo_disponivel
    ].copy()


# ------------------------------------------------------------
# Garantir que existem candidatos
# ------------------------------------------------------------

if candidatos.empty:
    raise ValueError(
        "Nenhum corte atingiu o limite mínimo de lucro."
    )


# ------------------------------------------------------------
# Escolher maior cobertura entre os candidatos
#
# Em caso de empate:
# 1. Maior cobertura
# 2. Maior lucro líquido
# 3. Maior precisão
# 4. Menor percentual bloqueado
# ------------------------------------------------------------

candidatos = candidatos.sort_values(
    by=[
        "Cobertura",
        "Lucro Líquido",
        "Precisão",
        "% Real Selecionado"
    ],
    ascending=[
        False,
        False,
        False,
        True
    ]
).reset_index(drop=True)


melhor_corte_teste = (
    candidatos
    .iloc[0]
    .copy()
)


melhor_lucro_com_bloqueio = float(
    melhor_corte_teste[
        "Lucro Líquido"
    ]
)


# ------------------------------------------------------------
# Comparar o melhor corte com aprovar tudo
# ------------------------------------------------------------

if lucro_base_teste >= melhor_lucro_com_bloqueio:

    recomendacao_final = (
        "APROVAR TODAS AS TRANSAÇÕES"
    )

    threshold_final_score_risco = np.nan
    threshold_final_original = np.nan

    lucro_maximo_teste = (
        lucro_base_teste
    )

else:

    recomendacao_final = (
        melhor_corte_teste[
            "Top Acumulado"
        ]
    )

    # Esta coluna vem da função avaliar_threshold_original()
    threshold_final_score_risco = float(
        melhor_corte_teste[
            "Threshold Score Risco"
        ]
    )

    threshold_final_original = float(
        melhor_corte_teste[
            "Threshold Score Original"
        ]
    )

    lucro_maximo_teste = (
        melhor_lucro_com_bloqueio
    )


# ------------------------------------------------------------
# Exibir candidatos e corte escolhido
# ------------------------------------------------------------

print("\n" + "=" * 120)
print("CANDIDATOS À SELEÇÃO DO CORTE")
print("=" * 120)

print(
    f"Maior lucro disponível: "
    f"R$ {lucro_maximo_disponivel:,.2f}"
)

print(
    f"Percentual mínimo preservado: "
    f"{PERCENTUAL_MINIMO_LUCRO:.0%}"
)

print(
    f"Limite mínimo de lucro: "
    f"R$ {limite_minimo_lucro:,.2f}"
)

display(
    candidatos[
        [
            "Decil Acumulado",
            "Top Acumulado",
            "Threshold Score Original",
            "% Real Selecionado",
            "Precisão",
            "Cobertura",
            "F1",
            "Lucro Líquido",
            "Ganho vs Aprovar Tudo"
        ]
    ]
)


print("\n" + "=" * 120)
print("CORTE SELECIONADO")
print("=" * 120)

print(
    f"Estratégia: "
    f"{recomendacao_final}"
)

print(
    f"Lucro da estratégia: "
    f"R$ {lucro_maximo_teste:,.2f}"
)

print(
    f"Lucro aprovando tudo: "
    f"R$ {lucro_base_teste:,.2f}"
)

print(
    f"Ganho vs aprovar tudo: "
    f"R$ {lucro_maximo_teste - lucro_base_teste:,.2f}"
)

if recomendacao_final != "APROVAR TODAS AS TRANSAÇÕES":

    print(
        f"Threshold na escala de risco: "
        f"{threshold_final_score_risco:.8f}"
    )

    print(
        f"Threshold na escala original: "
        f"{threshold_final_original:.8f}"
    )

    print(
        f"Cobertura escolhida: "
        f"{melhor_corte_teste['Cobertura']:.2%}"
    )

    print(
        f"Precisão escolhida: "
        f"{melhor_corte_teste['Precisão']:.2%}"
    )

    print(
        f"Percentual real bloqueado: "
        f"{melhor_corte_teste['% Real Selecionado']:.2%}"
    )
# ============================================================
# 13. EXIBIR RESULTADOS ACUMULADOS DO TESTE
# ============================================================

colunas_exibicao = [
    "Base Avaliada",
    "Decil Acumulado",
    "Top Acumulado",
    "Threshold Score Original",
    "Selecionados para Bloqueio",
    "% Real Selecionado",
    "TP - Fraudes Bloqueadas",
    "FP - Legítimas Bloqueadas",
    "FN - Fraudes Aprovadas",
    "TN - Legítimas Aprovadas",
    "Precisão",
    "Cobertura",
    "F1",
    "AUC Global",
    "Average Precision Global",
    "KS Global",
    "Balanced Accuracy",
    "Erro_por_Acerto",
    "Lift",
    "Receita Comissão",
    "Perda com Fraude",
    "Comissão Perdida por Bloqueio",
    "Lucro Líquido",
    "Lucro Aprovar Tudo",
    "Ganho vs Aprovar Tudo",
    "Ganho Percentual vs Aprovar Tudo"
]


print("\n" + "=" * 120)
print("DESEMPENHO DOS THRESHOLDS NO TESTE")
print("=" * 120)

display(
    comparativo_teste[
        colunas_exibicao
    ]
)


print("\n" + "=" * 120)
print("CORTE ESCOLHIDO NO TESTE")
print("=" * 120)

print(
    f"AUC global no teste: "
    f"{auc_global_teste:.4f}"
)

print(
    f"Average Precision no teste: "
    f"{average_precision_global_teste:.4f}"
)

print(
    f"KS no teste: "
    f"{ks_global_teste:.4f}"
)

print(
    f"Lucro aprovando tudo: "
    f"R$ {lucro_base_teste:,.2f}"
)

print(
    f"Estratégia escolhida: "
    f"{recomendacao_final}"
)

print(
    f"Lucro máximo no teste: "
    f"R$ {lucro_maximo_teste:,.2f}"
)

print(
    f"Ganho no teste: "
    f"R$ {lucro_maximo_teste - lucro_base_teste:,.2f}"
)

if recomendacao_final != "APROVAR TODAS AS TRANSAÇÕES":

    operador = (
        ">="
        if SCORE_MAIOR_INDICA_MAIS_RISCO
        else "<="
    )

    print(
        f"Regra escolhida: "
        f"{COLUNA_SCORE} {operador} "
        f"{threshold_final_original:.8f}"
    )

    print(
        f"Decil escolhido: "
        f"{int(melhor_corte_teste['Decil Acumulado'])}"
    )

    print(
        f"Top escolhido: "
        f"{melhor_corte_teste['Top Acumulado']}"
    )


# ============================================================
# 14. ATRIBUIR DECIS AOS REGISTROS DO TESTE
# ============================================================

thresholds_score_risco = (
    limites_decis_score
    .sort_values("Decil Acumulado")
    ["Threshold Score Risco Treino"]
    .tolist()
)



teste_score["decil_score_atual"] = (
    teste_score["score_risco"]
    .apply(atribuir_decil_score_original)
)


if recomendacao_final == "APROVAR TODAS AS TRANSAÇÕES":

    teste_score["decisao_score_atual"] = "APROVAR"

else:

    teste_score["decisao_score_atual"] = np.where(
        teste_score["score_risco"]
        >= threshold_final_score_risco,
        "BLOQUEAR",
        "APROVAR"
    )


# ============================================================
# 15. AVALIAÇÃO FINAL NA VALIDAÇÃO
#
# Não procura outro corte.
# Aplica somente a estratégia escolhida no teste.
# ============================================================

auc_global_validacao, average_precision_global_validacao, ks_global_validacao = (
    calcular_metricas_globais_original(
        validacao_score
    )
)

cenario_base_validacao = calcular_cenario_aprovar_tudo_original(
    validacao_score
)

lucro_base_validacao = cenario_base_validacao[
    "Lucro Aprovar Tudo"
]


if recomendacao_final == "APROVAR TODAS AS TRANSAÇÕES":

    total_validacao = len(validacao_score)

    total_fraudes_validacao = int(
        validacao_score[TARGET].sum()
    )

    total_legitimas_validacao = int(
        total_validacao
        - total_fraudes_validacao
    )

    resultado_final_validacao = {
        "Origem Score": "Score Atual",
        "Base Avaliada": "Validação",
        "Decil Acumulado": np.nan,
        "Top Acumulado": (
            "Aprovar todas as transações"
        ),
        "Percentual Teórico": 0.0,
        "Threshold Score Risco": np.nan,
        "Threshold Score Original": np.nan,
        "Total Base": total_validacao,
        "Selecionados para Bloqueio": 0,
        "% Real Selecionado": 0.0,
        "TP - Fraudes Bloqueadas": 0,
        "FP - Legítimas Bloqueadas": 0,
        "FN - Fraudes Aprovadas": (
            total_fraudes_validacao
        ),
        "TN - Legítimas Aprovadas": (
            total_legitimas_validacao
        ),
        "Precisão": 0.0,
        "Cobertura": 0.0,
        "F1": 0.0,
        "AUC Global": auc_global_validacao,
        "Average Precision Global": (
            average_precision_global_validacao
        ),
        "KS Global": ks_global_validacao,
        "Balanced Accuracy": 0.5,
        "Erro_por_Acerto": np.nan,
        "Lift": 0.0,
        "Receita Comissão": (
            cenario_base_validacao[
                "Receita Aprovar Tudo"
            ]
        ),
        "Perda com Fraude": (
            cenario_base_validacao[
                "Perda Aprovar Tudo"
            ]
        ),
        "Comissão Perdida por Bloqueio": 0.0,
        "Lucro Líquido": lucro_base_validacao,
        "Lucro Aprovar Tudo": lucro_base_validacao,
        "Ganho vs Aprovar Tudo": 0.0,
        "Ganho Percentual vs Aprovar Tudo": 0.0
    }

    validacao_score[
        "decisao_score_atual"
    ] = "APROVAR"

else:

    resultado_final_validacao = avaliar_threshold_original(
        base=validacao_score,
        threshold_score_risco=(
            threshold_final_score_risco
        ),
        nome_base="Validação",
        auc_global=auc_global_validacao,
        average_precision_global=(
            average_precision_global_validacao
        ),
        ks_global=ks_global_validacao,
        lucro_base_aprovar_tudo=(
            lucro_base_validacao
        ),
        decil_acumulado=int(
            melhor_corte_teste[
                "Decil Acumulado"
            ]
        ),
        percentual_teorico=float(
            melhor_corte_teste[
                "Percentual Teórico"
            ]
        ),
        top_acumulado=(
            melhor_corte_teste[
                "Top Acumulado"
            ]
        )
    )

    validacao_score[
        "decisao_score_atual"
    ] = np.where(
        validacao_score["score_risco"]
        >= threshold_final_score_risco,
        "BLOQUEAR",
        "APROVAR"
    )


validacao_score["decil_score_atual"] = (
    validacao_score["score_risco"]
    .apply(atribuir_decil_score_original)
)

resultado_final_validacao = pd.DataFrame([
    resultado_final_validacao
])


# ============================================================
# 16. EXIBIR RESULTADO FINAL DA VALIDAÇÃO
# ============================================================

print("\n" + "=" * 120)
print("RESULTADO FINAL NA VALIDAÇÃO")
print("=" * 120)

print(
    f"Estratégia definida no teste: "
    f"{recomendacao_final}"
)

if recomendacao_final != "APROVAR TODAS AS TRANSAÇÕES":

    print(
        f"Threshold aplicado na validação: "
        f"{threshold_final_original:.8f}"
    )

print(
    f"AUC global na validação: "
    f"{auc_global_validacao:.4f}"
)

print(
    f"Average Precision na validação: "
    f"{average_precision_global_validacao:.4f}"
)

print(
    f"KS na validação: "
    f"{ks_global_validacao:.4f}"
)

print(
    f"Lucro aprovando tudo na validação: "
    f"R$ {lucro_base_validacao:,.2f}"
)

print(
    f"Lucro com a estratégia escolhida: "
    f"R$ "
    f"{resultado_final_validacao.loc[0, 'Lucro Líquido']:,.2f}"
)

print(
    f"Ganho vs aprovar tudo: "
    f"R$ "
    f"{resultado_final_validacao.loc[0, 'Ganho vs Aprovar Tudo']:,.2f}"
)

display(
    resultado_final_validacao[
        colunas_exibicao
    ]
)


# ============================================================
# 17. RESUMO FINAL TESTE X VALIDAÇÃO
# ============================================================

resumo_final = pd.DataFrame([
    {
        "Base": "Teste",
        "Função da Base": "Escolha do corte",
        "Estratégia": recomendacao_final,
        "Threshold Score Original": (
            threshold_final_original
        ),
        "AUC Global": auc_global_teste,
        "Average Precision Global": (
            average_precision_global_teste
        ),
        "KS Global": ks_global_teste,
        "Precisão": (
            melhor_corte_teste["Precisão"]
            if recomendacao_final
            != "APROVAR TODAS AS TRANSAÇÕES"
            else 0.0
        ),
        "Cobertura": (
            melhor_corte_teste["Cobertura"]
            if recomendacao_final
            != "APROVAR TODAS AS TRANSAÇÕES"
            else 0.0
        ),
        "F1": (
            melhor_corte_teste["F1"]
            if recomendacao_final
            != "APROVAR TODAS AS TRANSAÇÕES"
            else 0.0
        ),
        "Lucro Líquido": lucro_maximo_teste,
        "Lucro Aprovar Tudo": lucro_base_teste,
        "Ganho vs Aprovar Tudo": (
            lucro_maximo_teste
            - lucro_base_teste
        )
    },
    {
        "Base": "Validação",
        "Função da Base": "Avaliação final",
        "Estratégia": recomendacao_final,
        "Threshold Score Original": (
            threshold_final_original
        ),
        "AUC Global": auc_global_validacao,
        "Average Precision Global": (
            average_precision_global_validacao
        ),
        "KS Global": ks_global_validacao,
        "Precisão": resultado_final_validacao.loc[
            0,
            "Precisão"
        ],
        "Cobertura": resultado_final_validacao.loc[
            0,
            "Cobertura"
        ],
        "F1": resultado_final_validacao.loc[
            0,
            "F1"
        ],
        "Lucro Líquido": resultado_final_validacao.loc[
            0,
            "Lucro Líquido"
        ],
        "Lucro Aprovar Tudo": lucro_base_validacao,
        "Ganho vs Aprovar Tudo": (
            resultado_final_validacao.loc[
                0,
                "Ganho vs Aprovar Tudo"
            ]
        )
    }
])


print("\n" + "=" * 120)
print("RESUMO FINAL: TESTE X VALIDAÇÃO")
print("=" * 120)

display(resumo_final)


# ============================================================
# 18. SALVAR RESULTADOS
# ============================================================

limites_decis_score.to_csv(
    "outputs/modeling/limites_decis_score_atual_treino.csv",
    index=False,
    sep=";"
)

comparativo_teste.to_csv(
    "outputs/modeling/desempenho_decis_score_atual_teste.csv",
    index=False,
    sep=";"
)

teste_score.to_csv(
    "outputs/predictions/teste_com_decil_e_decisao_score_atual.csv",
    index=False,
    sep=";"
)

resultado_final_validacao.to_csv(
    "outputs/modeling/resultado_final_score_atual_validacao.csv",
    index=False,
    sep=";"
)

validacao_score.to_csv(
    "outputs/predictions/validacao_com_decil_e_decisao_score_atual.csv",
    index=False,
    sep=";"
)

resumo_final.to_csv(
    "outputs/modeling/resumo_final_score_atual_teste_validacao.csv",
    index=False,
    sep=";"
)


print("\nArquivos salvos:")

print(
    "limites_decis_score_atual_treino.csv"
)

print(
    "desempenho_decis_score_atual_teste.csv"
)

print(
    "teste_com_decil_e_decisao_score_atual.csv"
)

print(
    "resultado_final_score_atual_validacao.csv"
)

print(
    "validacao_com_decil_e_decisao_score_atual.csv"
)

print(
    "resumo_final_score_atual_teste_validacao.csv"
)

# ============================================================
# 16.1 CENÁRIO ALTERNATIVO:
# BLOQUEAR EXATAMENTE OS 20% DE MAIOR SCORE NA VALIDAÇÃO
#
# Este cenário é apenas comparativo.
# Não altera a estratégia escolhida no teste.
# ============================================================

PROPORCAO_FIXA_VALIDACAO = 0.20

validacao_top20 = validacao_score.copy()

total_validacao_top20 = len(
    validacao_top20
)

quantidade_top20 = int(
    np.ceil(
        total_validacao_top20
        * PROPORCAO_FIXA_VALIDACAO
    )
)


# ------------------------------------------------------------
# Ordenar do maior risco para o menor risco
#
# O índice é usado como critério de desempate para que sejam
# selecionados exatamente 20% dos registros.
# ------------------------------------------------------------

indices_top20 = (
    validacao_top20
    .sort_values(
        by="score_risco",
        ascending=False,
        kind="mergesort"
    )
    .head(quantidade_top20)
    .index
)


# ------------------------------------------------------------
# Marcar bloqueio
# ------------------------------------------------------------

validacao_top20[
    "bloqueio_top20_exato"
] = 0

validacao_top20.loc[
    indices_top20,
    "bloqueio_top20_exato"
] = 1


# ------------------------------------------------------------
# Threshold observado no Top 20%
# ------------------------------------------------------------

threshold_risco_top20 = float(
    validacao_top20.loc[
        indices_top20,
        "score_risco"
    ].min()
)

threshold_original_top20 = (
    threshold_risco_top20
    if SCORE_MAIOR_INDICA_MAIS_RISCO
    else -threshold_risco_top20
)


# ------------------------------------------------------------
# Matriz de confusão
# ------------------------------------------------------------

y_real_top20 = validacao_top20[
    TARGET
].astype(int)

y_pred_top20 = validacao_top20[
    "bloqueio_top20_exato"
].astype(int)

tn_top20, fp_top20, fn_top20, tp_top20 = (
    confusion_matrix(
        y_real_top20,
        y_pred_top20,
        labels=[0, 1]
    ).ravel()
)


# ------------------------------------------------------------
# Métricas do Top 20%
# ------------------------------------------------------------

precisao_top20 = precision_score(
    y_real_top20,
    y_pred_top20,
    zero_division=0
)

cobertura_top20 = recall_score(
    y_real_top20,
    y_pred_top20,
    zero_division=0
)

f1_top20 = f1_score(
    y_real_top20,
    y_pred_top20,
    zero_division=0
)

balanced_accuracy_top20 = balanced_accuracy_score(
    y_real_top20,
    y_pred_top20
)

erro_por_acerto_top20 = (
    fp_top20 / tp_top20
    if tp_top20 > 0
    else np.nan
)

taxa_fraude_validacao = (
    y_real_top20.mean()
)

lift_top20 = (
    precisao_top20
    / taxa_fraude_validacao
    if taxa_fraude_validacao > 0
    else np.nan
)

percentual_real_top20 = (
    y_pred_top20.sum()
    / len(validacao_top20)
)


# ------------------------------------------------------------
# Resultado financeiro do Top 20%
# ------------------------------------------------------------

mascara_bloqueada_top20 = (
    y_pred_top20 == 1
)

mascara_aprovada_top20 = (
    y_pred_top20 == 0
)

valor_legitimo_aprovado_top20 = float(
    validacao_top20.loc[
        mascara_aprovada_top20
        & (validacao_top20[TARGET] == 0),
        COLUNA_VALOR
    ].sum()
)

valor_fraude_aprovada_top20 = float(
    validacao_top20.loc[
        mascara_aprovada_top20
        & (validacao_top20[TARGET] == 1),
        COLUNA_VALOR
    ].sum()
)

valor_legitimo_bloqueado_top20 = float(
    validacao_top20.loc[
        mascara_bloqueada_top20
        & (validacao_top20[TARGET] == 0),
        COLUNA_VALOR
    ].sum()
)

valor_fraude_bloqueada_top20 = float(
    validacao_top20.loc[
        mascara_bloqueada_top20
        & (validacao_top20[TARGET] == 1),
        COLUNA_VALOR
    ].sum()
)

receita_comissao_top20 = (
    TAXA_COMISSAO
    * valor_legitimo_aprovado_top20
)

perda_com_fraude_top20 = (
    PERDA_FRAUDE
    * valor_fraude_aprovada_top20
)

comissao_perdida_top20 = (
    TAXA_COMISSAO
    * valor_legitimo_bloqueado_top20
)

perda_fraude_evitada_top20 = (
    PERDA_FRAUDE
    * valor_fraude_bloqueada_top20
)

lucro_liquido_top20 = (
    receita_comissao_top20
    - perda_com_fraude_top20
)

ganho_top20_vs_aprovar_tudo = (
    lucro_liquido_top20
    - lucro_base_validacao
)

ganho_percentual_top20 = (
    ganho_top20_vs_aprovar_tudo
    / abs(lucro_base_validacao)
    if lucro_base_validacao != 0
    else np.nan
)


# ------------------------------------------------------------
# Resultado do cenário Top 20%
# ------------------------------------------------------------

resultado_top20_validacao = pd.DataFrame([
    {
        "Origem Score": "Score Atual",
        "Base Avaliada": "Validação",
        "Tipo de Avaliação": (
            "Top 20% exato na validação"
        ),
        "Decil Acumulado": 2,
        "Top Acumulado": "Top 20%",
        "Percentual Teórico": 0.20,
        "Threshold Score Risco": (
            threshold_risco_top20
        ),
        "Threshold Score Original": (
            threshold_original_top20
        ),
        "Total Base": (
            total_validacao_top20
        ),
        "Selecionados para Bloqueio": int(
            y_pred_top20.sum()
        ),
        "% Real Selecionado": (
            percentual_real_top20
        ),
        "TP - Fraudes Bloqueadas": int(
            tp_top20
        ),
        "FP - Legítimas Bloqueadas": int(
            fp_top20
        ),
        "FN - Fraudes Aprovadas": int(
            fn_top20
        ),
        "TN - Legítimas Aprovadas": int(
            tn_top20
        ),
        "Precisão": precisao_top20,
        "Cobertura": cobertura_top20,
        "F1": f1_top20,
        "AUC Global": (
            auc_global_validacao
        ),
        "Average Precision Global": (
            average_precision_global_validacao
        ),
        "KS Global": (
            ks_global_validacao
        ),
        "Balanced Accuracy": (
            balanced_accuracy_top20
        ),
        "Erro_por_Acerto": (
            erro_por_acerto_top20
        ),
        "Lift": lift_top20,
        "Valor Fraude Bloqueada": (
            valor_fraude_bloqueada_top20
        ),
        "Perda de Fraude Evitada": (
            perda_fraude_evitada_top20
        ),
        "Receita Comissão": (
            receita_comissao_top20
        ),
        "Perda com Fraude": (
            perda_com_fraude_top20
        ),
        "Comissão Perdida por Bloqueio": (
            comissao_perdida_top20
        ),
        "Lucro Líquido": (
            lucro_liquido_top20
        ),
        "Lucro Aprovar Tudo": (
            lucro_base_validacao
        ),
        "Ganho vs Aprovar Tudo": (
            ganho_top20_vs_aprovar_tudo
        ),
        "Ganho Percentual vs Aprovar Tudo": (
            ganho_percentual_top20
        )
    }
])


# ============================================================
# 16.2 COMPARAR ESTRATÉGIA ESCOLHIDA X TOP 20%
# ============================================================

resultado_escolhido_comparacao = pd.DataFrame([
    {
        "Cenário": (
            f"Estratégia escolhida: "
            f"{recomendacao_final}"
        ),
        "Threshold Score Original": (
            threshold_final_original
        ),
        "% Real Bloqueado": (
            resultado_final_validacao.loc[
                0,
                "% Real Selecionado"
            ]
        ),
        "TP - Fraudes Bloqueadas": (
            resultado_final_validacao.loc[
                0,
                "TP - Fraudes Bloqueadas"
            ]
        ),
        "FP - Legítimas Bloqueadas": (
            resultado_final_validacao.loc[
                0,
                "FP - Legítimas Bloqueadas"
            ]
        ),
        "Precisão": (
            resultado_final_validacao.loc[
                0,
                "Precisão"
            ]
        ),
        "Cobertura": (
            resultado_final_validacao.loc[
                0,
                "Cobertura"
            ]
        ),
        "F1": (
            resultado_final_validacao.loc[
                0,
                "F1"
            ]
        ),
        "Erro_por_Acerto": (
            resultado_final_validacao.loc[
                0,
                "Erro_por_Acerto"
            ]
        ),
        "Lucro Líquido": (
            resultado_final_validacao.loc[
                0,
                "Lucro Líquido"
            ]
        ),
        "Lucro Aprovar Tudo": (
            lucro_base_validacao
        ),
        "Ganho vs Aprovar Tudo": (
            resultado_final_validacao.loc[
                0,
                "Ganho vs Aprovar Tudo"
            ]
        )
    },
    {
        "Cenário": (
            "Top 20% exato na validação"
        ),
        "Threshold Score Original": (
            threshold_original_top20
        ),
        "% Real Bloqueado": (
            percentual_real_top20
        ),
        "TP - Fraudes Bloqueadas": (
            tp_top20
        ),
        "FP - Legítimas Bloqueadas": (
            fp_top20
        ),
        "Precisão": (
            precisao_top20
        ),
        "Cobertura": (
            cobertura_top20
        ),
        "F1": (
            f1_top20
        ),
        "Erro_por_Acerto": (
            erro_por_acerto_top20
        ),
        "Lucro Líquido": (
            lucro_liquido_top20
        ),
        "Lucro Aprovar Tudo": (
            lucro_base_validacao
        ),
        "Ganho vs Aprovar Tudo": (
            ganho_top20_vs_aprovar_tudo
        )
    }
])


print("\n" + "=" * 120)
print("RESULTADO DO TOP 20% EXATO NA VALIDAÇÃO")
print("=" * 120)

display(
    resultado_top20_validacao
)


print("\n" + "=" * 120)
print("ESTRATÉGIA ESCOLHIDA X TOP 20% DA VALIDAÇÃO")
print("=" * 120)

display(
    resultado_escolhido_comparacao
)


# ============================================================
# 16.3 SALVAR O RESULTADO ADICIONAL
# ============================================================

resultado_top20_validacao.to_csv(
    "outputs/modeling/resultado_top20_score_atual_validacao.csv",
    index=False,
    sep=";"
)

resultado_escolhido_comparacao.to_csv(
    "outputs/modeling/comparacao_estrategia_escolhida_vs_top20_validacao.csv",
    index=False,
    sep=";"
)

validacao_top20.to_csv(
    "outputs/predictions/validacao_score_atual_top20_detalhada.csv",
    index=False,
    sep=";"
)

print("\nArquivos adicionais salvos:")

print(
    "resultado_top20_score_atual_validacao.csv"
)

print(
    "comparacao_estrategia_escolhida_vs_top20_validacao.csv"
)

print(
    "validacao_score_atual_top20_detalhada.csv"
)

BASES PREPARADAS
Treino: (105000, 51)
Teste: (22500, 51)
Validação: (22500, 51)

Taxa de fraude no treino: 5.00%
Taxa de fraude no teste: 5.00%
Taxa de fraude na validação: 5.00%

VALIDAÇÃO DO SENTIDO DO SCORE NO TREINO


,fraude,Quantidade,Score_Médio,Score_Mediano,Score_Mínimo,Score_Máximo
0,0,99750,46.856662,46.0,0,100
1,1,5250,69.896000,85.0,0,100


AUC do score no treino: 0.7265
Average Precision no treino: 0.1732
KS no treino: 0.4287

THRESHOLDS DOS DECIS DEFINIDOS NO TREINO


,Decil Acumulado,Percentual Acumulado,Top Acumulado,Threshold Score Risco Treino,Threshold Score Original
0,1,0.1,Top 10%,88.0,88.0
1,2,0.2,Top 20%,78.0,78.0
2,3,0.3,Top 30%,68.0,68.0
3,4,0.4,Top 40%,58.0,58.0
4,5,0.5,Top 50%,48.0,48.0
5,6,0.6,Top 60%,38.0,38.0
6,7,0.7,Top 70%,28.0,28.0
7,8,0.8,Top 80%,18.0,18.0
8,9,0.9,Top 90%,8.0,8.0
9,10,1.0,Top 100%,-inf,-inf



CENÁRIO BASE NO TESTE
Valor legítimo total: R$ 895,824.57
Valor de fraude total: R$ 94,386.58
Receita aprovando tudo: R$ 89,582.46
Perda aprovando tudo: R$ 94,386.58
Lucro aprovando tudo: R$ -4,804.12

CANDIDATOS À SELEÇÃO DO CORTE
Maior lucro disponível: R$ 40,080.21
Percentual mínimo preservado: 85%
Limite mínimo de lucro: R$ 34,068.17


...


Arquivos salvos:
limites_decis_score_atual_treino.csv
desempenho_decis_score_atual_teste.csv
teste_com_decil_e_decisao_score_atual.csv
resultado_final_score_atual_validacao.csv
validacao_com_decil_e_decisao_score_atual.csv
resumo_final_score_atual_teste_validacao.csv

RESULTADO DO TOP 20% EXATO NA VALIDAÇÃO


,Origem Score,Base Avaliada,Tipo de Avaliação,Decil Acumulado,Top Acumulado,Percentual Teórico,Threshold Score Risco,Threshold Score Original,Total Base,Selecionados para Bloqueio,...,Lift,Valor Fraude Bloqueada,Perda de Fraude Evitada,Receita Comissão,Perda com Fraude,Comissão Perdida por Bloqueio,Lucro Líquido,Lucro Aprovar Tudo,Ganho vs Aprovar Tudo,Ganho Percentual vs Aprovar Tudo
0,Score Atual,Validação,Top 20% exato na validação,2,Top 20%,0.2,78.0,78.0,22500,4500,...,2.986667,44381.87,44381.87,68724.03,27910.91,22831.018,40813.12,19262.268,21550.852,1.118812



ESTRATÉGIA ESCOLHIDA X TOP 20% DA VALIDAÇÃO


,Cenário,Threshold Score Original,% Real Bloqueado,TP - Fraudes Bloqueadas,FP - Legítimas Bloqueadas,Precisão,Cobertura,F1,Erro_por_Acerto,Lucro Líquido,Lucro Aprovar Tudo,Ganho vs Aprovar Tudo
0,Estratégia escolhida: Top 30%,68.0,0.306667,799,6101,0.115797,0.710222,0.199128,7.635795,40538.848,19262.268,21276.580
1,Top 20% exato na validação,78.0,0.200000,672,3828,0.149333,0.597333,0.238933,5.696429,40813.120,19262.268,21550.852



Arquivos adicionais salvos:
resultado_top20_score_atual_validacao.csv
comparacao_estrategia_escolhida_vs_top20_validacao.csv
validacao_score_atual_top20_detalhada.csv
